# 01 — Recovery YOLO26n · Fine-tuning standar · Filkom

Notebook standalone untuk recovery structured pruning LAMP + DepGraph pada Poltrada V3. Loss deteksi native. Kontrol fine-tuning ini memakai raw dan anggaran training yang sama dengan eksperimen KD.
Isi `RAW_ARTIFACT_REFS` dengan versi tetap W&B yang kamu terima dari tahap pruning. `PRUNING_SUITE_DIR` atau `RAW_MANIFEST_PATHS` dapat dipakai untuk hasil lokal.
Ketika input belum diisi, notebook melaporkan `NOT_RUN`. Default eksperimen 100 epoch dengan AdamW dan FP32. Untuk pemeriksaan pipeline, pilih `MODE="smoke"`.

EDA membandingkan baseline, raw pruned, dan recovered pada validation yang sama. Pilihan struktur student tetap memakai full validation tahap raw pruning.

## 02 — Konfigurasi data, artefak, device, dan protokol

In [ ]:
import os
from pathlib import Path

WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"
WANDB_PROJECT = "yolom26-kd-indotraffic"
MODEL_ARTIFACTS = {
    "n": "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/03_baseline_yolo26n-9uu3meve-checkpoints:v84",
    "s": "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62",
}
WORKSPACE_ROOT = Path(os.environ.get("SKRIPSI_ROOT", Path.cwd()))
DATASET_ROOT = WORKSPACE_ROOT / "datasets/poltrada-v3"
DATA_YAML = DATASET_ROOT / "data.yaml"
SEED = 42
IMGSZ = 640
DEVICE = "auto"
RUN_LATENCY_BENCHMARK = True
BATCH = 16
WORKERS = 2
LATENCY_WARMUP = 50
LATENCY_REPEATS = 200
RELOAD_TIMEOUT_SECONDS = 300
INSTALL_DEPENDENCIES = True
EDA_DPI = 300
EDA_MAX_LAYERS_PER_PAGE = 32
EDA_DISPLAY_RATIO = 0.50

PLATFORM = 'filkom'
RECOVERY_METHOD = 'finetune'
NOTEBOOK_VARIANT = 'n'
SOURCE_CHECKPOINT_SHA256 = {
    "n": "380a087fd621e8a43fe2b9b4d6b1bb6fb00a4d0443cc5afe81ef3efdf0775f0f",
    "s": "5bc3ae61f8da1c9f03916922de10146805c27c64a11e46edd766a00d0f6f18fd",
}
DATASET_ID = "ilhamrafiqin/poltrada-v3-yolo26-clean"
OUTPUT_DIR = WORKSPACE_ROOT / f'outputs/filkom/recovery/{RECOVERY_METHOD}/yolo26{NOTEBOOK_VARIANT}'

MODE = "full_val"  # "smoke" untuk satu epoch dan subset tetap maksimal 128 gambar per split.
PRUNING_SUITE_DIR = None  # Folder yang berisi student_candidates.json.
RAW_MANIFEST_PATHS = ()   # Alternatif untuk paket raw lokal tepercaya.
RAW_ARTIFACT_REFS = ()    # Contoh format: ("entity/project/raw-artifact:v0",)
EXPECTED_RAW_SHA256 = {}  # Isi {target: "sha256"} dari student_lock.json untuk perbandingan KD.
RECOVERY_TARGETS = ()     # () = semua target yang tersedia. Contoh: (1.02,)
GFLOPS_TARGETS = (1.02, 1.05, 1.10, 1.20, 1.70)
DATASET_MANIFEST_PATH = None  # Manifest sumber Poltrada V3 untuk salinan Filkom.
RESUME_SUITE_DIR = None
WANDB_TRACKING = True   # False hanya untuk smoke lokal.


RECOVERY_EPOCHS = 1 if MODE == "smoke" else 100
RECOVERY_BATCH = 16
RECOVERY_NBS = 64
RECOVERY_WORKERS = 2


def recovery_settings():
    return {"epochs": RECOVERY_EPOCHS, "optimizer": "AdamW", "lr0": 0.0001, "lrf": 0.01,
            "weight_decay": 0.0005, "momentum": 0.937, "warmup_epochs": 3, "warmup_bias_lr": 0.0,
            "close_mosaic": 10 if MODE == "full_val" else 0, "patience": 0,
            "batch": RECOVERY_BATCH, "nbs": RECOVERY_NBS, "workers": RECOVERY_WORKERS, "imgsz": IMGSZ,
            "device": str(TARGET_DEVICE), "seed": SEED, "deterministic": True, "amp": False,
            "resume": False, "freeze": None, "cache": False, "save": True, "val": True,
            "plots": True, "cos_lr": False, "mosaic": 1.0, "mixup": 0.0, "copy_paste": 0.0, "cutmix": 0.0,
            "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4, "degrees": 0.0, "translate": 0.1,
            "scale": 0.5, "shear": 0.0, "perspective": 0.0, "flipud": 0.0, "fliplr": 0.5, "multi_scale": False}

IMPLEMENTATION_SHA256 = 'c14b7fc0141185d21ebc6d37be24a516c240cc97bed2a0612358766956e4e004'
# Kebijakan I/O; tidak mengubah kontrak training/resume.
CHECKPOINT_RETENTION = "best-and-latest-v1"
AUTO_DELETE_SMOKE_ARTIFACTS = True  # Sesudah evaluasi, EDA, upload dan hash terverifikasi.
AUTO_DELETE_LOCAL_SMOKE_WEIGHTS = True  # Hanya salinan biner di folder smoke ini.


## 03 — Bootstrap kernel dan preflight

**Penyimpanan best/latest:** model inference hanya `best.pt` dan `last.pt`. Snapshot lokal mempertahankan model terbaik dan state resume terbaru. W&B menyimpan satu versi paket resume terbaru yang mencakup keduanya; versi dari run ini yang lebih lama dihapus hanya setelah penggantinya diunduh dan SHA-256 cocok. Run lain dan input raw/baseline/teacher tetap dilindungi. Loss, seluruh epoch history, Tables, Images, dan EDA eksperimen penuh tetap dicatat.

**Smoke:** setelah training, evaluasi, EDA dan verifikasi publikasi selesai, artefak diagnostik W&B serta salinan checkpoint lokal di folder smoke ini otomatis dihapus. Receipt SHA-256, metrik, tabel, grafik dan log lokal tetap disimpan. Registry menandai `DELETED_AFTER_SMOKE`; referensinya menjadi bukti historis dan tidak dapat dipakai recovery. Interupsi sebelum selesai tetap dapat di-resume. Untuk uji restore snapshot, nonaktifkan dua opsi cleanup selama pengujian lalu bersihkan setelah verifikasi.


In [ ]:
import csv
import hashlib
import importlib.metadata as metadata
import inspect
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import traceback
import uuid
import warnings
from contextlib import contextmanager
from copy import deepcopy
from datetime import datetime, timezone
from types import SimpleNamespace

PINNED = {"ultralytics": "8.4.155", "wandb": "0.28.1", "torch-pruning": "1.6.1"}
REQUIRED_CHECKS = (
    "baseline_loaded", "baseline_forward", "dependency_graph", "structural_channels_removed",
    "parameters_reduced", "pruned_forward", "protected_head_unchanged", "checkpoint_saved",
    "checkpoint_reloaded", "reload_consistent", "detection_prediction",
)


def write_json(path, value):
    """Tulis JSON standar; NaN/Inf ditolak, bukan disamarkan sebagai metrik."""
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def sha256_file(path):
    """Hash berkas tanpa membaca seluruh checkpoint ke RAM sekaligus."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
# Backward-compatible N helper aliases; public inputs are the per-model mappings.
WANDB_ARTIFACT = MODEL_ARTIFACTS["n"]
LOCAL_CHECKPOINT_PATH = None
LOCAL_EXPECTED_SHA256 = None


def model_spec(model_variant="n"):
    if model_variant not in {"n", "s"}:
        raise ValueError("Model variant harus n atau s.")
    local = LOCAL_CHECKPOINT_PATHS.get(model_variant)
    expected_hash = LOCAL_EXPECTED_SHA256S.get(model_variant)
    if model_variant == "n" and LOCAL_CHECKPOINT_PATH is not None:
        local, expected_hash = LOCAL_CHECKPOINT_PATH, LOCAL_EXPECTED_SHA256
    return {"model_variant": model_variant, "model_name": f"YOLO26{model_variant}",
            "artifact": MODEL_ARTIFACTS[model_variant], "local_checkpoint": local,
            "expected_sha256": expected_hash}


def policies_to_run(policy):
    if policy not in {"conservative", "extended", "both"}:
        raise ValueError("PRUNING_POLICY harus conservative, extended, atau both.")
    return ["conservative", "extended"] if policy == "both" else [policy]


def configuration(policy=None, ratio=None, model_variant=None):
    keys = ("MODEL_VARIANTS MODEL_ARTIFACTS LOCAL_CHECKPOINT_PATHS LOCAL_EXPECTED_SHA256S CHECKPOINT_SELECTOR "
            "DATASET_ROOT DATA_YAML OUTPUT_DIR PRUNING_POLICY PRUNING_RATIOS GLOBAL_PRUNING "
            "ITERATIVE_STEPS MAX_PRUNING_RATIO ROUND_TO IMPORTANCE_P SEED IMGSZ DEVICE "
            "RUN_FULL_VALIDATION RUN_LATENCY_BENCHMARK BATCH WORKERS INSTALL_DEPENDENCIES "
            "WANDB_TRACKING UPLOAD_WANDB_ARTIFACTS LATENCY_WARMUP LATENCY_REPEATS "
            "RELOAD_TIMEOUT_SECONDS RECOVERY_ENABLED FINETUNING_ENABLED KD_ENABLED "
            "EDA_DPI EDA_MAX_LAYERS_PER_PAGE EDA_DISPLAY_RATIO").split()
    def serializable(value):
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, dict):
            return {key: serializable(item) for key, item in value.items()}
        if isinstance(value, (tuple, list)):
            return [serializable(item) for item in value]
        return value
    result = {key: serializable(globals()[key]) for key in keys}
    if policy is not None:
        result["policy"] = policy
    if ratio is not None:
        result["ratio"] = ratio
    if model_variant is not None:
        result["selected_model"] = serializable(model_spec(model_variant))
    return result


def validate_configuration():
    if not MODEL_VARIANTS or len(set(MODEL_VARIANTS)) != len(MODEL_VARIANTS):
        raise ValueError("MODEL_VARIANTS harus nonkosong dan tanpa duplikat.")
    for model_variant in MODEL_VARIANTS:
        artifact = model_spec(model_variant)["artifact"]
        if artifact.count("/") != 2 or ":" not in artifact:
            raise ValueError("Artifact wajib entity/project/artifact:version.")
    policies_to_run(PRUNING_POLICY)
    if not PRUNING_RATIOS or len(set(PRUNING_RATIOS)) != len(PRUNING_RATIOS):
        raise ValueError("PRUNING_RATIOS harus nonkosong dan tanpa duplikat.")
    if any(isinstance(r, bool) or not isinstance(r, (int, float)) or not math.isfinite(r)
           or not 0 < r <= MAX_PRUNING_RATIO for r in PRUNING_RATIOS):
        raise ValueError("Setiap ratio wajib finite dan 0 < ratio <= MAX_PRUNING_RATIO.")
    if MAX_PRUNING_RATIO != 0.50 or ROUND_TO is not None or not GLOBAL_PRUNING:
        raise ValueError("V2 memakai global ranking, cap 0.50, dan ROUND_TO=None.")
    if ITERATIVE_STEPS != 1 or IMPORTANCE_P != 2:
        raise ValueError("V2 hanya mendukung one-shot dengan p=2.")
    if any((RECOVERY_ENABLED, FINETUNING_ENABLED, KD_ENABLED)):
        raise ValueError("Recovery, fine-tuning, dan KD wajib disabled.")
    if IMGSZ < 32 or IMGSZ % 32 or BATCH < 1 or WORKERS < 0:
        raise ValueError("IMGSZ kelipatan 32, BATCH positif, WORKERS nonnegatif.")
    if EDA_DPI < 72 or EDA_MAX_LAYERS_PER_PAGE < 1 or not 0 < EDA_DISPLAY_RATIO <= 0.5:
        raise ValueError("Konfigurasi EDA DPI/page/ratio invalid.")
    if LATENCY_WARMUP < 1 or LATENCY_REPEATS < 1 or RELOAD_TIMEOUT_SECONDS < 1:
        raise ValueError("Jumlah pengukuran dan timeout wajib positif.")


def write_csv(path, rows):
    if not rows:
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict))
                             else value for key, value in row.items()})


def model_state_sha256(core):
    """Buktikan setiap konfigurasi dimulai dari bobot/buffer FP32 yang identik."""
    digest = hashlib.sha256()
    for name, value in sorted(core.state_dict().items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str((str(tensor.dtype), tuple(tensor.shape))).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def metric_comparison(before, after):
    result = {}
    for key, label in (("parameters", "parameter"), ("gflops", "gflops"),
                       ("checkpoint_bytes", "checkpoint")):
        base, pruned = before.get(key), after.get(key)
        result[label + "_reduction_pct"] = 100 * (base - pruned) / base if base else None
    base = (before.get("validation") or {}).get("map50_95")
    pruned = (after.get("validation") or {}).get("map50_95")
    delta = pruned - base if base is not None and pruned is not None else None
    result.update({"delta_map50_95": delta, "delta_map50_95_pp": 100 * delta if delta is not None else None,
                   "relative_map50_95_pct": 100 * delta / base if delta is not None and base else None})
    return result


def lamp_v2_display_name(policy, ratio_tag, model_variant, recovery_epochs=None):
    model_spec(model_variant)
    policy_suffix = "_c" if policy == "conservative" else ""
    recovery_suffix = f"_ft{recovery_epochs}" if recovery_epochs is not None else ""
    return f"prun_lamp2_yolo26{model_variant}{policy_suffix}{recovery_suffix}_pruned-{ratio_tag}"


def run_identity(policy, ratio, suite_id, model_variant="n"):
    tag = f"pr{ratio:g}"
    return {"name": lamp_v2_display_name(policy, tag, model_variant),
            "id": uuid.uuid4().hex, "resume": "never", "group": f"lamp-v2-{suite_id}",
            "job_type": "lamp-structured-pruning-v2",
            "tags": [f"yolo26{model_variant}", "lamp", "v2", "structured", "pruned", policy,
                     "no-recovery", tag, f"seed{SEED}"]}


def safe_error(error):
    message = str(error)
    secret = os.environ.get("WANDB_API_KEY")
    return message.replace(secret, "[REDACTED]") if secret else message

In [ ]:
def ensure_dependencies(output_dir, install=True):
    """Pasang dependency tambahan sambil mengunci Torch/torchvision environment."""
    if sys.version_info < (3, 11):
        raise RuntimeError("Gunakan Python >=3.11.")
    preserved = {name: metadata.version(name) for name in ("torch", "torchvision")}
    needed = []
    for name, expected in PINNED.items():
        try:
            found = metadata.version(name)
        except metadata.PackageNotFoundError:
            found = None
        if found != expected:
            needed.append(f"{name}=={expected}")
    if needed and not install:
        raise RuntimeError(f"Dependency belum cocok: {needed}; aktifkan INSTALL_DEPENDENCIES.")
    if needed:
        constraints = Path(output_dir) / "preserve_torch.txt"
        constraints.write_text("\n".join(f"{k}=={v}" for k, v in preserved.items()) + "\n", encoding="utf-8")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        "--constraint", str(constraints), *needed], check=True)
    actual = {name: metadata.version(name) for name in preserved}
    if actual != preserved:
        raise RuntimeError(f"Torch/torchvision berubah: {preserved} -> {actual}; restart runtime asli.")
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Versi {name} tidak sesuai pin {expected}; restart kernel bila perlu.")

In [ ]:
"""Shared standalone persistence, provenance and W&B helpers for pruning/recovery."""
# ruff: noqa: F821 -- Notebook declarations supply SDK imports and configuration.
import json
import os
import re
import uuid
from contextlib import contextmanager
from datetime import UTC
from pathlib import Path


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def write_csv(path, rows):
    import csv
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        writer.writerows({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value
                         for key, value in row.items()} for row in rows)
    os.replace(temporary, path)


def workflow_digest(value):
    import hashlib
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str, allow_nan=False).encode()).hexdigest()


def workflow_bind(folder, name, contract):
    """Reject semantic changes on resume, allowing absolute paths to relocate."""
    path = Path(folder) / name
    contract = json.loads(json.dumps(contract, sort_keys=True, default=str, allow_nan=False))
    record = {"contract": contract, "sha256": workflow_digest(contract)}
    if path.is_file() and json.loads(path.read_text(encoding="utf-8")) != record:
        raise ValueError(f"Kontrak resume berbeda: {name}. Gunakan suite baru.")
    write_json(path, record)
    return record


def workflow_save_checkpoint(core, path, provenance):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = YOLO(str(path), task="detect").model.cpu().float().eval()
        if model_state_sha256(existing) != model_state_sha256(core):
            raise ValueError(f"Checkpoint immutable sudah berisi state berbeda: {path}")
        return {"path": str(path.resolve()), "sha256": sha256_file(path), "file_bytes": path.stat().st_size}
    temporary = path.with_name(path.stem + "_" + uuid.uuid4().hex + ".pt")
    saved = save_fp32_checkpoint(core, temporary, provenance)
    os.replace(temporary, path)
    return {**saved, "path": str(path.resolve())}


def workflow_suite_dir(output, resume=None):
    if resume:
        path = Path(resume).expanduser().resolve(strict=True)
        if not path.is_dir():
            raise NotADirectoryError(path)
        return path
    from datetime import datetime
    path = Path(output).expanduser().resolve() / (datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
    path.mkdir(parents=True, exist_ok=False)
    return path


def workflow_status(statuses, *, useful=False):
    statuses = list(statuses)
    if statuses and all(status == "PASS" for status in statuses):
        return "PASS"
    if useful or any(status in {"PASS", "PARTIAL"} for status in statuses):
        return "PARTIAL"
    return "FAIL" if any(status == "FAIL" for status in statuses) else "NOT_RUN"


def workflow_run_name(variant, target, recovery=False):
    tag = f"{target:.2f}".replace(".", "p")
    method = globals().get("RECOVERY_METHOD", "finetune")
    recovery_tag = f"_ft{RECOVERY_EPOCHS}" if method == "finetune" else f"_{method}{RECOVERY_EPOCHS}"
    ft = recovery_tag if recovery else ""
    smoke = "_smoke" if MODE == "smoke" else ""
    return f"prun_lamp_gflops_yolo26{variant}_{PLATFORM}{ft}_su{tag}_seed{SEED}{smoke}"


def workflow_login():
    import getpass

    import wandb
    key = os.environ.get("WANDB_API_KEY")
    if not key and PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except (ImportError, RuntimeError, ValueError):
            pass
    if not key:
        key = getpass.getpass("WANDB_API_KEY: ")
    if not key:
        raise ValueError("W&B memerlukan API key untuk publikasi artefak.")
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key, relogin=False)


def workflow_version_ref(reference):
    if not isinstance(reference, str) or not re.fullmatch(r"[^/:]+/[^/:]+/[^/:]+:v\d+", reference):
        raise ValueError("Input W&B memerlukan entity/project/artifact:vN (versi tetap).")
    return reference


@contextmanager
def workflow_tracking(folder, variant, target, *, recovery=False, source_ref=None, group=None, registration=False):
    import wandb
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    run = None
    if not WANDB_TRACKING:
        yield None
        return
    identity_path = folder / "wandb_identity.json"
    existed = identity_path.is_file()
    name = workflow_run_name(variant, target, recovery) + ("_register_raw" if registration else "")
    identity = json.loads(identity_path.read_text(encoding="utf-8")) if existed else {
        "id": uuid.uuid4().hex, "name": name, "group": group or f"lamp-gflops-{PLATFORM}-{folder.parent.name}"}
    if identity["name"] != name:
        raise ValueError("Identitas run resume berbeda dari konfigurasi.")
    write_json(identity_path, identity)
    try:
        workflow_login()
        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=name, id=identity["id"],
                         resume="allow" if existed else "never", group=identity["group"],
                         job_type=("register-raw" if registration else
                                   globals().get("RECOVERY_METHOD", "finetune") + "-recovery" if recovery else "pruning-no-recovery"),
                         tags=[PLATFORM, f"yolo26{variant}", "lamp", "depgraph", f"seed{SEED}",
                               "recovery" if recovery else "raw-pruned", MODE],
                         config={"target_speedup": target, "model_variant": variant, "platform": PLATFORM,
                                 "seed": SEED, "mode": MODE})
        if source_ref:
            run.use_artifact(workflow_version_ref(source_ref), type="model")
        yield run
    finally:
        if run is not None:
            run.finish()


class WorkflowUploadPending(RuntimeError):
    """Local computation is committed; only publication needs retrying."""


def workflow_publish(run, folder, label, files, *, metadata_info=None, aliases=None, event=None, kind="model"):
    import wandb
    folder = Path(folder)
    path = folder / "artifact_publications.json"
    publications = json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}
    files = {name: Path(value).resolve(strict=True) for name, value in files.items()}
    hashes = {name: sha256_file(value) for name, value in files.items()}
    key = label + (":" + event if event else "")
    prior = publications.get(key)
    if prior and prior.get("status") == "CONFIRMED" and prior.get("files_sha256") == hashes:
        return prior
    record = {"status": "NOT_RUN", "files_sha256": hashes, "label": label, "event": event}
    if run is None:
        publications[key] = record
        write_json(path, publications)
        return record
    try:
        artifact = wandb.Artifact(f"{run.name}-{run.id}-{label}", type=kind,
                                 metadata={**(metadata_info or {}), "files_sha256": hashes})
        for name, value in files.items():
            artifact.add_file(str(value), name=name)
        uploaded = run.log_artifact(artifact, aliases=aliases or ["latest"])
        uploaded.wait(timeout=300)
        reference = workflow_version_ref(uploaded.qualified_name)
        record.update(status="CONFIRMED", reference=reference)
    except Exception as error:
        record.update(status="FAIL", error=safe_error(error))
        publications[key] = record
        write_json(path, publications)
        raise WorkflowUploadPending(safe_error(error)) from error
    publications[key] = record
    write_json(path, publications)
    return record


def workflow_report_files(folder):
    folder = Path(folder)
    return {path.relative_to(folder).as_posix(): path for path in folder.rglob("*")
            if path.is_file() and path.suffix in {".json", ".csv", ".yaml", ".md", ".png", ".svg"}
            and "wandb" not in path.relative_to(folder).parts and path.name != "artifact_publications.json"}


def workflow_dataset_manifest(dataset, source_yaml, root, folder):
    """Portable full train/val content identity, never an evaluation train=val placeholder."""
    raw = yaml.safe_load(Path(source_yaml).read_text(encoding="utf-8-sig"))
    names = ordered_names(raw["names"])
    root = Path(root).resolve()
    train = _gflops_split_images(root, raw.get("train"))
    val = _gflops_split_images(root, raw.get("val"))
    if any(image.with_suffix(".npy").is_file() for image in (*train, *val)):
        raise ValueError("Gunakan sumber dataset tanpa cache gambar .npy.")
    train_fp, train_rows = _gflops_fingerprint(root, train, "train", names)
    val_fp, val_rows = _gflops_fingerprint(root, val, "val", names)
    if {row["image_sha256"] for row in train_rows}.intersection(row["image_sha256"] for row in val_rows):
        raise ValueError("Train dan val berbagi konten gambar; recovery tidak boleh memakai train=val.")
    record = {"format_version": 1, "dataset_id": DATASET_ID, "class_names": names,
              "train_fingerprint": train_fp, "val_fingerprint": val_fp,
              "provenance_status": dataset.get("provenance_status", "UNVERIFIED"),
              "train_count": len(train), "val_count": len(val), "source_yaml_sha256": sha256_file(source_yaml)}
    write_json(Path(folder) / "dataset_manifest.json", record)
    return record, train, val

In [ ]:
SUITE_DIR = workflow_suite_dir(OUTPUT_DIR, RESUME_SUITE_DIR)
try:
    ensure_dependencies(SUITE_DIR, install=INSTALL_DEPENDENCIES)
except Exception as error:
    failure = {"status": "FAIL", "stage": "environment", "error": safe_error(error)}
    write_json(SUITE_DIR / "environment_failure.json", failure)
    if not (SUITE_DIR / "suite_manifest.json").is_file():
        write_json(SUITE_DIR / "suite_manifest.json", failure)
    raise

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch_pruning as tp
import ultralytics
import wandb
import yaml
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from ultralytics.nn.modules import Bottleneck, C2PSA, Detect
from ultralytics.nn.modules.block import AAttn, Attention
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import _attention_ops
from thop import profile as thop_profile

In [ ]:
def seed_environment(seed):
    """Samakan RNG dan backend FP32 di notebook dan proses restore."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")


def environment_report():
    """Periksa API aktual, bukan API yang diasumsikan dari contoh YOLOv8."""
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Pin {name} tidak terpenuhi.")
    if ultralytics.__version__ != PINNED["ultralytics"]:
        raise RuntimeError("Modul Ultralytics lama masih di kernel; restart runtime.")
    signature = inspect.signature(tp.pruner.BasePruner)
    for key in ("global_pruning", "max_pruning_ratio", "round_to", "output_transform", "forward_fn"):
        if key not in signature.parameters:
            raise RuntimeError(f"BasePruner tidak mendukung {key}.")
    return {
        "python": sys.version, "platform": platform.platform(),
        "packages": {name: metadata.version(name) for name in (*PINNED, "torch", "torchvision", "numpy", "ultralytics-thop")},
        "cuda_available": torch.cuda.is_available(), "cuda_build": torch.version.cuda,
        "device": str(TARGET_DEVICE),
        "gpu": torch.cuda.get_device_name(TARGET_DEVICE) if TARGET_DEVICE.type == "cuda" else None,
        "precision": "FP32", "tf32": False, "cudnn_benchmark": False,
        "base_pruner_signature": str(signature),
        "lamp_signature": str(inspect.signature(tp.importance.LAMPImportance)),
        "lamp_source_sha256": hashlib.sha256(inspect.getsource(tp.importance.LAMPImportance).encode()).hexdigest(),
        "importance_module_sha256": sha256_file(inspect.getfile(tp.importance.LAMPImportance)),
        "flops_method": "THOP full-resolution Conv/BN + separate Ultralytics attention matmul counter; GFLOPs=2*MACs/1e9",
        "thop_profile_signature": str(inspect.signature(thop_profile)),
        "thop_distribution_versions": thop_versions(),
    }


def thop_versions():
    """Catat kedua distribution yang dapat menyediakan namespace thop yang sama."""
    result = {}
    for name in ("thop", "ultralytics-thop"):
        try:
            result[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            result[name] = None
    return result

In [ ]:
TARGET_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else torch.device(DEVICE)
seed_environment(SEED)
ENVIRONMENT = environment_report()
write_json(SUITE_DIR / "environment.json", ENVIRONMENT)
print("Runtime", TARGET_DEVICE, "| FP32 | Seed", SEED)
print("Kernel", sys.executable, "| cwd", Path.cwd(), "| platform", PLATFORM)
print("GPU", ENVIRONMENT.get("gpu"), "| CUDA", torch.version.cuda, "| Output", SUITE_DIR)
display(pd.DataFrame(ENVIRONMENT["packages"].items(), columns=["package", "version"]))

## 04 — Audit input dan evaluator identik
Fingerprint dibentuk dari isi train dan val. Setiap checkpoint diperiksa dari manifest dan hash, lalu diukur dengan evaluator pruning yang sama.
Latensi memakai raw forward dual-head FP32 pada device, ukuran input, dan batch yang sama. Setiap hasil mencatat scope pengukuran.

In [ ]:
def ordered_names(names):
    """Pertahankan identitas dan urutan indeks kelas."""
    if isinstance(names, dict):
        indexed = {int(k): str(v) for k, v in names.items()}
        if sorted(indexed) != list(range(len(indexed))):
            raise ValueError("Indeks kelas wajib berurutan mulai dari nol.")
        return [indexed[i] for i in range(len(indexed))]
    return [str(value) for value in names]


def assert_class_match(model_names, dataset_names):
    """Jangan remap kelas diam-diam untuk membuat evaluasi berjalan."""
    if ordered_names(model_names) != ordered_names(dataset_names):
        raise ValueError(f"Urutan/nama kelas berbeda: model={ordered_names(model_names)}, dataset={ordered_names(dataset_names)}")


def architecture_audit(core, model_variant="n"):
    """Bangun allowlist hidden channel berdasarkan tipe modul dan hubungan konsumen."""
    if not isinstance(core, DetectionModel):
        raise TypeError("Checkpoint bukan DetectionModel Ultralytics.")
    heads = [(name, module) for name, module in core.named_modules() if isinstance(module, Detect)]
    if len(heads) != 1:
        raise ValueError("Diperlukan tepat satu Detect head.")
    head_name, head = heads[0]
    cfg = core.yaml
    first_conv = next(module for module in core.modules() if isinstance(module, torch.nn.Conv2d))
    if "yolo26" not in str(cfg.get("yaml_file", "")).lower() or cfg.get("reg_max") != 1 or first_conv.out_channels != {"n": 16, "s": 32}[model_variant]:
        raise ValueError("Metadata/struktur tidak cocok dengan YOLO26 N/S standar; audit manual diperlukan.")
    if cfg.get("scale") not in (None, "", model_variant):
        raise ValueError("Checkpoint bukan skala model yang dipilih.")
    branches = ("cv2", "cv3", "one2one_cv2", "one2one_cv3")
    if any(getattr(head, attr, None) is None or len(getattr(head, attr)) != head.nl for attr in branches):
        raise ValueError("Kedua cabang head wajib tersedia; checkpoint fused/head berubah ditolak.")
    names = ordered_names(core.names)
    if len(names) != head.nc or head.no != head.nc + 4 * head.reg_max:
        raise ValueError("Metadata kelas/dimensi Detect tidak konsisten.")
    modules = dict(core.named_modules())
    attention_prefixes = [name for name, m in modules.items() if isinstance(m, (C2PSA, Attention, AAttn))]
    allowed = {}
    for name, module in modules.items():
        if not isinstance(module, Bottleneck) or name.startswith(head_name + "."):
            continue
        if any(name == prefix or name.startswith(prefix + ".") for prefix in attention_prefixes):
            continue
        producer, consumer = module.cv1.conv, module.cv2.conv
        bn = getattr(module.cv1, "bn", None)
        if not isinstance(bn, torch.nn.BatchNorm2d) or producer.groups != 1 or consumer.groups != 1:
            raise ValueError(f"Bottleneck {name} tidak mendukung kebijakan hidden-channel konservatif.")
        if producer.out_channels != consumer.in_channels or producer.out_channels != bn.num_features:
            raise ValueError(f"Dimensi hidden channel {name} tidak konsisten.")
        allowed[name + ".cv1.conv"] = {"root": producer, "bn": bn, "consumer": consumer,
                                        "initial_channels": producer.out_channels, "block": name}
    if not allowed:
        raise ValueError("Tidak ditemukan hidden-channel group yang eligible.")
    backbone_count = len(cfg["backbone"])
    stages = []
    for i, module in enumerate(core.model):
        stages.append({"index": i, "type": type(module).__name__, "from": module.f,
                       "region": "detection_head" if module is head else ("backbone" if i < backbone_count else "neck"),
                       "parameters": sum(p.numel() for p in module.parameters())})
    inventory = [{"name": name, "type": type(module).__name__,
                  "residual_add": bool(getattr(module, "add", False))}
                 for name, module in modules.items()
                 if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d, Bottleneck, C2PSA, Attention, AAttn))
                 or type(module).__name__ in {"Concat", "C3k2", "SPPF"}]
    protection = []
    for name, module in modules.items():
        if isinstance(module, torch.nn.Conv2d) and name not in allowed:
            reason = "head one2many/one2one utuh" if name.startswith(head_name + ".") else "dimensi CSP/residual/attention/antartahap dilindungi"
            protection.append({"name": name, "reason": reason,
                               "protected_dimension": "all" if name.startswith(head_name + ".") else "out_channels",
                               "dependent_input_pruning_allowed": any(module is entry["consumer"] for entry in allowed.values())})
    return {"head": head, "head_name": head_name, "eligible": allowed, "modules": modules,
            "report": {"stages": stages, "module_inventory": inventory, "protected_layers": protection,
                       "eligible_layers": list(allowed), "head_feature_sources": list(head.f),
                       "class_names": names, "nc": head.nc, "reg_max": head.reg_max, "nl": head.nl,
                       "end2end": head.end2end, "unknown_policy": "fail explicitly"}}


def raw_branches(output):
    """Ambil raw kedua cabang tanpa mengganti forward atau algoritme deteksi."""
    if not isinstance(output, tuple) or len(output) != 2 or not isinstance(output[1], dict):
        raise ValueError("Output eval YOLO26 harus (decoded, raw_predictions).")
    branches = output[1]
    if set(branches) != {"one2many", "one2one"}:
        raise ValueError(f"Cabang YOLO26 tidak lengkap: {list(branches)}")
    return branches


def trace_outputs(output):
    """Root autograd mencakup boxes/scores kedua cabang, tanpa top-k/NMS."""
    branches = raw_branches(output)
    return [branches[branch][key] for branch in ("one2many", "one2one") for key in ("boxes", "scores")]


def finite_tensors(value):
    """Validasi semua tensor, termasuk feature maps dalam output nested."""
    if isinstance(value, torch.Tensor):
        if not torch.isfinite(value).all():
            raise ValueError("Output model mengandung NaN/Inf.")
    elif isinstance(value, dict):
        for item in value.values():
            finite_tensors(item)
    elif isinstance(value, (tuple, list)):
        for item in value:
            finite_tensors(item)


def forward_snapshot(core, image):
    """Forward FP32, cek kontrak raw/decoded, simpan tensor kecil untuk restore parity."""
    core.eval()
    with torch.no_grad():
        output = core(image)
    finite_tensors(output)
    branches = raw_branches(output)
    head = core.model[-1]
    snapshot = {"decoded": output[0].detach().cpu().clone()}
    anchors = None
    for branch in ("one2many", "one2one"):
        boxes, scores = branches[branch]["boxes"], branches[branch]["scores"]
        if boxes.ndim != 3 or scores.ndim != 3 or boxes.shape[:2] != (image.shape[0], 4 * head.reg_max):
            raise ValueError(f"Bentuk boxes {branch} invalid: {tuple(boxes.shape)}")
        if scores.shape[:2] != (image.shape[0], head.nc) or boxes.shape[2] != scores.shape[2]:
            raise ValueError(f"Bentuk scores {branch} tidak cocok dengan jumlah kelas.")
        if anchors is not None and anchors != boxes.shape[2]:
            raise ValueError("Jumlah anchor kedua cabang berbeda.")
        anchors = boxes.shape[2]
        snapshot[branch + "/boxes"] = boxes.detach().cpu().clone()
        snapshot[branch + "/scores"] = scores.detach().cpu().clone()
    decoded = snapshot["decoded"]
    if head.end2end:
        if decoded.ndim != 3 or decoded.shape[0] != image.shape[0] or decoded.shape[-1] != 6:
            raise ValueError("Output end-to-end bukan [batch, detections, 6].")
        ids = decoded[..., 5]
        if ((ids < 0) | (ids >= head.nc) | (ids != ids.floor())).any():
            raise ValueError("Indeks kelas pada output deteksi invalid.")
    elif decoded.ndim != 3 or decoded.shape[:2] != (image.shape[0], head.nc + 4):
        raise ValueError("Output one-to-many tidak memiliki dimensi nc+4.")
    return snapshot


def model_measurements(core, imgsz):
    """Hitung Conv/BN dan attention matmul tanpa menghilangkan child Conv attention."""
    parameters = sum(p.numel() for p in core.parameters())
    copy = deepcopy(core).eval()
    parameter = next(copy.parameters())
    sample = torch.zeros(1, 3, imgsz, imgsz, device=parameter.device, dtype=torch.float32)
    attention_macs = []
    def count_attention(module, inputs, output):
        counter = SimpleNamespace(total_ops=0, num_heads=module.num_heads, head_dim=module.head_dim,
                                  key_dim=getattr(module, "key_dim", module.head_dim), area=getattr(module, "area", 1))
        _attention_ops(counter, inputs, output)
        attention_macs.append(float(counter.total_ops))
    handles = [module.register_forward_hook(count_attention) for module in copy.modules()
               if isinstance(module, (Attention, AAttn))]
    try:
        with torch.no_grad():
            # A custom Attention handler in classic THOP would hide its child Conv/BN counts.
            macs, _ = thop_profile(copy, inputs=(sample,), verbose=False)
    finally:
        for handle in handles:
            handle.remove()
    gflops = (float(macs) + sum(attention_macs)) * 2 / 1e9
    if not math.isfinite(gflops) or gflops <= 0:
        raise RuntimeError("Profiling GFLOPs gagal/bernilai nol; cek thop dan source dependency.")
    return {"parameters": parameters, "gflops": gflops, "precision": "FP32", "fusion": False}


def head_snapshot(head):
    """Salin parameter/buffer serta kontrak dimensi seluruh head yang dilindungi."""
    return {"state": {key: value.detach().cpu().clone() for key, value in head.state_dict().items()},
            "contract": (head.nc, head.no, head.nl, head.reg_max, head.end2end)}


def assert_head_unchanged(head, before):
    """Pruning tersembunyi tidak boleh menyentuh bobot atau dimensi head."""
    after = head_snapshot(head)
    if after["contract"] != before["contract"] or after["state"].keys() != before["state"].keys():
        raise ValueError("Kontrak/struktur detection head berubah.")
    if any(not torch.equal(after["state"][key], value) for key, value in before["state"].items()):
        raise ValueError("Parameter/buffer detection head berubah.")

In [ ]:
def assert_layer_consistency(core):
    """Dimensi atribut modul harus sesuai dengan tensor parameter/buffer aktual."""
    for name, module in core.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            expected = (module.out_channels, module.in_channels // module.groups, *module.kernel_size)
            if tuple(module.weight.shape) != expected or module.in_channels % module.groups or module.out_channels % module.groups:
                raise ValueError(f"Bentuk Conv invalid pada {name}.")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if any(t.numel() != module.num_features for t in (module.weight, module.bias, module.running_mean, module.running_var)):
                raise ValueError(f"Bentuk BatchNorm invalid pada {name}.")

In [ ]:
RELOAD_SCRIPT = r'''
import json, sys
from pathlib import Path
import numpy as np
import torch
from ultralytics import YOLO

checkpoint, reference, result, device = sys.argv[1:]
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
payload = torch.load(reference, map_location="cpu", weights_only=True)
wrapper = YOLO(checkpoint, task="detect")
core = wrapper.model.to(device).float().eval()
with torch.no_grad():
    output = core(payload["image"].to(device))
snapshot = {"decoded": output[0].detach().cpu()}
for branch in ("one2many", "one2one"):
    for key in ("boxes", "scores"):
        snapshot[branch + "/" + key] = output[1][branch][key].detach().cpu()
for key, expected in payload["expected"].items():
    if not torch.isfinite(snapshot[key]).all():
        raise RuntimeError("Non-finite output after reload: " + key)
    torch.testing.assert_close(snapshot[key], expected, rtol=1e-4, atol=1e-5)
if sum(p.numel() for p in core.parameters()) != payload["parameters"]:
    raise RuntimeError("Parameter count changed after reload")
if not all(p.dtype == torch.float32 for p in core.parameters()):
    raise RuntimeError("Reload is not FP32")
torch.save(snapshot, result)
# Fresh wrapper protects the raw unfused model from inference backend fusion.
predictor = YOLO(checkpoint, task="detect")
size = payload["image"].shape[-1]
predictions = predictor.predict(source=np.zeros((size, size, 3), dtype=np.uint8),
    imgsz=size, device=device, conf=0.001, iou=0.7, max_det=300,
    quantize=None, nms=None, verbose=False, save=False)
if len(predictions) != 1 or predictions[0].boxes is None:
    raise RuntimeError("No valid detection Results after reload")
boxes = predictions[0].boxes.data.detach().cpu()
if boxes.ndim != 2 or boxes.shape[1] != 6 or not torch.isfinite(boxes).all():
    raise RuntimeError("Invalid detection tensor")
nc = len(core.names)
if boxes.numel() and (((boxes[:, 5] < 0) | (boxes[:, 5] >= nc) | (boxes[:, 5] != boxes[:, 5].floor())).any()
                     or ((boxes[:, 4] < 0) | (boxes[:, 4] > 1)).any()):
    raise RuntimeError("Invalid detection class/confidence")
Path(result).with_suffix(".json").write_text(json.dumps({"prediction_count": len(boxes),
    "parameters": payload["parameters"], "direct_yolo_load": True,
    "raw_parity": True, "prediction_shape": list(boxes.shape)}), encoding="utf-8")
'''


def save_fp32_checkpoint(core, path, provenance):
    """Simpan arsitektur hasil pruning memakai format model-object Ultralytics."""
    model = deepcopy(core).cpu().float().eval().requires_grad_(False)
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()
    model.args = {"task": "detect", "imgsz": IMGSZ, "data": str(DATA_YAML), "nms": None, "quantize": None}
    torch.save({"model": model, "ema": None, "optimizer": None, "epoch": -1,
                "train_args": dict(model.args), "version": ultralytics.__version__,
                "date": datetime.now(timezone.utc).isoformat(), "lamp_format_version": 1,
                "lamp_provenance": provenance, "recovery_enabled": False}, path)
    return {"path": str(Path(path).resolve()), "sha256": sha256_file(path), "file_bytes": Path(path).stat().st_size}


def verify_fresh_reload(checkpoint, core, image, output_dir, *, timeout=300):
    """Restore di interpreter baru; exit code/error tidak boleh disembunyikan."""
    output_dir = Path(output_dir).resolve()
    script = output_dir / "verify_reload.py"
    reference = output_dir / "reload_reference.pt"
    result = output_dir / "reload_output.pt"
    script.write_text(RELOAD_SCRIPT, encoding="utf-8")
    expected = forward_snapshot(core, image)
    torch.save({"image": image.detach().cpu(), "expected": expected,
                "parameters": sum(p.numel() for p in core.parameters())}, reference)
    child_env = dict(os.environ)
    child_env.pop("PYTHONPATH", None)
    child_env["WANDB_MODE"] = "disabled"
    child_env["YOLO_AUTOINSTALL"] = "false"
    completed = subprocess.run([sys.executable, "-B", str(script), str(Path(checkpoint).resolve()),
                                str(reference), str(result), str(next(core.parameters()).device)],
                               cwd=output_dir, env=child_env, capture_output=True, text=True,
                               encoding="utf-8", errors="replace", timeout=timeout)
    (output_dir / "reload_stdout.txt").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "reload_stderr.txt").write_text(completed.stderr, encoding="utf-8")
    if completed.returncode:
        raise RuntimeError(f"Fresh-process reload gagal (exit {completed.returncode}): {completed.stderr[-6000:]}")
    actual = torch.load(result, map_location="cpu", weights_only=True)
    for key in expected:
        torch.testing.assert_close(actual[key], expected[key], rtol=1e-4, atol=1e-5)
    report = json.loads(result.with_suffix(".json").read_text(encoding="utf-8"))
    report["max_absolute_difference"] = {key: float((actual[key] - expected[key]).abs().max()) for key in expected}
    report["process"] = "fresh Python, no repository PYTHONPATH"
    return report

In [ ]:
def prepare_validation_dataset(data_yaml, dataset_root, output_dir):
    """Validasi dataset lokal tanpa download atau normalisasi label otomatis."""
    source = Path(data_yaml).expanduser()
    if not source.is_file():
        return None
    raw = yaml.safe_load(source.read_text(encoding="utf-8"))
    if not isinstance(raw, dict) or not raw.get("val") or not raw.get("names"):
        raise ValueError("DATA_YAML tersedia tetapi tidak berisi val/names yang valid.")
    raw.pop("download", None)
    root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else source.resolve().parent
    if not root.is_dir():
        raise FileNotFoundError(f"DATASET_ROOT tidak tersedia: {root}")
    raw["path"] = str(root)
    copy_path = Path(output_dir) / "evaluation_data.yaml"
    copy_path.write_text(yaml.safe_dump(raw, sort_keys=False, allow_unicode=True), encoding="utf-8")
    info = check_det_dataset(str(copy_path), autodownload=False)
    splits = info["val"] if isinstance(info["val"], list) else [info["val"]]
    if any(not Path(split).exists() for split in splits):
        raise FileNotFoundError("Validation split tidak tersedia.")
    return {"yaml": str(copy_path.resolve()), "info": info, "source_yaml": str(source.resolve()),
            "source_yaml_sha256": sha256_file(source), "evaluation_yaml_sha256": sha256_file(copy_path),
            "val_paths": [str(split) for split in splits], "class_names": ordered_names(info["names"])}


def validation_arguments(data_yaml, output_dir, label):
    """Satu sumber pengaturan evaluasi untuk baseline dan pruned model."""
    return {"data": str(data_yaml), "split": "val", "imgsz": IMGSZ, "batch": BATCH,
            "workers": WORKERS, "device": str(TARGET_DEVICE), "conf": 0.001,
            "iou": 0.7, "max_det": 300, "quantize": None, "nms": None,
            "augment": False, "plots": False, "verbose": False, "save_json": False,
            "project": str(output_dir), "name": label, "exist_ok": False}


def evaluate_checkpoint(path, dataset, output_dir, label):
    """Evaluasi wrapper baru agar fusion validator tidak mengubah model pengukuran."""
    wrapper = YOLO(str(path), task="detect")
    assert_class_match(wrapper.names, dataset["info"]["names"])
    metric = wrapper.val(**validation_arguments(dataset["yaml"], output_dir, label))
    box = metric.box
    per_class = [{"class_id": i, "name": name, "ap50": None, "ap50_95": None}
                 for i, name in enumerate(dataset["class_names"])]
    for row, class_id in enumerate(box.ap_class_index):
        per_class[int(class_id)].update({"ap50": float(box.ap50[row]), "ap50_95": float(box.ap[row])})
    result = {"map50_95": float(box.map), "map50": float(box.map50),
              "precision": float(box.mp), "recall": float(box.mr), "per_class": per_class,
              "split": "val", "precision_mode": "FP32", "nms": None}
    if any(not math.isfinite(result[key]) for key in ("map50_95", "map50", "precision", "recall")):
        raise ValueError("Metrik evaluasi mengandung NaN/Inf.")
    return result


def cuda_latency(core, image, warmup=50, repeats=200):
    """Latency forward model batch satu; tidak mencakup preprocessing/NMS/disk IO."""
    if image.device.type != "cuda":
        return {"status": "NOT_RUN", "reason": "CUDA tidak tersedia; tidak mengklaim benchmark GPU."}
    core = deepcopy(core).eval().requires_grad_(False)
    samples = []
    with torch.inference_mode():
        for _ in range(warmup):
            core(image)
        torch.cuda.synchronize(image.device)
        for _ in range(repeats):
            torch.cuda.synchronize(image.device)
            started = time.perf_counter()
            core(image)
            torch.cuda.synchronize(image.device)
            samples.append((time.perf_counter() - started) * 1000)
    return {"status": "PASS", "median_ms": float(np.median(samples)), "mean_ms": float(np.mean(samples)),
            "p05_ms": float(np.percentile(samples, 5)), "p95_ms": float(np.percentile(samples, 95)),
            "warmup": warmup, "repeats": repeats, "batch": 1, "imgsz": list(image.shape[-2:]),
            "precision": "FP32", "device": str(image.device), "fusion": False, "scope": "raw model forward"}

In [ ]:
"""Validation declarations embedded verbatim in standalone LAMP V2 notebooks.

The source never imports repository modules at notebook runtime. Numerical exports
come from the actual Ultralytics validator; missing measurements remain null.
"""
# ruff: noqa: F821 -- Notebook configuration/helpers provide these runtime globals.


def validation_arguments(data_yaml, output_dir, label):
    """Identical FP32 protocol for baseline/pruned; plots also collect confusion counts."""
    return {"data": str(data_yaml), "split": "val", "imgsz": IMGSZ, "batch": BATCH,
            "workers": WORKERS, "device": str(TARGET_DEVICE), "conf": 0.001,
            "iou": 0.7, "max_det": 300, "quantize": None, "nms": None,
            "augment": False, "plots": True, "verbose": False, "save_json": False,
            "project": str(output_dir), "name": label, "exist_ok": False}


def _validation_number(value):
    """JSON scalar: unsupported/nonfinite auxiliary metrics are unavailable, not zero."""
    if value is None:
        return None
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return number if math.isfinite(number) else None


def _validation_array(value):
    import numpy as np

    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=float)


def _validation_json_array(value):
    import numpy as np

    array = _validation_array(value)
    return np.where(np.isfinite(array), array, None).tolist()


def _validation_per_class(metric, class_names):
    """Map SDK result rows via ap_class_index, never by names or array position."""
    import numpy as np

    box = metric.box
    support = getattr(metric, "nt_per_class", None)
    if support is not None:
        support = _validation_array(support)
        if (support.shape != (len(class_names),) or not np.isfinite(support).all()
                or (support < 0).any() or not np.equal(support, np.floor(support)).all()):
            raise ValueError("Support kelas SDK tidak valid atau tidak cocok dengan jumlah kelas.")
    rows = [{"class_id": i, "name": name, "ap50": None, "ap50_95": None,
             "precision": None, "recall": None, "f1": None,
             "support": int(support[i]) if support is not None else None}
            for i, name in enumerate(class_names)]
    indices = list(getattr(box, "ap_class_index", []))
    validated_indices = []
    for value in indices:
        scalar = _validation_number(value)
        if scalar is None or scalar != int(scalar) or not 0 <= int(scalar) < len(rows):
            raise ValueError("Indeks kelas SDK tidak valid.")
        validated_indices.append(int(scalar))
    if len(set(validated_indices)) != len(validated_indices):
        raise ValueError("Indeks kelas SDK duplikat.")
    for offset, class_id in enumerate(validated_indices):
        for key, attribute in (("ap50", "ap50"), ("ap50_95", "ap"),
                               ("precision", "p"), ("recall", "r"), ("f1", "f1")):
            values = getattr(box, attribute, None)
            if values is not None and len(values) > offset:
                rows[class_id][key] = _validation_number(values[offset])
    return rows, validated_indices


def _validation_curves(metric, class_names, class_ids):
    """Keep SDK axes and class mapping, replacing nonfinite samples with JSON null."""
    import numpy as np

    try:
        curve_values = metric.curves_results
    except (AttributeError, TypeError, ValueError) as error:
        return {"status": "UNAVAILABLE", "reason": str(error), "curves": []}
    names = getattr(metric, "curves", [])
    records = []
    for index, values in enumerate(curve_values):
        record = {"name": names[index] if index < len(names) else f"SDK curve {index}",
                  "class_ids": list(class_ids), "class_names": [class_names[i] for i in class_ids],
                  "status": "UNAVAILABLE", "x": [], "y": []}
        try:
            x_value, y_value, x_label, y_label = values
            x, y = _validation_array(x_value), _validation_array(y_value)
            record.update({"x_label": str(x_label), "y_label": str(y_label)})
            if y.ndim == 1 and len(class_ids) == 1:
                y = y[None, :]
            if (not class_ids or x.ndim != 1 or not x.size or y.ndim != 2
                    or y.shape != (len(class_ids), x.size)):
                record["reason"] = "Dimensi kurva tidak memiliki pemetaan kelas SDK yang valid."
            else:
                record.update({"x": _validation_json_array(x), "y": _validation_json_array(y),
                               "status": "MEASURED" if np.isfinite(x).all() and np.isfinite(y).all()
                               else "PARTIAL"})
                if record["status"] == "PARTIAL":
                    record["reason"] = "Sampel SDK NaN/Inf disimpan sebagai null."
        except (TypeError, ValueError, OverflowError) as error:
            record["reason"] = str(error)
        records.append(record)
    states = {item["status"] for item in records}
    status = "MEASURED" if states == {"MEASURED"} else (
        "PARTIAL" if states & {"MEASURED", "PARTIAL"} else "UNAVAILABLE")
    return {"status": status, "curves": records}


def _validation_confusion_matrix(validator, class_names, arguments):
    """Export actual counts and the distinct matching/NMS IoU thresholds."""
    import inspect

    import numpy as np

    matrix_object = getattr(validator, "confusion_matrix", None)
    result = {"status": "UNAVAILABLE", "labels": [*class_names, "background"],
              "orientation": "rows=predicted, columns=true", "matrix": None,
              "normalized_by_true": None, "confidence_threshold": None,
              "matching_iou_threshold": None, "validation_iou": arguments["iou"],
              "normalization": "Divide each true-class column by its sum; empty columns stay zero."}
    if matrix_object is None or getattr(matrix_object, "matrix", None) is None:
        result["reason"] = "Validator tidak menyediakan confusion matrix."
        return result
    matrix = _validation_array(matrix_object.matrix)
    expected = len(class_names) + 1
    if (matrix.shape != (expected, expected) or not np.isfinite(matrix).all() or (matrix < 0).any()):
        raise ValueError("Confusion matrix SDK tidak valid atau class order berbeda.")
    threshold = _validation_number(getattr(validator, "confusion_matrix_conf", None))
    matching_iou = None
    try:
        parameter = inspect.signature(matrix_object.process_batch).parameters.get("iou_thres")
        if parameter is not None and parameter.default is not inspect.Parameter.empty:
            matching_iou = _validation_number(parameter.default)
    except (AttributeError, TypeError, ValueError):
        pass
    totals = matrix.sum(axis=0, keepdims=True)
    normalized = np.divide(matrix, totals, out=np.zeros_like(matrix), where=totals != 0)
    result.update({"status": "MEASURED", "matrix": matrix.tolist(),
                   "normalized_by_true": normalized.tolist(),
                   "zero_support_columns": [int(i) for i in np.flatnonzero(totals[0] == 0)],
                   "confidence_threshold": threshold, "matching_iou_threshold": matching_iou,
                   "matching_iou_source": "Signature default of actual ConfusionMatrix.process_batch; validator calls without iou_thres.",
                   "confidence_source": "Actual DetectionValidator.confusion_matrix_conf"})
    return result


def _validation_metric_semantics(metric):
    """Record the SDK best-F1 reporting point without confusing it with CM confidence."""
    import numpy as np
    from ultralytics.utils.metrics import smooth

    result = {"precision_recall_f1": "SDK values at the global confidence maximizing smoothed mean F1 (IoU=0.50).",
              "reporting_confidence": None,
              "reporting_confidence_source": "SDK px[smooth(f1_curve.mean(0), 0.1).argmax()]",
              "map_iou_thresholds": [round(0.5 + 0.05 * i, 2) for i in range(10)],
              "absent_class": "Support 0 denotes no ground truth; class metrics without an SDK row remain null.",
              "confusion_matrix": "A separate operating point: see confidence_threshold and matching_iou_threshold."}
    box = metric.box
    try:
        px, f1 = _validation_array(box.px), _validation_array(box.f1_curve)
        if (px.ndim == 1 and f1.ndim == 2 and f1.shape[0] > 0 and f1.shape[1] == len(px)
                and np.isfinite(px).all() and np.isfinite(f1).all()):
            result["reporting_confidence"] = float(px[int(smooth(f1.mean(0), 0.1).argmax())])
    except (AttributeError, TypeError, ValueError):
        pass
    return result


def evaluate_checkpoint(path, dataset, output_dir, label):
    """Evaluate a fresh wrapper; preserve SDK metrics, curves and populated confusion counts."""
    wrapper = YOLO(str(path), task="detect")
    assert_class_match(wrapper.names, dataset["info"]["names"])
    class_names = list(dataset["class_names"])
    assert_class_match(wrapper.names, class_names)
    arguments = gflops_validation_arguments(dataset, output_dir, label)
    captured = {}
    capture_validator = None
    if hasattr(wrapper, "add_callback"):
        # Model.val() keeps its validator local in Ultralytics 8.4.155.
        # Read the actual instance at callback time rather than assuming an attribute.
        def capture_validator(validator):
            captured["validator"] = validator

        wrapper.add_callback("on_val_end", capture_validator)
    try:
        with gflops_require_complete_nms(), gflops_disable_label_cache():
            metric = wrapper.val(**arguments)
    finally:
        # Avoid a callback -> capture -> validator cycle retaining GPU allocations
        # between baseline/ratio runs. Removing it also affects the shared callbacks.
        callbacks = getattr(wrapper, "callbacks", {}).get("on_val_end", [])
        if capture_validator in callbacks:
            callbacks.remove(capture_validator)
    box = metric.box
    per_class, class_ids = _validation_per_class(metric, class_names)
    result = {"map50_95": _validation_number(box.map), "map50": _validation_number(box.map50),
              "precision": _validation_number(box.mp), "recall": _validation_number(box.mr),
              "per_class": per_class, "split": "val", "precision_mode": "FP32", "nms": None}
    if any(result[key] is None for key in ("map50_95", "map50", "precision", "recall")):
        raise ValueError("Metrik evaluasi mengandung NaN/Inf atau tidak tersedia.")
    result["curves"] = _validation_curves(metric, class_names, class_ids)
    validator = captured.get("validator", getattr(wrapper, "validator", None))
    result.update(gflops_validation_metadata(validator, arguments, dataset))
    result["confusion_matrix"] = _validation_confusion_matrix(validator, class_names, arguments)
    result["metric_semantics"] = _validation_metric_semantics(metric)
    export_dir = Path(output_dir) / label
    export_dir.mkdir(parents=True, exist_ok=True)
    write_csv(export_dir / "per_class_metrics.csv", per_class)
    write_json(export_dir / "curves.json", result["curves"])
    write_json(export_dir / "confusion_matrix.json", result["confusion_matrix"])
    write_json(export_dir / "metric_semantics.json", result["metric_semantics"])
    result["exports"] = {name: str(export_dir / name) for name in (
        "per_class_metrics.csv", "curves.json", "confusion_matrix.json", "metric_semantics.json")}
    return result

In [ ]:
"""Dataset and student-selection declarations embedded in notebook 05."""
# ruff: noqa: F821 -- Notebook runtime supplies the V2 evaluator and helpers.

import hashlib
import json
import logging
import math
from contextlib import contextmanager
from pathlib import Path
from unittest.mock import patch

POLTRADA_V3_ID = "ilhamrafiqin/poltrada-v3-yolo26-clean"
POLTRADA_V3_MOUNT = Path("/kaggle/input/datasets") / POLTRADA_V3_ID


def _gflops_split_images(root, entries):
    """Resolve directories and image lists using dataset-relative identities."""
    from ultralytics.data.utils import IMG_FORMATS

    if isinstance(entries, str):
        entries = [entries]
    if not isinstance(entries, list) or not entries:
        raise ValueError("Split Poltrada harus berupa direktori atau daftar gambar lokal.")
    images = []
    for entry in entries:
        if not isinstance(entry, str) or not entry.strip():
            raise ValueError("Path split kosong atau tidak valid.")
        path = (root / entry).resolve()
        if (not path.exists() or not path.is_relative_to(root)) and entry.startswith("../"):
            # Roboflow exports commonly use ../valid/images from data.yaml.
            path = (root / entry.removeprefix("../")).resolve()
        if not path.is_relative_to(root):
            raise ValueError(f"Path split berada di luar DATASET_ROOT: {path}")
        if path.is_dir():
            files = (item for item in path.rglob("*") if item.is_file())
        elif path.is_file() and path.suffix.lower() == ".txt":
            files = (path.parent / line.strip() for line in path.read_text(encoding="utf-8-sig").splitlines()
                     if line.strip())
        else:
            raise FileNotFoundError(f"Direktori atau daftar gambar split tidak tersedia: {path}")
        for image in files:
            if image.suffix.lower().removeprefix(".") not in IMG_FORMATS:
                continue
            image = image.resolve()
            if not image.is_file() or not image.is_relative_to(root):
                raise ValueError(f"Gambar tidak tersedia atau berada di luar DATASET_ROOT: {image}")
            images.append(image)
    if not images:
        raise ValueError("Split Poltrada tidak berisi gambar yang dapat dievaluasi.")
    if len(set(images)) != len(images):
        raise ValueError("Split Poltrada memuat path gambar duplikat.")
    return sorted(images, key=lambda image: image.relative_to(root).as_posix())


def _gflops_fingerprint(root, images, split, names):
    """Hash every selected image and label; missing and empty labels stay distinct."""
    from ultralytics.data.utils import img2label_paths

    records = []
    for image, label in zip(images, img2label_paths([str(image) for image in images]), strict=True):
        label = Path(label).resolve()
        if not label.is_relative_to(root):
            raise ValueError(f"Label berada di luar DATASET_ROOT: {label}")
        records.append({"image": image.relative_to(root).as_posix(), "image_sha256": sha256_file(image),
                        "label": label.relative_to(root).as_posix(),
                        "label_sha256": sha256_file(label) if label.is_file() else None})
    # Encoding and sorted path order make the digest independent of the mount point.
    payload = json.dumps({"split": split, "class_names": names, "records": records},
                         ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest(), records


def gflops_prepare_dataset(mode, data_yaml, dataset_root, suite_dir, *, seed,
                           expected_fingerprint=None, dataset_id=None):
    """Freeze the input set before evaluating any model in the suite."""
    import yaml

    if mode not in {"smoke", "full_val", "test"}:
        raise ValueError("Mode evaluasi harus smoke, full_val, atau test.")
    if expected_fingerprint is not None:
        if (not isinstance(expected_fingerprint, str) or len(expected_fingerprint) != 64
                or any(char not in "0123456789abcdefABCDEF" for char in expected_fingerprint)):
            raise ValueError("Fingerprint full val harus berupa SHA-256 heksadesimal.")
        expected_fingerprint = expected_fingerprint.lower()
    source = Path(data_yaml).expanduser().resolve()
    if not source.is_file():
        return None
    root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else source.parent
    if not root.is_dir():
        raise FileNotFoundError(f"DATASET_ROOT tidak tersedia: {root}")
    raw = yaml.safe_load(source.read_text(encoding="utf-8-sig"))
    if not isinstance(raw, dict) or not isinstance(raw.get("names"), (dict, list)) or not raw["names"]:
        raise ValueError("DATA_YAML harus memuat names Poltrada yang berurutan.")
    class_names = ordered_names(raw["names"])
    if raw.get("nc", len(class_names)) != len(class_names):
        raise ValueError("nc di DATA_YAML berbeda dari jumlah names.")

    split = "test" if mode == "test" else "val"
    val_images = _gflops_split_images(root, raw.get("val"))
    full_val_fingerprint = None
    if mode != "smoke" or expected_fingerprint is not None:
        full_val_fingerprint, val_records = _gflops_fingerprint(root, val_images, "val", class_names)
        if expected_fingerprint is not None and full_val_fingerprint != expected_fingerprint:
            raise ValueError("Fingerprint full val tidak cocok dengan konfigurasi yang dikunci.")

    official_mount = root == POLTRADA_V3_MOUNT.resolve() and source.is_relative_to(root)
    verified_copy = dataset_id == POLTRADA_V3_ID and expected_fingerprint is not None
    provenance_status = "VERIFIED" if official_mount or verified_copy else "UNVERIFIED"
    provenance_basis = ("KAGGLE_MOUNT" if official_mount else
                        "EXPECTED_FULL_VAL_FINGERPRINT" if verified_copy else "USER_PATH_ONLY")
    if mode == "test":
        if not raw.get("test") or not raw.get("train"):
            return None
        images = _gflops_split_images(root, raw["test"])
        fingerprint, records = _gflops_fingerprint(root, images, split, class_names)
        test_hashes = {row["image_sha256"] for row in records}
        if any(row["image_sha256"] in test_hashes for row in val_records):
            raise ValueError("Split test berbagi konten gambar dengan full val.")
        train_images = _gflops_split_images(root, raw["train"])
        if any(sha256_file(image) in test_hashes for image in train_images):
            raise ValueError("Split test berbagi konten gambar dengan train.")
    elif mode == "smoke":
        ranked = sorted(val_images, key=lambda image: (
            hashlib.sha256(f"{seed}:{image.relative_to(root).as_posix()}".encode()).hexdigest(),
            image.relative_to(root).as_posix()))
        images = sorted(ranked[:128], key=lambda image: image.relative_to(root).as_posix())
        fingerprint, records = _gflops_fingerprint(root, images, split, class_names)
    else:
        images = val_images
        fingerprint, records = full_val_fingerprint, val_records
    if any(image.with_suffix(".npy").is_file() for image in images):
        raise ValueError("Cache gambar .npy berada di samping citra evaluasi. Gunakan salinan dataset tanpa cache.")

    folder = Path(suite_dir).resolve()
    folder.mkdir(parents=True, exist_ok=True)
    paths = folder / f"evaluation_{mode}_images.txt"
    paths.write_text("".join(f"{image.as_posix()}\n" for image in images), encoding="utf-8")
    # This descriptor permits no download or implicit split fallback. An unused train
    # key is required by Ultralytics even though this notebook only calls val().
    descriptor = {"path": str(root), "train": str(paths), "val": str(paths),
                  "names": class_names, "nc": len(class_names)}
    if split == "test":
        descriptor["test"] = str(paths)
    evaluation_yaml = folder / f"evaluation_{mode}_data.yaml"
    evaluation_yaml.write_text(yaml.safe_dump(descriptor, sort_keys=False, allow_unicode=True), encoding="utf-8")
    manifest = {"mode": mode, "split": split, "seed": seed, "image_count": len(images),
                "full_val_image_count": len(val_images), "fingerprint": fingerprint,
                "full_val_fingerprint": full_val_fingerprint, "provenance_status": provenance_status,
                "provenance_basis": provenance_basis,
                "dataset_id": POLTRADA_V3_ID if official_mount else dataset_id,
                "source_yaml": str(source), "source_yaml_sha256": sha256_file(source),
                "evaluation_yaml_sha256": sha256_file(evaluation_yaml), "class_names": class_names,
                "test_content_disjoint_from_train_val": True if mode == "test" else None}
    manifest_path = folder / f"evaluation_{mode}_manifest.json"
    write_json(manifest_path, {**manifest, "files": records})
    return {**manifest, "manifest_path": str(manifest_path), "yaml": str(evaluation_yaml),
            "info": descriptor, "val_paths": [str(paths)], "image_files": [str(image) for image in images]}


def gflops_validation_arguments(dataset, output_dir, label):
    """Preserve notebook 03 settings and explicitly select the prepared split."""
    arguments = validation_arguments(dataset["yaml"], output_dir, label)
    arguments.update({"split": dataset["split"], "fraction": 1.0})
    return arguments


@contextmanager
def gflops_disable_label_cache():
    """Force label parsing from the fingerprinted files without changing the input dataset."""
    from ultralytics.data import dataset as sdk_dataset

    def set_in_memory_version(_prefix, _path, records, version):
        # get_labels() expects the writer's in-memory version field.
        records["version"] = version

    with (patch.object(sdk_dataset, "load_dataset_cache_file", side_effect=FileNotFoundError),
          patch.object(sdk_dataset, "save_dataset_cache_file", side_effect=set_in_memory_version)):
        yield


@contextmanager
def gflops_require_complete_nms():
    """Fail validation if the SDK stops processing a batch because of its NMS time limit."""
    from ultralytics.utils import LOGGER

    timeouts = []

    class TimeoutHandler(logging.Handler):
        def emit(self, record):
            message = record.getMessage()
            if "NMS time limit" in message and "exceeded" in message:
                timeouts.append(message)

    handler = TimeoutHandler()
    LOGGER.addHandler(handler)
    try:
        yield
    finally:
        LOGGER.removeHandler(handler)
    if timeouts:
        raise RuntimeError(f"Validasi parsial karena NMS timeout: {timeouts[0]}")


def gflops_validation_metadata(validator, arguments, dataset):
    """Verify the actual SDK backend and loaded images before exporting mAP."""
    if validator is None or getattr(validator, "args", None) is None:
        raise ValueError("Validator SDK aktual tidak terekam.")
    fields = ("split", "imgsz", "batch", "workers", "device", "conf", "iou", "max_det",
              "quantize", "nms", "augment", "plots", "fraction", "rect", "data")
    effective = {field: getattr(validator.args, field, None) for field in fields}
    effective["device"] = str(validator.device)
    effective["data"] = str(effective["data"])
    effective["end2end"] = getattr(validator, "end2end", None)
    if (effective["split"] != dataset["split"] or effective["end2end"] is not False
            or effective["nms"] is not None or effective["quantize"] is not None):
        raise ValueError("Validator SDK tidak memakai split, FP32, atau mode one-to-many dengan NMS yang ditetapkan.")
    loaded = getattr(getattr(validator.dataloader, "dataset", None), "im_files", ())
    if sorted(str(Path(path).resolve()) for path in loaded) != sorted(dataset["image_files"]):
        raise ValueError("Daftar gambar validator berbeda dari manifest evaluasi.")
    effective["dataloader_batch"] = validator.dataloader.batch_size
    if effective["batch"] != arguments["batch"] or effective["dataloader_batch"] != min(arguments["batch"], len(loaded)):
        raise ValueError("Batch validator aktual berbeda dari protokol (atau batas ukuran subset).")
    return {"split": dataset["split"], "evaluation_mode": dataset["mode"],
            "dataset_fingerprint": dataset["fingerprint"],
            "dataset_provenance_status": dataset["provenance_status"],
            "validation_image_count": len(loaded), "validation_arguments_requested": arguments,
            "validation_arguments_effective": effective}


def _gflops_complete_student_row(row):
    if (row.get("evaluation_mode") != "full_val" or row.get("dataset_provenance_status") != "VERIFIED"
            or any(row.get(key) != "PASS" for key in ("structural_status", "reload_status", "validation_status"))
            or row.get("latency_status") not in {"PASS", "MEASURED"}):
        return False
    fields = ("target_speedup", "gflops_speedup", "map50_95", "baseline_map50_95", "gflops",
              "parameters", "latency_median_ms")
    if any(not isinstance(row.get(key), (float, int)) or isinstance(row[key], bool)
           or not math.isfinite(row[key]) for key in fields):
        return False
    return (row.get("model_variant") in {"n", "s"} and row["target_speedup"] in {1.02, 1.05, 1.10, 1.20, 1.70}
            and row.get("target_status", "REACHED") == "REACHED"
            and row.get("baseline_validation_parity_status", "PASS") == "PASS"
            and row["gflops_speedup"] >= row["target_speedup"]
            and 0 <= row["map50_95"] <= 1 and 0 <= row["baseline_map50_95"] <= 1
            and row["baseline_map50_95"] - row["map50_95"] <= 0.02 + 1e-12
            and all(row[key] > 0 for key in ("gflops", "parameters", "latency_median_ms")))


def gflops_select_student(rows, mode):
    """Select one comparable full-validation student from both variants."""
    if mode != "full_val":
        return {"status": "NOT_RUN", "reason": "Student dipilih hanya dari full_val.", "selected": None}
    eligible = [row for row in rows if _gflops_complete_student_row(row)]
    if not eligible:
        return {"status": "NO_QUALIFYING_STUDENT", "selected": None}
    fingerprints = {row.get("dataset_fingerprint") for row in eligible}
    if len(fingerprints) != 1 or not next(iter(fingerprints)):
        raise ValueError("Kandidat student berasal dari fingerprint full val berbeda atau tidak tercatat.")
    best = min(eligible, key=lambda row: (-row["target_speedup"], -row["map50_95"], row["latency_median_ms"],
                                          row["parameters"], row["gflops"], row["model_variant"],
                                          row.get("checkpoint_sha256") or ""))
    return {"status": "SELECTED", "selected": dict(best)}


def _gflops_verified_checkpoint(path, expected):
    checkpoint = Path(path).resolve()
    if not checkpoint.is_file() or sha256_file(checkpoint) != expected:
        raise ValueError(f"Checkpoint berbeda dari SHA-256 yang dikunci: {checkpoint}")
    return checkpoint


def gflops_select_candidate_files(paths):
    """Combine independent N/S full-val exports under the same measurement contract."""
    files = [Path(path).resolve() for path in paths]
    if len(files) != len(set(files)):
        raise ValueError("File kandidat yang sama tidak boleh digabung dua kali.")
    variants, fingerprints, contracts, rows = set(), set(), [], []
    hashes = {str(path): sha256_file(path) for path in files}
    for path in files:
        record = json.loads(path.read_text(encoding="utf-8"))
        if (record.get("format_version") not in {1, 2} or record.get("mode") != "full_val"
                or record.get("status") not in {"PASS", "PARTIAL"}
                or record.get("evaluation_status") not in {"PASS", "PARTIAL"}
                or record.get("dataset_provenance_status") != "VERIFIED"):
            raise ValueError("Gabungan student memerlukan ekspor full_val VERIFIED yang selesai PASS.")
        members = set(record.get("model_variants", []))
        if not members or not members <= {"n", "s"} or variants.intersection(members):
            raise ValueError("Setiap varian harus mempunyai tepat satu sumber full_val.")
        variants.update(members)
        fingerprints.add(record.get("dataset_fingerprint"))
        contract = record.get("comparison_contract")
        if not isinstance(contract, dict) or not contract:
            raise ValueError("Kontrak evaluasi dan perangkat latency tidak tercatat.")
        contracts.append(contract)
        for source in record.get("rows", []):
            if source.get("model_variant") not in members:
                raise ValueError("Varian baris kandidat berbeda dari deklarasi ekspor.")
            row = dict(source)
            for key in ("checkpoint", "baseline_checkpoint"):
                relative = row.get(key + "_relative_path")
                if relative:
                    relocated = (path.parent / relative).resolve()
                    if not relocated.is_relative_to(path.parent):
                        raise ValueError("Path checkpoint kandidat keluar dari folder ekspor.")
                    row[key + "_path"] = str(relocated)
            row["selection_source_sha256"] = hashes
            rows.append(row)
    if len(fingerprints) > 1 or (fingerprints and not next(iter(fingerprints))):
        raise ValueError("Fingerprint full validation N/S berbeda atau kosong.")
    if contracts and any(contract != contracts[0] for contract in contracts[1:]):
        raise ValueError("Protokol, implementasi, atau perangkat latency N/S berbeda.")
    if variants != {"n", "s"}:
        return {"status": "NOT_RUN", "selected": None,
                "reason": "Pemilihan akhir menunggu hasil full_val dari N dan S.",
                "available_variants": sorted(variants)}
    result = gflops_select_student(rows, "full_val")
    if result.get("selected"):
        selected = result["selected"]
        _gflops_verified_checkpoint(selected["checkpoint_path"], selected["checkpoint_sha256"])
        _gflops_verified_checkpoint(selected["baseline_checkpoint_path"], selected["baseline_checkpoint_sha256"])
    return {**result, "source_files": hashes, "model_variants": sorted(variants)}


def gflops_lock_student(selection, path):
    """Write an immutable-by-convention full-val lock, verified again before test."""
    if selection.get("status") != "SELECTED":
        return None
    student = selection["selected"]
    if not _gflops_complete_student_row(student) or not student.get("dataset_fingerprint"):
        raise ValueError("Lock student memerlukan hasil full val yang lolos seleksi.")
    _gflops_verified_checkpoint(student["checkpoint_path"], student["checkpoint_sha256"])
    _gflops_verified_checkpoint(student["baseline_checkpoint_path"], student["baseline_checkpoint_sha256"])
    lock = Path(path)
    record = {"format_version": 1, "status": "LOCKED", "selected": student}
    if lock.exists():
        if json.loads(lock.read_text(encoding="utf-8")) != record:
            raise FileExistsError(f"Lock student sudah berisi pilihan berbeda: {lock}")
    else:
        # Exclusive creation prevents another run replacing the frozen selection.
        with lock.open("x", encoding="utf-8") as stream:
            json.dump(record, stream, ensure_ascii=False, indent=2, allow_nan=False)
    return record


def gflops_run_locked_test(lock_path, dataset, output_dir, *, checkpoint_path=None, baseline_checkpoint_path=None):
    """Evaluate the fixed baseline/student pair only on an independent test split."""
    if dataset is None:
        return {"status": "NOT_RUN", "reason": "Split test/train independen belum tersedia."}
    if (dataset.get("mode") != "test" or dataset.get("split") != "test"
            or dataset.get("provenance_status") != "VERIFIED"
            or dataset.get("test_content_disjoint_from_train_val") is not True):
        raise ValueError("Test memerlukan split independen yang provenance-nya VERIFIED.")
    record = json.loads(Path(lock_path).read_text(encoding="utf-8"))
    student = record.get("selected", {})
    if (record.get("status") != "LOCKED" or record.get("format_version") != 1
            or not _gflops_complete_student_row(student)
            or student["dataset_fingerprint"] != dataset.get("full_val_fingerprint")):
        raise ValueError("Lock student tidak cocok dengan full val Poltrada yang telah diverifikasi.")
    baseline = _gflops_verified_checkpoint(baseline_checkpoint_path or student["baseline_checkpoint_path"],
                                           student["baseline_checkpoint_sha256"])
    pruned = _gflops_verified_checkpoint(checkpoint_path or student["checkpoint_path"], student["checkpoint_sha256"])
    before = evaluate_checkpoint(baseline, dataset, output_dir, "baseline_test")
    after = evaluate_checkpoint(pruned, dataset, output_dir, "student_test")
    return {"status": "PASS", "baseline": before, "student": after,
            "delta_map50_95_pp": 100 * (after["map50_95"] - before["map50_95"]),
            "test_fingerprint": dataset["fingerprint"], "checkpoint_sha256": student["checkpoint_sha256"]}

In [ ]:
"""Standalone analysis helpers embedded in notebook 03 by its generator.

All figures describe measured structural changes, not activation feature maps.
Missing historical mappings, scores, and validation values are never fabricated.
"""
import csv as _eda_csv
import json as _eda_json
import math as _eda_math
from pathlib import Path as _EdaPath

import matplotlib.pyplot as _eda_plt
import numpy as _eda_np
from matplotlib.colors import ListedColormap as _EdaColors


def _eda_write_json(path, value):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_eda_json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def _eda_write_csv(path, rows):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = _eda_csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        writer.writerows({key: _eda_json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
                         for key, value in row.items()} for row in rows)


def _eda_finite(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and _eda_math.isfinite(value)


def _eda_save_figure(fig, path):
    path = _EdaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path.with_suffix(".png"), dpi=int(globals().get("EDA_DPI", 300)), bbox_inches="tight")
    fig.savefig(path.with_suffix(".svg"), bbox_inches="tight")
    _eda_plt.close(fig)
    return {"png": path.with_suffix(".png").name, "svg": path.with_suffix(".svg").name}


def layer_eda_rows(inventory_before, inventory_after):
    """Join actual unfused inventory by module name, preserving baseline order."""
    previous = {row["layer_name"]: row for row in inventory_before}
    current = {row["layer_name"]: row for row in inventory_after}
    if len(previous) != len(inventory_before) or len(current) != len(inventory_after):
        raise ValueError("EDA inventory has duplicate module names.")
    if set(previous) != set(current):
        raise ValueError("EDA before/after module names differ; layer alignment is unsafe.")
    rows = []
    for name, before in previous.items():
        after = current[name]
        if before["module_type"] != after["module_type"]:
            raise ValueError(f"EDA module type changed at {name}.")
        if after["out_channels"] > before["out_channels"] or after["in_channels"] > before["in_channels"]:
            raise ValueError(f"Pruning unexpectedly increased channels at {name}.")
        parts = name.split(".")
        stage = before.get("stage", ".".join(parts[:2]))
        row = {"layer_name": name, "module_type": before["module_type"], "stage": stage,
               "region": before.get("region", "unknown"), "policy": before.get("policy"),
               "eligible": before.get("eligible", False), "protection_reason": before.get("protection_reason", ""),
               "dependency_group": before.get("dependency_group", "")}
        for key in ("in_channels", "out_channels", "parameters", "estimated_flops"):
            row[key + "_before"] = before.get(key)
            row[key + "_after"] = after.get(key)
            base, value = before.get(key), after.get(key)
            row[key + "_removed"] = base - value if _eda_finite(base) and _eda_finite(value) else None
            row[key + "_reduction_pct"] = 100 * (base - value) / base if _eda_finite(base) and _eda_finite(value) and base else None
        if (row["protection_reason"] and str(row["protection_reason"]).startswith(("detection", "attention"))
                and (row["out_channels_removed"] or row["in_channels_removed"])):
            raise ValueError(f"Protected head/attention channel dimension changed at {name}.")
        rows.append(row)
    return rows


def stage_eda_rows(rows):
    """Physical Conv-output sums; these are distinct from independent group channels."""
    stages = {}
    for row in rows:
        key = (str(row["stage"]), row["region"])
        stage = stages.setdefault(key, {"stage": row["stage"], "region": row["region"], "conv_layers": 0,
                                      "physical_conv_outputs_before": 0, "physical_conv_outputs_after": 0,
                                      "parameters_before": 0, "parameters_after": 0,
                                      "estimated_flops_before": 0, "estimated_flops_after": 0})
        if row["module_type"] == "Conv2d":
            stage["conv_layers"] += 1
            for suffix in ("before", "after"):
                stage["physical_conv_outputs_" + suffix] += row["out_channels_" + suffix]
        for metric in ("parameters", "estimated_flops"):
            for suffix in ("before", "after"):
                value = row[metric + "_" + suffix]
                if _eda_finite(value):
                    stage[metric + "_" + suffix] += value
    for row in stages.values():
        row["physical_conv_outputs_removed"] = row["physical_conv_outputs_before"] - row["physical_conv_outputs_after"]
        for key in ("physical_conv_outputs", "parameters", "estimated_flops"):
            base, after = row[key + "_before"], row[key + "_after"]
            row[key + "_reduction_pct"] = 100 * (base - after) / base if base else None
    return list(stages.values())


def make_channel_figure(rows, title="Channel comparison"):
    """Three panels share exact Conv order and Y limits, as in the reference image."""
    if not rows or any(row["module_type"] != "Conv2d" for row in rows):
        raise ValueError("Channel diagram requires a nonempty ordered Conv2d inventory.")
    count = len(rows)
    width = max(16, min(32, count * 0.16))
    fig, axes = _eda_plt.subplots(3, 1, figsize=(width, 12), sharex=True, sharey=True)
    indices = _eda_np.arange(count)
    before = [row["out_channels_before"] for row in rows]
    after = [row["out_channels_after"] for row in rows]
    upper = max(max(before), max(after), 1) * 1.12
    axes[0].bar(indices, before, color="blue", width=0.82, label="Before Pruning")
    axes[1].bar(indices, after, color="red", width=0.82, label="After Pruning")
    axes[2].bar(indices, before, color="blue", width=0.82, label="Before Pruning")
    axes[2].bar(indices, after, color="red", width=0.82, label="After Pruning")
    for ax, heading in zip(axes, ("Before Pruning", "After Pruning", "Channel Comparison")):
        ax.set_title(heading)
        ax.set_ylabel("Output channels")
        ax.set_ylim(0, upper)
        ax.set_xlim(-0.8, count - 0.2)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        ax.legend(loc="upper right")
        ax.tick_params(axis="x", labelbottom=True)
        ax.set_xticks(indices, [row["layer_name"] for row in rows], rotation=90, fontsize=6 if count > 32 else 8)
        ax.set_xlabel("Conv module (baseline order)")
    fig.suptitle(title, y=1.005, fontsize=14)
    fig.tight_layout()
    return fig


def _eda_channel_plots(conv_rows, folder, title):
    _eda_write_csv(folder / "channel_plot_data.csv", conv_rows)
    files = [_eda_save_figure(make_channel_figure(conv_rows, title), folder / "channels_overview")]
    page_size = min(32, max(1, int(globals().get("EDA_MAX_LAYERS_PER_PAGE", 32))))
    for offset in range(0, len(conv_rows), page_size):
        page = offset // page_size + 1
        files.append(_eda_save_figure(make_channel_figure(conv_rows[offset:offset + page_size],
                     f"{title} — layers {offset + 1}–{min(offset + page_size, len(conv_rows))}"),
                     folder / f"channels_detail_{page:02d}"))
    fig, ax = _eda_plt.subplots(figsize=(max(14, min(32, len(conv_rows) * 0.16)), 5))
    ax.bar(range(len(conv_rows)), [row["out_channels_reduction_pct"] for row in conv_rows], color="red")
    ax.set_xticks(range(len(conv_rows)), [row["layer_name"] for row in conv_rows], rotation=90, fontsize=6)
    ax.set_ylabel("Output channels removed (%)")
    ax.set_title(title + " — physical Conv-output reduction")
    ax.set_ylim(0, 100)
    fig.tight_layout()
    files.append(_eda_save_figure(fig, folder / "channel_reduction_percent"))
    return {"status": "MEASURED", "conv_layers": len(conv_rows), "detail_page_size": page_size, "files": files}


def _eda_mapping_plots(audit, conv_rows, folder):
    mapping = audit.get("channel_mapping")
    if not mapping:
        return {"status": "UNAVAILABLE", "reason": "Original channel indices were not recorded; no reconstruction attempted."}
    _eda_write_json(folder / "channel_mapping.json", mapping)
    _eda_write_csv(folder / "channel_mapping.csv", mapping)
    masks = audit.get("channel_masks")
    if masks:
        _eda_write_json(folder / "channel_masks.json", masks)
    by_layer = {row["layer"]: row for row in mapping if row.get("module_type") == "Conv2d"
                and row.get("dimension") in {"out", "out_channels", "output", "output_channels"}}
    selected = []
    for row in conv_rows:
        item = by_layer.get(row["layer_name"])
        if item is None:
            return {"status": "UNAVAILABLE", "reason": "Mapping does not cover every Conv output.", "layer": row["layer_name"]}
        before = int(item["channels_before"])
        kept, removed = item.get("kept_original_indices"), item.get("removed_original_indices")
        if not isinstance(kept, list) or not isinstance(removed, list):
            raise TypeError(f"Missing original-index ledger for {row['layer_name']}.")
        if (len(set(kept)) != len(kept) or len(set(removed)) != len(removed)
                or set(kept) & set(removed) or set(kept) | set(removed) != set(range(before))
                or len(kept) != item["channels_after"]
                or before != row["out_channels_before"] or len(kept) != row["out_channels_after"]):
            raise ValueError(f"Original-index ledger disagrees with measured channels at {row['layer_name']}.")
        selected.append((row["layer_name"], before, set(removed)))
    page_size = min(32, max(1, int(globals().get("EDA_MAX_LAYERS_PER_PAGE", 32))))
    files = []
    for offset in range(0, len(selected), page_size):
        page = selected[offset:offset + page_size]
        width = max(item[1] for item in page)
        data = _eda_np.full((len(page), width), _eda_np.nan)
        for index, (_name, count, removed) in enumerate(page):
            data[index, :count] = 0
            if removed:
                data[index, sorted(removed)] = 1
        fig, ax = _eda_plt.subplots(figsize=(16, max(4, len(page) * 0.25)))
        cmap = _EdaColors(["#4477cc", "#ee3333"]).with_extremes(bad="#eeeeee")
        plot = ax.imshow(data, interpolation="nearest", aspect="auto", cmap=cmap, vmin=0, vmax=1)
        ax.set_yticks(range(len(page)), [item[0] for item in page], fontsize=7)
        ax.set_xlabel("Original Conv-output channel index (before any pruning)")
        ax.set_title("Original-index mask: blue kept, red removed, grey outside layer width")
        fig.colorbar(plot, ax=ax, ticks=[0, 1]).ax.set_yticklabels(["Kept", "Removed"])
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"channel_mask_{offset // page_size + 1:02d}"))
    return {"status": "MEASURED", "conv_layers": len(selected), "files": files,
            "meaning": "Physical Conv-output masks, including coupled dimensions; never summed as independent channels."}


def _eda_lamp_plots(audit, folder):
    scores = audit.get("lamp_scores")
    if not scores:
        return {"status": "UNAVAILABLE", "reason": "Finite pre-cap LAMP scores were not recorded."}
    if any(not _eda_finite(row.get("score")) for row in scores):
        raise ValueError("LAMP EDA requires finite original scores; capacity sentinels must not be plotted.")
    _eda_write_csv(folder / "lamp_scores.csv", scores)
    _eda_write_json(folder / "lamp_scores.json", scores)
    fig, ax = _eda_plt.subplots(figsize=(10, 5))
    for removed, color, label in ((False, "blue", "Kept"), (True, "red", "Removed")):
        values = [row["score"] for row in scores if bool(row.get("removed")) == removed]
        if values:
            ax.hist(values, bins=min(50, max(5, int(len(values) ** 0.5))), alpha=0.55, color=color, label=label)
    ax.set_xlabel("Finite LAMP score before capacity mask")
    ax.set_ylabel("Independent root channels")
    ax.set_title("LAMP selection — descriptive score distribution")
    ax.legend()
    fig.tight_layout()
    files = [_eda_save_figure(fig, folder / "lamp_score_distribution")]
    capacities = {}
    for row in scores:
        group = capacities.setdefault(row["group_id"], {"channels": 0, "capacity_eligible": 0, "removed": 0})
        group["channels"] += 1
        group["capacity_eligible"] += bool(row.get("capacity_eligible"))
        group["removed"] += bool(row.get("removed"))
    capacity_rows = [{"group_id": key, **value} for key, value in capacities.items()]
    _eda_write_csv(folder / "lamp_group_capacity.csv", capacity_rows)
    if any(row["removed"] > row["capacity_eligible"] for row in capacity_rows):
        raise ValueError("LAMP selection removed channels outside the audited capacity mask.")
    fig, ax = _eda_plt.subplots(figsize=(max(12, min(24, len(capacity_rows) * .45)), 6))
    positions = _eda_np.arange(len(capacity_rows))
    blocked = [row["channels"] - row["capacity_eligible"] for row in capacity_rows]
    retained = [row["capacity_eligible"] - row["removed"] for row in capacity_rows]
    removed = [row["removed"] for row in capacity_rows]
    ax.bar(positions, blocked, color="grey", label="Protected by capacity cap")
    ax.bar(positions, retained, bottom=blocked, color="blue", label="Capacity eligible, kept")
    ax.bar(positions, removed, bottom=_eda_np.array(blocked) + retained, color="red", label="Removed")
    ax.set_xticks(positions, [row["group_id"] for row in capacity_rows], rotation=90, fontsize=7)
    ax.set_ylabel("Independent root channels")
    ax.set_title("Group capacity and actual LAMP selection")
    ax.legend()
    fig.tight_layout()
    files.append(_eda_save_figure(fig, folder / "lamp_group_capacity"))
    return {"status": "MEASURED", "finite_scores": len(scores),
            "capacity_eligible": sum(bool(row.get("capacity_eligible")) for row in scores),
            "removed": sum(bool(row.get("removed")) for row in scores), "files": files}


def _eda_efficiency_plots(stages, result, folder):
    files = []
    for metric, label in (("physical_conv_outputs", "Physical Conv-output channels"),
                          ("parameters", "Parameters (Conv + BN)"),
                          ("estimated_flops", "Estimated layer FLOPs (Conv + BN)")):
        fig, ax = _eda_plt.subplots(figsize=(14, 5))
        positions = _eda_np.arange(len(stages))
        ax.bar(positions - .2, [row[metric + "_before"] for row in stages], width=.4, color="blue", label="Before")
        ax.bar(positions + .2, [row[metric + "_after"] for row in stages], width=.4, color="red", label="After")
        ax.set_xticks(positions, [str(row["stage"]) for row in stages], rotation=90)
        ax.set_ylabel(label)
        ax.set_title("Measured stage inventory — " + label)
        ax.legend()
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / ("stage_" + metric)))
    baseline, pruned = result.get("baseline", {}), result.get("pruned", {})
    summary = {"independent_eligible_channels_before": result.get("eligible_channels_before"),
               "independent_eligible_channels_after": result.get("eligible_channels_after"),
               "actual_independent_channel_ratio": result.get("actual_eligible_channel_ratio"),
               "physical_conv_outputs_before": sum(row["physical_conv_outputs_before"] for row in stages),
               "physical_conv_outputs_after": sum(row["physical_conv_outputs_after"] for row in stages),
               "baseline": {key: baseline.get(key) for key in ("parameters", "gflops", "checkpoint_bytes", "latency")},
               "pruned": {key: pruned.get(key) for key in ("parameters", "gflops", "checkpoint_bytes", "latency")},
               "comparison": result.get("comparison", {}),
               "latency_scope": "raw model forward, not end-to-end",
               "layer_flops_scope": "Conv/BN estimate; model GFLOPs also counts functional attention"}
    _eda_write_json(folder / "efficiency_summary.json", summary)
    return {"status": "MEASURED", "files": files, **summary}


def _eda_evaluation_plots(result, folder):
    statuses, files = {}, []
    for identity in ("baseline", "pruned"):
        validation = (result.get(identity) or {}).get("validation")
        if not validation:
            statuses[identity] = {"status": "NOT_RUN", "reason": "Dataset validation was not run."}
            continue
        target = folder / identity
        target.mkdir(parents=True, exist_ok=True)
        per_class = validation.get("per_class") or []
        if per_class:
            _eda_write_csv(target / "per_class_metrics.csv", per_class)
            labels = [row.get("name", str(row.get("class_id"))) for row in per_class]
            fig, axes = _eda_plt.subplots(1, 3, figsize=(21, 5))
            for key, label in (("ap50", "AP50"), ("ap50_95", "AP50–95")):
                indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get(key))]
                if indices:
                    axes[0].plot(indices, [per_class[index][key] for index in indices], "o-", label=label)
            for key, label in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
                indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get(key))]
                if indices:
                    axes[1].plot(indices, [per_class[index][key] for index in indices], "o-", label=label)
            support_indices = [index for index, row in enumerate(per_class) if _eda_finite(row.get("support"))]
            if support_indices:
                axes[2].bar(support_indices, [per_class[index]["support"] for index in support_indices], color="grey")
            else:
                axes[2].text(.5, .5, "Class support unavailable", ha="center", va="center", transform=axes[2].transAxes)
            for ax in axes:
                ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
                if ax.lines:
                    ax.legend()
            for ax in axes[:2]:
                ax.set_ylim(0, 1)
                ax.set_ylabel("Measured metric")
            axes[2].set_ylabel("Ground-truth instance support")
            fig.suptitle(identity + " — measured class metrics; missing values are omitted")
            fig.tight_layout()
            files.append(_eda_save_figure(fig, target / "class_metrics"))
        curves = validation.get("curves")
        if curves:
            _eda_write_json(target / "curves.json", curves)
        curve_status = _eda_curve_plots(curves, target) if curves else {"status": "UNAVAILABLE"}
        confusion = validation.get("confusion_matrix") or {}
        confusion_status = {"status": "UNAVAILABLE", "reason": "No measured confusion matrix was collected."}
        if (confusion.get("measured") is True or confusion.get("status") == "MEASURED") and confusion.get("matrix") is not None:
            matrix = _eda_np.asarray(confusion["matrix"], dtype=float)
            labels = confusion.get("labels", [])
            if matrix.ndim != 2 or matrix.shape != (len(labels), len(labels)) or not _eda_np.isfinite(matrix).all():
                raise ValueError("Confusion matrix dimensions/labels/values are invalid.")
            _eda_write_json(target / "confusion_matrix.json", confusion)
            _eda_write_csv(target / "confusion_matrix.csv", [
                {"predicted_class": labels[index], **{label: float(value) for label, value in zip(labels, row)}}
                for index, row in enumerate(matrix)])
            fig, ax = _eda_plt.subplots(figsize=(9, 8))
            plotted = ax.imshow(matrix, cmap="Blues")
            ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
            ax.set_yticks(range(len(labels)), labels)
            ax.set_xlabel(confusion.get("x_label", "True class"))
            ax.set_ylabel(confusion.get("y_label", "Predicted class"))
            conf = confusion.get("confidence_threshold", confusion.get("conf"))
            matching_iou = confusion.get("matching_iou_threshold", confusion.get("iou"))
            ax.set_title(f"{identity}: measured confusion matrix\nconf={conf}, matching IoU={matching_iou}")
            fig.colorbar(plotted, ax=ax)
            fig.tight_layout()
            files.append(_eda_save_figure(fig, target / "confusion_matrix"))
            confusion_status = {"status": "MEASURED", "confidence_threshold": conf, "matching_iou_threshold": matching_iou,
                                "validation_iou": confusion.get("validation_iou")}
        statuses[identity] = {"status": "MEASURED", "per_class": "MEASURED" if per_class else "UNAVAILABLE",
                              "curves": curve_status, "confusion_matrix": confusion_status}
    return {"status": "MEASURED" if any(row["status"] == "MEASURED" for row in statuses.values()) else "NOT_RUN",
            "measurements": statuses, "files": files,
            "interpretation": "Descriptive association; layer changes do not establish a cause of mAP changes."}


def _eda_curve_plots(curves, folder):
    entries = curves if isinstance(curves, list) else curves.get("curves", curves.get("items", []))
    count, files = 0, []
    for entry in entries:
        if not isinstance(entry, dict) or entry.get("status") in {"UNAVAILABLE", "NOT_RUN"}:
            continue
        x, y = entry.get("x"), entry.get("y")
        if x is None or y is None:
            continue
        xs, ys = _eda_np.asarray(x, dtype=float), _eda_np.asarray(y, dtype=float)
        if xs.ndim != 1 or not xs.size or not _eda_np.isfinite(xs).all() or not _eda_np.isfinite(ys).all():
            continue
        if ys.ndim == 1:
            ys = ys[None, :]
        if ys.ndim != 2 or ys.shape[-1] != len(xs):
            continue
        fig, ax = _eda_plt.subplots(figsize=(10, 6))
        labels = entry.get("labels") or entry.get("class_names") or []
        for index, values in enumerate(ys):
            ax.plot(xs, values, label=str(labels[index]) if index < len(labels) else f"series {index}", alpha=.75)
        ax.set_xlabel(entry.get("x_label", entry.get("xlabel", "x")))
        ax.set_ylabel(entry.get("y_label", entry.get("ylabel", "y")))
        ax.set_title(entry.get("name", "Measured validation curve"))
        if len(ys) <= 20:
            ax.legend()
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"evaluation_curve_{count:02d}"))
        count += 1
    return {"status": "MEASURED" if count else "UNAVAILABLE", "curve_count": count, "files": files}


def _build_run_eda(result, run_dir, inventory_before, inventory_after):
    """Export ordered EDA from actual run records; failures propagate to run reports."""
    folder = _EdaPath(run_dir) / "eda"
    folder.mkdir(parents=True, exist_ok=True)
    rows = layer_eda_rows(inventory_before, inventory_after)
    conv_rows = [row for row in rows if row["module_type"] == "Conv2d"]
    stages = stage_eda_rows(rows)
    _eda_write_csv(folder / "layer_eda.csv", rows)
    _eda_write_csv(folder / "stage_eda.csv", stages)
    audit = result.get("audit") or {}
    title = f"{result.get('model_name', 'YOLO26' + result.get('model_variant', 'n'))} {result.get('policy', '')} pr{result.get('ratio', 0):g}"
    report = {"status": "MEASURED", "model_variant": result.get("model_variant", "n"),
              "model_name": result.get("model_name", "YOLO26n"), "policy": result.get("policy"), "ratio": result.get("ratio"),
              "section_order": ["identity_coverage", "channels", "masks_lamp", "efficiency", "validation_tradeoffs"],
              "identity_coverage": {"provenance": result.get("provenance"), "conv_layers": len(conv_rows),
                                    "bn_layers": sum(row["module_type"] == "BatchNorm2d" for row in rows),
                                    "eligible_layers": sum(bool(row["eligible"]) for row in rows),
                                    "protected_layers": sum(bool(row["protection_reason"]) for row in rows),
                                    "group_count": result.get("pruned_group_count"),
                                    "protection_reasons": sorted({str(row["protection_reason"]) for row in rows if row["protection_reason"]})},
              "channels": _eda_channel_plots(conv_rows, folder, title),
              "channel_mapping": _eda_mapping_plots(audit, conv_rows, folder),
              "lamp_scores": _eda_lamp_plots(audit, folder),
              "efficiency": _eda_efficiency_plots(stages, result, folder),
              "validation_tradeoffs": _eda_evaluation_plots(result, folder),
              "notes": ["Channel figures include all Conv outputs in original module order.",
                        "Independent root-channel counts differ from physical coupled Conv-output changes.",
                        "No activation feature maps, simulated accuracy, or reconstructed historical channel indices."]}
    _eda_write_json(folder / "eda_status.json", report)
    return report


def build_run_eda(result, run_dir, inventory_before, inventory_after):
    """Persist an explicit failed EDA report before propagating analysis errors."""
    try:
        return _build_run_eda(result, run_dir, inventory_before, inventory_after)
    except Exception as error:
        _eda_write_json(_EdaPath(run_dir) / "eda/eda_status.json",
                        {"status": "FAIL", "model_variant": result.get("model_variant", "n"),
                         "policy": result.get("policy"), "ratio": result.get("ratio"),
                         "error": {"type": type(error).__name__, "message": str(error)}})
        raise


def plot_model_comparison(suites, output_dir):
    """Compare N/S only where each point has measured accuracy and efficiency."""
    folder = _EdaPath(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    suites = list(suites.values()) if isinstance(suites, dict) else list(suites)
    points = []
    for suite in suites:
        variant = suite.get("model_variant", "n")
        name = suite.get("model_name", "YOLO26" + variant)
        baseline = suite.get("baseline") or {}
        for ratio, policy, metrics in [(0., "baseline", baseline)] + [
                (run["ratio"], run.get("policy", "extended"), run.get("pruned") or {})
                for run in suite.get("results", [])]:
            validation = metrics.get("validation") or {}
            latency = metrics.get("latency") or {}
            points.append({"model_variant": variant, "model_name": name, "ratio": ratio, "policy": policy,
                           "map50_95": validation.get("map50_95"), "parameters": metrics.get("parameters"),
                           "gflops": metrics.get("gflops"),
                           "latency_ms": latency.get("median_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                           "latency_p05_ms": latency.get("p05_ms") if latency.get("status") in {"PASS", "MEASURED"} else None,
                           "latency_p95_ms": latency.get("p95_ms") if latency.get("status") in {"PASS", "MEASURED"} else None})
    _eda_write_csv(folder / "model_comparison.csv", points)
    statuses = []
    for metric, label in (("parameters", "Parameters"), ("gflops", "GFLOPs"), ("latency_ms", "Median raw model forward latency (ms)")):
        valid = [row for row in points if _eda_finite(row.get(metric)) and _eda_finite(row.get("map50_95"))]
        if not valid:
            statuses.append({"plot": "map_vs_" + metric, "status": "NOT_RUN", "reason": "Paired measured accuracy/efficiency is unavailable."})
            continue
        fig, ax = _eda_plt.subplots(figsize=(10, 7))
        for variant in dict.fromkeys(row["model_variant"] for row in valid):
            rows = [row for row in valid if row["model_variant"] == variant]
            ax.scatter([row[metric] for row in rows], [row["map50_95"] for row in rows], label="YOLO26" + variant)
            for row in rows:
                if (metric == "latency_ms" and _eda_finite(row.get("latency_p05_ms"))
                        and _eda_finite(row.get("latency_p95_ms"))):
                    ax.hlines(row["map50_95"], row["latency_p05_ms"], row["latency_p95_ms"], alpha=.5)
                ax.annotate(f"{row['model_name']} {row['policy']} pr{row['ratio']:g}", (row[metric], row["map50_95"]),
                            xytext=(5, 5), textcoords="offset points", fontsize=8)
        ax.set_xlabel(label)
        ax.set_ylabel("Measured mAP50–95")
        ax.set_title("N/S compression–accuracy comparison")
        ax.legend()
        fig.tight_layout()
        files = _eda_save_figure(fig, folder / ("map_vs_" + metric))
        statuses.append({"plot": "map_vs_" + metric, "status": "MEASURED", "points": len(valid), "files": files})
    report = {"plots": statuses, "point_count": len(points), "scope": "Each model uses its own measured baseline; no missing accuracy is replaced with zero."}
    _eda_write_json(folder / "comparison_status.json", report)
    return report

In [ ]:
def gflops_latency(core, image):
    if not RUN_LATENCY_BENCHMARK:
        return {"status": "NOT_RUN", "reason": "Benchmark dimatikan."}
    try:
        return cuda_latency(core, image, LATENCY_WARMUP, LATENCY_REPEATS)
    except Exception as error:
        return {"status": "FAIL", "error": safe_error(error)}

def gflops_validation(row, checkpoint, dataset, folder):
    if dataset is None:
        return
    try:
        row["validation"] = evaluate_checkpoint(checkpoint, dataset, folder, "validation")
        row["validation_status"] = "PASS"
    except Exception as error:
        row["validation_status"] = "FAIL"
        row["errors"].append({"stage": "validation", "error": safe_error(error)})

## 05 — Training, checkpoint per epoch, dan resume
Trainer mempertahankan objek DetectionModel dengan channel pruned. Checkpoint epoch menyimpan model aktif, EMA FP32, state training dan gradien pending.
W&B mengonfirmasi raw sebelum training. Paket resume terbaru memuat state terakhir dan model dari epoch terbaik. Untuk resume, gunakan `RESUME_SUITE_DIR` lama dengan konfigurasi yang sama.

**Penyimpanan best/latest:** model inference hanya `best.pt` dan `last.pt`. Snapshot lokal mempertahankan model terbaik dan state resume terbaru. W&B menyimpan satu versi paket resume terbaru yang mencakup keduanya; versi dari run ini yang lebih lama dihapus hanya setelah penggantinya diunduh dan SHA-256 cocok. Run lain dan input raw/baseline/teacher tetap dilindungi. Loss, seluruh epoch history, Tables, Images, dan EDA eksperimen penuh tetap dicatat.

**Smoke:** setelah training, evaluasi, EDA dan verifikasi publikasi selesai, artefak diagnostik W&B serta salinan checkpoint lokal di folder smoke ini otomatis dihapus. Receipt SHA-256, metrik, tabel, grafik dan log lokal tetap disimpan. Registry menandai `DELETED_AFTER_SMOKE`; referensinya menjadi bukti historis dan tidak dapat dipakai recovery. Interupsi sebelum selesai tetap dapat di-resume. Untuk uji restore snapshot, nonaktifkan dua opsi cleanup selama pengujian lalu bersihkan setelah verifikasi.


In [ ]:
"""Standalone ordinary fine-tuning helpers for native structurally pruned YOLO models.

The trainer retains the DetectionModel object instead of rebuilding its original YAML.
The detection criterion, gradients, optimizer, EMA, augmentations and validation remain
Ultralytics' native implementation. Only model loading, logging isolation and auditing
are customized. Best FP32 EMA is captured before SDK FP16 checkpoint serialization.
"""
import hashlib as _recovery_hashlib
import json as _recovery_json
from copy import deepcopy as _recovery_deepcopy
from datetime import UTC as _RECOVERY_UTC
from datetime import datetime as _RecoveryDateTime
from pathlib import Path as _RecoveryPath

import torch as _recovery_torch
from ultralytics import __version__ as _recovery_ultralytics_version
from ultralytics.models.yolo.detect import DetectionTrainer as _RecoveryDetectionTrainer
from ultralytics.nn.tasks import DetectionModel as _RecoveryDetectionModel


def recovery_ordered_names(names):
    """Preserve class order and reject sparse or non-integer class identifiers."""
    if isinstance(names, dict):
        if set(names) != set(range(len(names))):
            raise ValueError("Recovery class IDs must be contiguous integers starting at zero.")
        names = [names[index] for index in range(len(names))]
    if not isinstance(names, (list, tuple)) or not names or not all(isinstance(n, str) for n in names):
        raise ValueError("Recovery requires an explicit nonempty ordered class-name list.")
    return list(names)


def recovery_structure_signature(model):
    """Parameter shapes and physical Conv/BN dimensions; independent of learned values."""
    return {
        "parameters": {name: list(value.shape) for name, value in model.named_parameters()},
        "layers": {
            name: {"type": type(module).__name__, "in": module.in_channels,
                   "out": module.out_channels, "groups": module.groups}
            if isinstance(module, _recovery_torch.nn.Conv2d)
            else {"type": type(module).__name__, "features": module.num_features}
            for name, module in model.named_modules()
            if isinstance(module, (_recovery_torch.nn.Conv2d, _recovery_torch.nn.BatchNorm2d))
        },
    }


def recovery_state_sha256(model, *, parameters_only=False):
    digest = _recovery_hashlib.sha256()
    values = dict(model.named_parameters()) if parameters_only else model.state_dict()
    for name, value in sorted(values.items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(tuple(tensor.shape)).encode())
        digest.update(str(tensor.dtype).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def _recovery_write_json(path, value):
    path = _RecoveryPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_recovery_json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def _recovery_native_snapshot(model):
    core = _recovery_deepcopy(model).to("cpu").float().eval()
    if type(core) is not _RecoveryDetectionModel:
        raise TypeError("Only an ordinary native DetectionModel can be serialized for recovery.")
    core.criterion = None
    return core


def _recovery_finite_state(model):
    return all(_recovery_torch.isfinite(v).all().item() for v in model.state_dict().values()
               if isinstance(v, _recovery_torch.Tensor) and v.is_floating_point())


class StructuralRecoveryTrainer(_RecoveryDetectionTrainer):
    """Native detection training with exact pruned-object loading and FP32 EMA snapshots."""

    def __init__(self, core, *, overrides, expected_names, source_sha256, epoch_callback=None):
        self.recovery_source_core = _recovery_native_snapshot(core)
        self.recovery_expected_names = recovery_ordered_names(expected_names)
        self.recovery_initial_signature = recovery_structure_signature(core)
        self.recovery_initial_state_sha256 = recovery_state_sha256(core)
        self.recovery_initial_parameters_sha256 = recovery_state_sha256(core, parameters_only=True)
        self.recovery_source_sha256 = source_sha256
        self.recovery_optimizer_steps = 0
        self.recovery_epochs = []
        self.recovery_best_core = None
        self.recovery_best_epoch = None
        self.recovery_initial_state_verified = False
        self.recovery_epoch_callback = epoch_callback
        self.recovery_filtered_callbacks = set()
        super().__init__(overrides=overrides)

    def run_callbacks(self, event):
        # BaseTrainer.__init__ dispatches integration events before returning. Filter
        # at dispatch time so W&B/Comet/Platform cannot initialize another run even
        # during construction. No global SETTINGS or callback registry is modified.
        for callback in self.callbacks.get(event, []):
            module = getattr(callback, "__module__", "")
            if module.startswith("ultralytics.utils.callbacks.") and module != "ultralytics.utils.callbacks.base":
                self.recovery_filtered_callbacks.add(module)
                continue
            callback(self)

    def setup_model(self):
        if self.resume or self.args.resume:
            raise ValueError("Recovery starts a new optimizer with resume=False.")
        if recovery_ordered_names(self.data["names"]) != self.recovery_expected_names:
            raise ValueError("Dataset class order differs from the pruned checkpoint.")
        self.model = self.recovery_source_core
        if recovery_ordered_names(self.model.names) != self.recovery_expected_names:
            raise ValueError("Checkpoint class order differs from recovery configuration.")
        self.model.criterion = None  # Native SDK builds its native criterion for this training run.
        self.model.requires_grad_(True)
        if recovery_structure_signature(self.model) != self.recovery_initial_signature:
            raise ValueError("Recovery loading changed the structurally pruned model.")
        if recovery_state_sha256(self.model) != self.recovery_initial_state_sha256:
            raise ValueError("Recovery loading changed the initial pruned weights.")
        self.recovery_initial_state_verified = True

    def get_model(self, cfg=None, weights=None, verbose=True):
        raise RuntimeError("Recovery must never rebuild the pruned model from its original YAML.")

    def optimizer_step(self):
        if not _recovery_torch.isfinite(self.loss).all().item():
            raise FloatingPointError("Recovery native detection loss became nonfinite.")
        if not all(_recovery_torch.isfinite(value).all().item() for value in self.loss_items.values()):
            raise FloatingPointError("Recovery native detection loss component became nonfinite.")
        super().optimizer_step()
        self.recovery_optimizer_steps += 1

    def _save_recovery_fp32(self, path, model):
        core = _recovery_native_snapshot(model)
        _recovery_torch.save({
            "model": core, "ema": None, "optimizer": None, "recovery_enabled": True,
            "epoch": int(self.epoch), "best_fitness": float(self.best_fitness),
            "train_args": vars(self.args).copy(), "version": _recovery_ultralytics_version,
            "date": _RecoveryDateTime.now(_RECOVERY_UTC).isoformat(),
            "recovery": {"format": "lamp-ordinary-recovery-fp32-v1",
                         "source_sha256": self.recovery_source_sha256,
                         "selection": "native validation fitness; EMA captured before SDK FP16 conversion"},
        }, path)
        return core

    def save_model(self):
        if recovery_structure_signature(self.model) != self.recovery_initial_signature:
            raise ValueError("Fine-tuning changed the pruned parameter shapes or Conv/BN dimensions.")
        if not _recovery_finite_state(self.model) or not _recovery_finite_state(self.ema.ema):
            raise FloatingPointError("Recovery model or EMA became nonfinite; no sanitized checkpoint is accepted.")
        if not _recovery_torch.isfinite(_recovery_torch.tensor(self.fitness)).item():
            raise FloatingPointError("Recovery validation fitness became nonfinite.")
        self.wdir.mkdir(parents=True, exist_ok=True)
        self._save_recovery_fp32(self.wdir / "last_fp32.pt", self.ema.ema)
        if self.best_fitness == self.fitness:
            self.recovery_best_core = self._save_recovery_fp32(self.wdir / "best_fp32.pt", self.ema.ema)
            self.recovery_best_epoch = int(self.epoch)
        row = {"epoch": int(self.epoch) + 1, "fitness": float(self.fitness),
               "losses": {k: float(v.detach().cpu()) for k, v in self.tloss.items()},
               "metrics": {k: float(v) for k, v in self.metrics.items()},
               "optimizer_steps": self.recovery_optimizer_steps,
               "learning_rates": [float(g["lr"]) for g in self.optimizer.param_groups]}
        self.recovery_epochs.append(row)
        _recovery_write_json(self.save_dir / "recovery_epoch_metrics.json", self.recovery_epochs)
        if self.recovery_epoch_callback is not None:
            self.recovery_epoch_callback(self, row)
        # Keep SDK best/last for native training diagnostics. The caller receives
        # the independently captured full-precision EMA, never these FP16 files.
        return super().save_model()


def recovery_training_arguments(checkpoint, data_yaml, output_dir, settings):
    """Explicit optimizer/protocol; callers can change the budget without changing method."""
    required = {"epochs", "optimizer", "lr0", "batch", "imgsz", "device", "seed"}
    if not required <= set(settings):
        raise ValueError(f"Missing recovery settings: {sorted(required - set(settings))}")
    if settings["optimizer"] == "auto":
        raise ValueError("Recovery requires an explicit optimizer so lr0 is not overridden.")
    if isinstance(settings["epochs"], bool) or not isinstance(settings["epochs"], int) or settings["epochs"] < 1:
        raise ValueError("Recovery epochs must be a positive integer.")
    arguments = {"model": str(checkpoint), "data": str(data_yaml), "project": str(_RecoveryPath(output_dir).parent),
                 "name": _RecoveryPath(output_dir).name, "exist_ok": False, "task": "detect", "mode": "train",
                 "resume": False, "pretrained": False, "amp": False, "quantize": None, "compile": False,
                 "distill_model": None, "freeze": None, "val": True, "save": True, "plots": True,
                 "patience": 0, "workers": 2, "cache": False, "deterministic": True, "channels_last": False}
    arguments.update(settings)
    for key, value in {"resume": False, "amp": False, "quantize": None, "compile": False,
                       "distill_model": None, "pretrained": False, "freeze": None}.items():
        if arguments[key] != value:
            raise ValueError(f"Ordinary FP32 recovery requires {key}={value!r}.")
    if not arguments["save"] or not arguments["val"]:
        raise ValueError("Recovery requires validation and checkpoint selection.")
    if not _RecoveryPath(checkpoint).is_file() or not _RecoveryPath(data_yaml).is_file():
        raise FileNotFoundError("Recovery checkpoint and dataset YAML must already exist locally.")
    return arguments


def fine_tune_pruned_checkpoint(checkpoint, data_yaml, output_dir, class_names, settings, epoch_callback=None):
    """Return (native best-EMA FP32 DetectionModel, report) after ordinary supervised fine-tuning."""
    output_dir = _RecoveryPath(output_dir)
    report_path = output_dir.parent / (output_dir.name + "_recovery_training_report.json")
    report = {"status": "RUNNING", "method": "ordinary-supervised-fine-tuning", "resume": False,
              "source_checkpoint": str(_RecoveryPath(checkpoint)), "errors": []}
    trainer = None
    try:
        arguments = recovery_training_arguments(checkpoint, data_yaml, output_dir, settings)
        source_sha256 = _recovery_hashlib.sha256(_RecoveryPath(checkpoint).read_bytes()).hexdigest()
        payload = _recovery_torch.load(checkpoint, map_location="cpu", weights_only=False)
        core = payload.get("ema") or payload.get("model")
        if type(core) is not _RecoveryDetectionModel:
            raise TypeError("Recovery input must contain a native structurally pruned DetectionModel.")
        if any(v.dtype != _recovery_torch.float32 for v in core.parameters() if v.is_floating_point()):
            raise ValueError("Recovery source must be saved in FP32; converting FP16 does not restore precision.")
        if recovery_ordered_names(core.names) != recovery_ordered_names(class_names):
            raise ValueError("Recovery source class order differs from requested classes.")
        report.update({"source_sha256": source_sha256, "arguments": arguments,
                       "initial_structure": recovery_structure_signature(core),
                       "initial_state_sha256": recovery_state_sha256(core)})
        _recovery_write_json(report_path, report)
        trainer = StructuralRecoveryTrainer(core, overrides=arguments, expected_names=class_names,
                                            source_sha256=source_sha256, epoch_callback=epoch_callback)
        trainer.train()
        if not trainer.recovery_initial_state_verified or trainer.recovery_optimizer_steps < 1:
            raise RuntimeError("Recovery did not verify the source or complete an optimizer step.")
        if trainer.recovery_best_core is None:
            raise RuntimeError("Recovery did not select a best validation EMA checkpoint.")
        if recovery_state_sha256(trainer.model, parameters_only=True) == trainer.recovery_initial_parameters_sha256:
            raise RuntimeError("Recovery completed without changing any trainable parameter.")
        recovered = trainer.recovery_best_core
        if recovery_structure_signature(recovered) != report["initial_structure"]:
            raise ValueError("Recovered model does not preserve the original pruned structure.")
        if not _recovery_finite_state(recovered):
            raise FloatingPointError("Recovered best EMA contains nonfinite weights.")
        report.update({"status": "PASS", "initial_state_verified": True, "structure_preserved": True,
                       "parameters_updated": True, "optimizer_steps": trainer.recovery_optimizer_steps,
                       "epochs_completed": len(trainer.recovery_epochs), "epochs": trainer.recovery_epochs,
                       "best_epoch": trainer.recovery_best_epoch + 1,
                       "best_selection": "native validation fitness; full-precision EMA snapshot",
                       "best_fitness": float(trainer.best_fitness),
                       "best_fp32_checkpoint": str(trainer.wdir / "best_fp32.pt"),
                       "last_fp32_checkpoint": str(trainer.wdir / "last_fp32.pt"),
                       "actual_save_dir": str(trainer.save_dir),
                       "filtered_sdk_integrations": sorted(trainer.recovery_filtered_callbacks),
                       "best_state_sha256": recovery_state_sha256(recovered)})
        _recovery_write_json(report_path, report)
        return recovered, report
    except Exception as error:
        report["status"] = "FAIL"
        report["errors"].append({"type": type(error).__name__, "message": str(error)})
        if trainer is not None:
            report.update({"optimizer_steps": trainer.recovery_optimizer_steps,
                           "epochs": trainer.recovery_epochs})
        _recovery_write_json(report_path, report)
        raise

In [ ]:
"""FP32 epoch resume for the existing structural recovery trainer (Ultralytics 8.4.155)."""
# ruff: noqa: F821, S102 -- Shared notebook declarations; compile only the installed pinned SDK loop.
import inspect
import random
import re
import textwrap
from copy import copy
from functools import lru_cache

import numpy as np
import torch
from ultralytics.engine import trainer as _sdk_trainer


@lru_cache(maxsize=1)
def recovery_native_loop():
    """Keep SDK loss/scheduler/EMA and patch its local accumulation cursor, not the SDK module."""
    if _recovery_ultralytics_version != "8.4.155":
        raise RuntimeError("Resume struktural memerlukan Ultralytics 8.4.155.")
    source = textwrap.dedent(inspect.getsource(_RecoveryDetectionTrainer._do_train))
    digest = hashlib.sha256(source.encode()).hexdigest()
    oom = re.compile(r"^([ ]+)except RuntimeError as e:.*?(?=^\1if ni - last_opt_step >= self.accumulate:)", re.MULTILINE | re.DOTALL)
    source, count = oom.subn(lambda match: match[1] + "except RuntimeError:\n" + match[1] + "    raise\n", source)
    if count != 1:
        raise RuntimeError("Kontrak loop SDK berubah pada penanganan OOM.")
    markers = {
        "last_opt_step = -1": "last_opt_step = self.recovery_last_opt_step",
        "last_opt_step = ni": "last_opt_step = ni\n                self.recovery_last_opt_step = ni",
        "loss, self.loss_items = self.model(batch)": "loss, self.loss_items = self._forward_recovery_batch(batch)",
        "self.optimizer.zero_grad()  # zero any resumed gradients to ensure stability on train start":
            "self._restore_pending_gradients()",
        '        if RANK in {-1, 0}:\n            if self.loss_names:':
            '        self._restart_epoch_loader(epoch)\n        if RANK in {-1, 0}:\n            if self.loss_names:',
    }
    for marker, replacement in markers.items():
        if source.count(marker) != 1:
            raise RuntimeError(f"Kontrak loop SDK tidak cocok: {marker}")
        source = source.replace(marker, replacement)
    namespace = dict(vars(_sdk_trainer))
    exec(compile(source, "<ultralytics-8.4.155-epoch-resume>", "exec"), namespace)
    return namespace["_do_train"], digest


def recovery_rng(device):
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None}


def recovery_restore_rng(state, device):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None:
        if device.type != "cuda":
            raise ValueError("Resume CUDA memerlukan device CUDA yang sebanding.")
        torch.cuda.set_rng_state(state["cuda"], device)


def recovery_atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.stem + "_" + uuid.uuid4().hex + ".pt")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def recovery_committed_state(folder, contract_sha, raw_sha):
    folder = Path(folder)
    index = folder / "latest_state.json"
    if not index.is_file():
        return None, None
    record = json.loads(index.read_text(encoding="utf-8"))
    state_path = folder / record["relative_path"]
    if sha256_file(state_path) != record["sha256"]:
        raise ValueError("SHA-256 checkpoint resume tidak cocok.")
    state = torch.load(state_path, map_location="cpu", weights_only=False)
    if (state.get("format_version") != 2 or state.get("stage") != "recovery_training_state"
            or state.get("contract_sha256") != contract_sha or state.get("raw_sha256") != raw_sha
            or state.get("epoch") != record["epoch"]):
        raise ValueError("Checkpoint resume tidak cocok dengan raw dan kontrak training.")
    best = folder / state["best"]["relative_path"]
    if sha256_file(best) != state["best"]["sha256"]:
        raise ValueError("Best EMA FP32 untuk resume tidak cocok.")
    return state, record


class ResumableStructuralTrainer(StructuralRecoveryTrainer):
    """Use the native pruned object and persist the full live training state per epoch."""

    def __init__(self, raw, *, overrides, expected_names, raw_sha, folder, contract_sha,
                 val_batch, resume_state=None, committed_callback=None):
        self.recovery_folder = Path(folder)
        self.recovery_contract_sha = contract_sha
        self.recovery_resume_state = resume_state
        self.recovery_val_batch = val_batch
        self.recovery_fixed_batch = overrides["batch"]
        self.recovery_last_opt_step = -1
        self.recovery_committed_callback = committed_callback
        self.recovery_best = None
        self.recovery_raw_state = recovery_state_sha256(raw)
        if resume_state is not None:
            if resume_state["raw_state_sha256"] != self.recovery_raw_state:
                raise ValueError("State raw awal berubah sebelum resume recovery.")
            for key in ("live_model", "ema"):
                model = resume_state[key]
                if (type(model) is not _RecoveryDetectionModel or recovery_structure_signature(model) != recovery_structure_signature(raw)
                        or any(p.dtype != torch.float32 for p in model.parameters())):
                    raise ValueError("Struktur atau presisi checkpoint training tidak sesuai raw.")
        core = resume_state["live_model"] if resume_state else raw
        super().__init__(core, overrides=overrides, expected_names=expected_names, source_sha256=raw_sha)

    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode="train"):
        return super().get_dataloader(dataset_path, self.recovery_val_batch if mode == "val" else batch_size, rank, mode)

    def _build_train_pipeline(self):
        if self.batch_size != self.recovery_fixed_batch or self.args.batch != self.recovery_fixed_batch:
            raise RuntimeError("Batch efektif berubah dari kontrak recovery.")
        super()._build_train_pipeline()
        self.args.max_det = 300
        self.model.set_head_attr(max_det=300)

    def _setup_train(self):
        super()._setup_train()
        state = self.recovery_resume_state
        if not state:
            return
        self.start_epoch = state["epoch"]
        if not 0 < self.start_epoch < self.epochs:
            raise ValueError("Epoch resume di luar budget total.")
        self.optimizer.load_state_dict(state["optimizer"])
        self.scheduler.load_state_dict(state["scheduler"])
        self.scaler.load_state_dict(state["scaler"])
        self.ema.ema.load_state_dict(state["ema"].state_dict(), strict=True)
        self.ema.updates = state["ema_updates"]
        self.ema.ema.end2end = state["ema"].end2end
        self.accumulate = state["accumulate"]
        self.recovery_last_opt_step = state["last_opt_step"]
        self.recovery_optimizer_steps = state["optimizer_steps"]
        self.best_fitness = state["best"]["fitness"]
        self.recovery_best = state["best"]
        self.recovery_best_epoch = self.recovery_best["epoch"] - 1
        self.recovery_epochs = list(state["epoch_rows"])
        self.loss_names = tuple(state["loss_names"])
        self.recovery_restore_criterion(state["criterion"])
        self.recovery_restore_extra(state.get("method_state", {}))
        if self.args.close_mosaic and self.start_epoch > self.epochs - self.args.close_mosaic:
            self._close_dataloader_mosaic()
        if state.get("loader_generator") is not None:
            self.train_loader.generator.set_state(state["loader_generator"])
        recovery_restore_rng(state["rng"], self.device)

    def recovery_criterion_state(self):
        return {name: getattr(self.model.criterion, name) for name in ("updates", "o2m", "o2o")
                if hasattr(self.model.criterion, name)}

    def recovery_restore_criterion(self, state):
        self.model.criterion = self.model.init_criterion()
        for key, value in state.items():
            setattr(self.model.criterion, key, value)

    def recovery_extra_state(self):
        return {}

    def _forward_recovery_batch(self, batch):
        return self.model(batch)

    def recovery_restore_extra(self, state):
        if state:
            raise ValueError("Checkpoint memuat state metode lain.")

    def _restart_epoch_loader(self, epoch):
        # Reset the augmentation buffer and workers at each epoch boundary.
        # This makes sampling independent of the previous iterator's prefetch.
        dataset = self.train_loader.dataset
        for index in dataset.buffer:
            dataset.ims[index] = dataset.im_hw0[index] = dataset.im_hw[index] = None
        dataset.buffer.clear()
        if self.train_loader.num_workers == 0:
            # No worker fork isolates optional Albumentations' internal RNG here.
            if self.args.close_mosaic and epoch >= self.epochs - self.args.close_mosaic:
                dataset.close_mosaic(hyp=copy(self.args))
            else:
                dataset.transforms = dataset.build_transforms(hyp=copy(self.args))
        self.train_loader.generator.manual_seed((6148914691236517205 + self.args.seed + epoch) % (1 << 64))
        self.train_loader.reset()

    def _restore_pending_gradients(self):
        self.optimizer.zero_grad()
        state = self.recovery_resume_state
        if state:
            parameters = dict(self.model.named_parameters())
            for name, grad in state["gradients"].items():
                parameter = parameters[name]
                if grad.shape != parameter.shape or grad.dtype != parameter.dtype:
                    raise ValueError("Gradien resume tidak sesuai tensor raw.")
                parameter.grad = grad.to(parameter.device).clone()

    def _do_train(self):
        method, _ = recovery_native_loop()
        return method(self)

    def _handle_nan_recovery(self, epoch):
        if not torch.isfinite(self.loss).all() or not math.isfinite(self.fitness):
            raise FloatingPointError(f"Loss atau fitness nonfinite pada epoch {epoch + 1}.")
        return False

    def validate(self):
        with gflops_require_complete_nms():
            result = super().validate()
        if (self.validator.end2end or self.validator.args.quantize is not None or self.validator.args.nms is not None
                or self.test_loader.batch_size != min(self.recovery_val_batch, len(self.test_loader.dataset))):
            raise ValueError("Validator per epoch tidak memakai protokol FP32 one-to-many/batch yang dikunci.")
        return result

    def final_eval(self):
        # Notebook measures three immutable FP32 stages using the same evaluator.
        return None

    def save_model(self):
        if (recovery_structure_signature(self.model) != self.recovery_initial_signature
                or not _recovery_finite_state(self.model) or not _recovery_finite_state(self.ema.ema)):
            raise ValueError("Training mengubah struktur raw atau menghasilkan state nonfinite.")
        epoch = int(self.epoch) + 1
        folder = self.recovery_folder
        if (self.recovery_best is None or self.fitness > self.recovery_best["fitness"]
                or (not self.recovery_best.get("optimizer_steps") and self.recovery_optimizer_steps)):
            best_path = folder / "checkpoints" / f"best-{epoch:03d}-{uuid.uuid4().hex[:8]}.pt"
            best_path.parent.mkdir(parents=True, exist_ok=True)
            inference = _recovery_native_snapshot(self.ema.ema)
            inference.end2end = True
            self._save_recovery_fp32(best_path, inference)
            self.recovery_best = {"epoch": epoch, "fitness": float(self.fitness), "sha256": sha256_file(best_path),
                                  "optimizer_steps": self.recovery_optimizer_steps,
                                  "relative_path": best_path.relative_to(folder).as_posix()}
        row = {"epoch": epoch, "fitness": float(self.fitness), "optimizer_steps": self.recovery_optimizer_steps,
               "losses": {name: float(value.detach().cpu()) for name, value in self.tloss.items()},
               "metrics": {name: float(value) for name, value in self.metrics.items()},
               "learning_rates": [float(group["lr"]) for group in self.optimizer.param_groups]}
        self.recovery_epochs.append(row)
        rng = recovery_rng(self.device)
        identity_path = folder / "wandb_identity.json"
        state = {"format_version": 2, "stage": "recovery_training_state", "epoch": epoch,
                 "epochs_total": self.epochs, "raw_sha256": self.recovery_source_sha256,
                 "raw_state_sha256": self.recovery_raw_state, "contract_sha256": self.recovery_contract_sha,
                 "live_model": _recovery_native_snapshot(self.model), "ema": _recovery_native_snapshot(self.ema.ema),
                 "ema_updates": self.ema.updates, "optimizer": self.optimizer.state_dict(),
                 "scheduler": self.scheduler.state_dict(), "scaler": self.scaler.state_dict(),
                 "optimizer_steps": self.recovery_optimizer_steps, "last_opt_step": self.recovery_last_opt_step,
                 "accumulate": self.accumulate, "gradients": {name: value.grad.detach().cpu().clone()
                     for name, value in self.model.named_parameters() if value.grad is not None},
                 "criterion": self.recovery_criterion_state(), "method_state": self.recovery_extra_state(),
                 "rng": rng, "loader_generator": self.train_loader.generator.get_state(),
                 "sampler": {"completed_epoch": epoch, "policy": "seeded shuffle and fresh workers/mosaic buffer per epoch"},
                 "loss_names": list(self.loss_names), "epoch_rows": self.recovery_epochs, "best": self.recovery_best,
                 "wandb_identity": json.loads(identity_path.read_text(encoding="utf-8")) if identity_path.is_file() else None}
        path = folder / "checkpoints" / f"resume-{epoch:03d}-{uuid.uuid4().hex[:8]}.pt"
        recovery_atomic_torch_save(state, path)
        index = {"epoch": epoch, "relative_path": path.relative_to(folder).as_posix(), "sha256": sha256_file(path),
                 "best": self.recovery_best, "optimizer_steps": self.recovery_optimizer_steps}
        write_json(folder / "epoch_metrics.json", self.recovery_epochs)
        write_json(folder / "latest_state.json", index)
        if self.recovery_committed_callback:
            self.recovery_committed_callback(index, row)
        recovery_restore_rng(rng, self.device)
        return True


def recovery_fit(raw, raw_path, raw_sha, data_yaml, names, folder, settings, contract_sha, *,
                 val_batch, committed_callback=None, trainer_class=ResumableStructuralTrainer, trainer_kwargs=None):
    """Resume only from a committed epoch; retry that publication before another step."""
    folder = Path(folder)
    state, index = recovery_committed_state(folder, contract_sha, raw_sha)
    if state and committed_callback:
        write_json(folder / "epoch_metrics.json", state["epoch_rows"])
        committed_callback(index, state["epoch_rows"][-1])
    if state and state["epoch"] == settings["epochs"]:
        if (state["optimizer_steps"] < 1 or recovery_state_sha256(state["live_model"], parameters_only=True)
                == recovery_state_sha256(raw, parameters_only=True)):
            raise RuntimeError("Training selesai tanpa perubahan bobot raw.")
        return state, index
    attempt = folder / "attempts" / uuid.uuid4().hex[:10]
    arguments = recovery_training_arguments(raw_path, data_yaml, attempt, settings)
    arguments.update(conf=0.001, iou=0.7, max_det=300, nms=None)
    write_json(folder / "training_config.json", {"settings": settings, "validation_batch": val_batch,
               "sdk_loop_sha256": recovery_native_loop()[1], "raw_sha256": raw_sha})
    trainer = trainer_class(raw, overrides=arguments, expected_names=names, raw_sha=raw_sha, folder=folder,
                            contract_sha=contract_sha, val_batch=val_batch, resume_state=state,
                            committed_callback=committed_callback, **(trainer_kwargs or {}))
    with gflops_disable_label_cache():
        trainer.train()
    state, index = recovery_committed_state(folder, contract_sha, raw_sha)
    if (not trainer.recovery_initial_state_verified or not state or state["epoch"] != settings["epochs"]
            or state["optimizer_steps"] < 1 or recovery_state_sha256(state["live_model"], parameters_only=True)
            == recovery_state_sha256(raw, parameters_only=True)):
        raise RuntimeError("Training belum menyelesaikan budget atau belum mengubah bobot raw.")
    return state, index

In [ ]:
"""EDA for measured baseline, raw pruned, and recovered DetectionModels."""
# ruff: noqa: F821 -- Notebook cells supply shared persistence and EDA declarations.

RECOVERY_STAGE_STYLES = {
    "baseline": ("Baseline", "#2863AD"),
    "raw_pruned": ("Raw pruned", "#BA492E"),
    "recovered": ("Recovered", "#217854"),
}
RECOVERY_ACCURACY_FIELDS = {"map50_95": "mAP50–95", "map50": "mAP50", "precision": "Precision", "recall": "Recall"}
RECOVERY_CLASS_FIELDS = {"ap50_95": "AP50–95", "ap50": "AP50", "precision": "Precision", "recall": "Recall", "f1": "F1"}
RECOVERY_LATENCY_CONTRACT = ("device", "precision", "batch", "imgsz", "fusion", "scope", "warmup", "repeats")
RECOVERY_EDA_METHODS = {"finetune": "Fine-tuning standar", "native": "Native KD", "crosskd": "CrossKD", "mgd": "MGD"}


def _recovery_eda_num(value):
    return value if _eda_finite(value) else None


def _recovery_eda_ratio(before, after):
    return before / after if before is not None and after is not None and after > 0 else None


def _recovery_eda_delta(after, before):
    return 100 * (after - before) if after is not None and before is not None else None


def _recovery_eda_metadata(result):
    return {key: result.get(key) for key in ("model_variant", "method", "platform", "mode", "configuration_id",
                                             "target_speedup", "target_status", "alias_of", "raw_sha256")}


def recovery_comparison_rows(result):
    """Use successful measurements only and join class data by stable ID."""
    stages = result.get("stages", {})
    metadata = _recovery_eda_metadata(result)
    rows, per_class_by_stage, class_names = [], {}, {}
    latency_rows = [stage.get("latency", {}) for stage in stages.values()]
    latency_contracts = [tuple(item.get(key) for key in RECOVERY_LATENCY_CONTRACT)
                         for item in latency_rows if item.get("status") in {"PASS", "MEASURED"}]
    comparable_latency = all(contract == latency_contracts[0] for contract in latency_contracts)
    for stage in RECOVERY_STAGE_STYLES:
        measured = stages.get(stage, {})
        validation_status = measured.get("validation_status", "NOT_RUN")
        validation = measured.get("validation", {}) if validation_status == "PASS" else {}
        latency = measured.get("latency", {})
        latency_status = latency.get("status", "NOT_RUN")
        if latency_status in {"PASS", "MEASURED"} and not comparable_latency:
            latency_status = "NOT_COMPARABLE"
        latency = latency if latency_status in {"PASS", "MEASURED"} else {}
        rows.append({**metadata, "stage": stage, "validation_status": validation_status, "latency_status": latency_status,
                     "validation_image_count": validation.get("validation_image_count"),
                     "validation_fingerprint": validation.get("dataset_fingerprint"),
                     "validation_device": validation.get("validation_arguments_effective", {}).get("device"),
                     "latency_device": latency.get("device"), "latency_scope": latency.get("scope"),
                     **{key: _recovery_eda_num(validation.get(key)) for key in RECOVERY_ACCURACY_FIELDS},
                     **{key: _recovery_eda_num(measured.get(key)) for key in ("parameters", "gflops", "checkpoint_bytes", "conv_output_channels")},
                     "latency_median_ms": _recovery_eda_num(latency.get("median_ms")),
                     "latency_p95_ms": _recovery_eda_num(latency.get("p95_ms"))})
        indexed = {}
        for item in validation.get("per_class", []):
            class_id, name = item["class_id"], item["name"]
            if class_id in indexed or class_names.setdefault(class_id, name) != name:
                raise ValueError(f"Identitas kelas tidak konsisten: {stage}, {class_id}, {name}")
            indexed[class_id] = item
        per_class_by_stage[stage] = indexed
    baseline, raw, recovered = rows
    class_rows = []
    for row in rows:
        stage = row["stage"]
        row["gflops_speedup"] = _recovery_eda_ratio(baseline["gflops"], row["gflops"])
        row["latency_speedup"] = _recovery_eda_ratio(baseline["latency_median_ms"], row["latency_median_ms"])
        for key in RECOVERY_ACCURACY_FIELDS:
            row[key + "_drop_pp"] = _recovery_eda_delta(baseline[key], row[key])
            row[key + "_recovery_gain_pp"] = _recovery_eda_delta(recovered[key], raw[key]) if stage == "recovered" else None
        for key, name in (("parameters", "parameter"), ("gflops", "gflops"), ("conv_output_channels", "channel"),
                          ("checkpoint_bytes", "checkpoint")):
            ratio = _recovery_eda_ratio(row[key], baseline[key])
            row[name + "_reduction_pct"] = None if ratio is None else 100 * (1 - ratio)
        for class_id, name in sorted(class_names.items()):
            record = per_class_by_stage[stage].get(class_id, {})
            class_row = {**metadata, "stage": stage, "class_id": class_id, "name": name,
                         "support": _recovery_eda_num(record.get("support")),
                         **{key: _recovery_eda_num(record.get(key)) for key in RECOVERY_CLASS_FIELDS}}
            for key in ("ap50_95", "ap50"):
                class_row[key + "_drop_pp"] = _recovery_eda_delta(
                    _recovery_eda_num(per_class_by_stage["baseline"].get(class_id, {}).get(key)), class_row[key])
                class_row[key + "_recovery_gain_pp"] = _recovery_eda_delta(
                    class_row[key], _recovery_eda_num(per_class_by_stage["raw_pruned"].get(class_id, {}).get(key))
                ) if stage == "recovered" else None
            class_rows.append(class_row)
    return rows, class_rows


def recovery_epoch_rows(rows):
    """Flatten epoch records for plotting and logging without inventing missing loss values."""
    return [{"epoch": row["epoch"], "optimizer_steps": row.get("optimizer_steps"), "fitness": row.get("fitness"),
             **{f"train/{name.removeprefix('train/')}": _recovery_eda_num(value)
                for name, value in row.get("losses", {}).items()},
             **{name: _recovery_eda_num(value) for name, value in row.get("metrics", {}).items()},
             **{f"lr/pg{index}": _recovery_eda_num(value) for index, value in enumerate(row.get("learning_rates", []))}}
            for row in rows]


def _recovery_draw_bar_panel(ax, records, *, title, unit, bounded=False):
    groups = list(dict.fromkeys(record["group"] for record in records))
    labels = {record["group"]: record["group_label"] for record in records}
    positions = {group: index for index, group in enumerate(groups)}
    width = .24
    plotted = 0
    for offset, (stage, (_label, color)) in enumerate(RECOVERY_STAGE_STYLES.items()):
        observed = [record for record in records if record["stage"] == stage and _eda_finite(record["value"])]
        if observed:
            bars = ax.bar([positions[record["group"]] + (offset - 1) * width for record in observed],
                          [record["value"] for record in observed], width=width, color=color)
            if len(groups) <= 4:
                ax.bar_label(bars, labels=[f"{record['value']:.4g}" for record in observed], fontsize=7, padding=2)
            plotted += len(observed)
    if not plotted:
        ax.text(.5, .55, "Belum terukur", ha="center", va="center", transform=ax.transAxes, color="#57616B")
    ax.set_xticks(list(positions.values()), [labels[group] for group in groups],
                  rotation=30 if len(groups) > 4 else 0, ha="right" if len(groups) > 4 else "center")
    ax.set_title(title)
    ax.set_ylabel(unit)
    if bounded:
        ax.set_ylim(0, 1.1)
    else:
        ax.margins(y=.15)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=.18)
    return plotted


def recovery_plot_stage_bars(result, folder, rows=None, class_rows=None):
    """Export exact plot inputs and separate axes for each unit of measurement."""
    from matplotlib.patches import Patch

    folder = _EdaPath(folder)
    if rows is None or class_rows is None:
        rows, class_rows = recovery_comparison_rows(result)
    figures, data = [], []
    metadata = _recovery_eda_metadata(result)
    variant = result.get("model_variant", "?")
    target = result.get("target_speedup")
    target_label = f"{target:.2f}×" if _eda_finite(target) else "—"
    status = result.get("target_status") or "NOT_RUN"
    method = RECOVERY_EDA_METHODS.get(result.get("method"), result.get("method") or "Fine-tuning standar")
    achieved = rows[1]["gflops_speedup"]
    achieved_label = f"{achieved:.3f}×" if _eda_finite(achieved) else "belum terukur"

    def record(panel, group, label, row, key, unit):
        return {**metadata, "panel": panel, "group": group, "group_label": label, "stage": row["stage"],
                "metric": key, "value": row[key], "unit": unit}

    def plot(name, panels, *, group_cols=2):
        nrows = (len(panels) + group_cols - 1) // group_cols
        fig, axes = _eda_plt.subplots(nrows, group_cols, figsize=(6.3 * group_cols, 3.5 * nrows + 1.6), squeeze=False)
        for ax, (title, unit, records, bounded) in zip(axes.flat, panels):
            _recovery_draw_bar_panel(ax, records, title=title, unit=unit, bounded=bounded)
            data.extend({"figure": name, **item} for item in records)
        for ax in list(axes.flat)[len(panels):]:
            ax.set_visible(False)
        fig.suptitle(f"YOLO26{variant}  |  {method}  |  Target {target_label}  |  Aktual {achieved_label}\n{status}",
                     fontsize=13, y=.99)
        fig.legend(handles=[Patch(facecolor=color, label=label) for label, color in RECOVERY_STAGE_STYLES.values()],
                   loc="upper center", bbox_to_anchor=(.5, .885), ncol=3, frameon=False)
        fig.text(.5, .01, "Nilai yang belum diukur dibiarkan kosong. Tinggi nol hanya untuk nilai nol yang terukur.",
                 ha="center", fontsize=9, color="#57616B")
        fig.tight_layout(rect=(0, .025, 1, .82))
        figures.append(_eda_save_figure(fig, folder / f"comparison_{name}"))

    def panel(title, unit, metrics, bounded=False):
        return (title, unit, [record(title, key, label, row, key, unit)
                              for key, label in metrics for row in rows], bounded)

    plot("accuracy", [panel("Metrik deteksi", "Proporsi", tuple(RECOVERY_ACCURACY_FIELDS.items()), True)], group_cols=1)
    plot("complexity", [panel(title, unit, ((key, label),)) for key, title, unit, label in (
        ("parameters", "Parameter", "Jumlah parameter", "Parameter"),
        ("gflops", "Komputasi model", "GFLOPs", "Komputasi"),
        ("checkpoint_bytes", "Checkpoint FP32", "Byte", "Ukuran"),
        ("conv_output_channels", "Jumlah channel", "Total output Conv2d", "Channel"))])
    plot("latency", [panel("Raw forward FP32 · batch 1", "Milidetik", (("latency_median_ms", "p50"), ("latency_p95_ms", "p95"))),
                     panel("Speed-up terhadap baseline", "Rasio (×)", (("gflops_speedup", "GFLOPs"), ("latency_speedup", "Latency p50")))])
    plot("changes", [panel("Penurunan metrik terhadap baseline", "Poin persentase",
                           tuple((key + "_drop_pp", label) for key, label in RECOVERY_ACCURACY_FIELDS.items())),
                     panel("Reduksi terhadap baseline", "%", (("parameter_reduction_pct", "Parameter"),
                           ("gflops_reduction_pct", "GFLOPs"), ("checkpoint_reduction_pct", "Checkpoint"),
                           ("channel_reduction_pct", "Channel"))),
                     panel("Recovery gain terhadap raw pruned", "Poin persentase",
                           tuple((key + "_recovery_gain_pp", label) for key, label in RECOVERY_ACCURACY_FIELDS.items()))])
    plot("per_class", [(label + " per kelas", "Proporsi", [record(label, str(row["class_id"]), row["name"], row, key, "Proporsi")
                          for row in class_rows], True) for key, label in list(RECOVERY_CLASS_FIELDS.items())[:2]])
    write_json(folder / "plot_data.json", data)
    write_csv(folder / "plot_data.csv", data)
    return figures


def _recovery_epoch_plots(rows, folder):
    losses = sorted({key.split("/", 1)[1] for row in rows for key in row if key.startswith(("train/", "val/"))})
    groups = [(key, (f"train/{key}", f"val/{key}"), "Loss") for key in losses]
    groups.extend((
        ("Akurasi validation", ("metrics/mAP50-95(B)", "metrics/mAP50(B)", "metrics/precision(B)", "metrics/recall(B)"), "Proporsi"),
        ("Learning rate", sorted({key for row in rows for key in row if key.startswith("lr/")}), "Learning rate"),
        ("Optimizer", ("optimizer_steps",), "Jumlah langkah kumulatif"),
    ))
    files = []
    for index, (title, keys, ylabel) in enumerate(groups):
        observed = {key: [(row["epoch"], row[key]) for row in rows if _eda_finite(row.get(key))] for key in keys}
        if not any(observed.values()):
            continue
        fig, ax = _eda_plt.subplots(figsize=(9, 4))
        for key, points in observed.items():
            if points:
                ax.plot(*zip(*points), label=key, marker="." if len(points) < 10 else None)
        ax.set(title=title, xlabel="Epoch", ylabel=ylabel)
        ax.grid(alpha=.18)
        ax.legend(frameon=False)
        fig.tight_layout()
        files.append(_eda_save_figure(fig, folder / f"epochs_{index:02d}"))
    return files


def _recovery_stage_validation_plots(result, folder):
    reports = {}
    for stage, (label, _color) in RECOVERY_STAGE_STYLES.items():
        measured = result.get("stages", {}).get(stage, {})
        validation = measured.get("validation", {}) if measured.get("validation_status") == "PASS" else {}
        if not validation:
            reports[stage] = {"status": "NOT_RUN"}
            continue
        target = folder / stage
        curves = validation.get("curves", {})
        write_json(target / "curves.json", curves)
        curve_report = _eda_curve_plots(curves, target)
        matrix = validation.get("confusion_matrix", {})
        matrix_files = []
        if matrix.get("status") == "MEASURED":
            write_json(target / "confusion_matrix.json", matrix)
            labels = matrix["labels"]
            for field, title, cmap in (("matrix", "Jumlah prediksi", "Blues"),
                                       ("normalized_by_true", "Normalisasi per kelas aktual", "Purples")):
                values = _eda_np.asarray(matrix[field], dtype=float)
                if values.shape != (len(labels), len(labels)) or not _eda_np.isfinite(values).all():
                    raise ValueError(f"Confusion matrix tidak valid: {stage}/{field}")
                write_csv(target / (field + ".csv"), [{"predicted_class": labels[i], **dict(zip(labels, row.tolist()))}
                                                     for i, row in enumerate(values)])
                fig, ax = _eda_plt.subplots(figsize=(9, 8))
                im = ax.imshow(values, cmap=cmap, vmin=0, vmax=1 if field == "normalized_by_true" else None)
                ax.set_xticks(range(len(labels)), labels, rotation=35, ha="right")
                ax.set_yticks(range(len(labels)), labels)
                ax.set(xlabel="Kelas aktual", ylabel="Kelas prediksi",
                       title=f"{label} · {title}\nconfidence {matrix.get('confidence_threshold')} | matching IoU {matrix.get('matching_iou_threshold')}")
                fig.colorbar(im, ax=ax)
                fig.tight_layout()
                matrix_files.append(_eda_save_figure(fig, target / field))
        reports[stage] = {"status": "PASS" if matrix_files and curve_report["status"] == "MEASURED" else "PARTIAL",
                          "curves": curve_report, "confusion_matrix": matrix_files}
    return reports


def _recovery_pruning_diagnostics(result, files, folder):
    if "inventories.json" not in files:
        return {"status": "NOT_RUN", "reason": "Manifest input belum menyertakan inventory pruning."}
    inventories = recovery_read_json(files["inventories.json"])
    rows = layer_eda_rows(inventories["before"], inventories["after"])
    conv = [row for row in rows if row["module_type"] == "Conv2d"]
    stages = stage_eda_rows(rows)
    audit = recovery_read_json(files["dependency_audit.json"]) if "dependency_audit.json" in files else {}
    write_csv(folder / "layer_eda.csv", rows)
    write_csv(folder / "stage_eda.csv", stages)
    channels = _eda_channel_plots(conv, folder, f"YOLO26{result['model_variant']} · struktur baseline dan raw")
    mapping = _eda_mapping_plots(audit, conv, folder)
    importance = _eda_lamp_plots(audit, folder)
    return {"status": "PASS" if mapping["status"] == importance["status"] == "MEASURED" else "PARTIAL",
            "channels": channels, "channel_mapping": mapping, "lamp_scores": importance}


def recovery_build_alias_eda(result, job_folder):
    """Write target-specific summary plots even when the raw checkpoint is shared."""
    folder = _EdaPath(job_folder) / "eda"
    rows, class_rows = recovery_comparison_rows(result)
    write_json(folder / "stage_metrics.json", rows)
    write_csv(folder / "stage_metrics.csv", rows)
    write_json(folder / "per_class_metrics.json", class_rows)
    write_csv(folder / "per_class_metrics.csv", class_rows)
    figures = recovery_plot_stage_bars(result, folder, rows, class_rows)
    if result.get("alias_of"):
        write_json(folder / "eda_status.json", {"status": result.get("eda_status", "NOT_RUN"), "bars": figures,
                   "canonical_diagnostics": f"../../{result['alias_of']}/eda"})
    return figures


def recovery_build_eda(result, folder, epoch_rows, files):
    folder = _EdaPath(folder)
    target = folder / "eda"
    bars = recovery_build_alias_eda(result, folder)
    epochs = recovery_epoch_rows(epoch_rows)
    write_json(target / "epoch_metrics.json", epochs)
    write_csv(target / "epoch_metrics.csv", epochs)
    report = {"status": "PASS", "bars": bars, "epochs": _recovery_epoch_plots(epochs, target / "training"),
              "validation": _recovery_stage_validation_plots(result, target / "validation"),
              "pruning_diagnostics": _recovery_pruning_diagnostics(result, files, target / "pruning")}
    if (not report["epochs"] or report["pruning_diagnostics"]["status"] != "PASS"
            or any(value["status"] != "PASS" for value in report["validation"].values())):
        report["status"] = "PARTIAL"
    write_json(target / "eda_status.json", report)
    return report


def recovery_log_comparison(run, result, folder):
    """Log the exact local tables and comparison figures to the recovery run."""
    import wandb

    rows, classes = recovery_comparison_rows(result)
    payload = {}
    for name, records in (("three_stage_metrics", rows), ("three_stage_per_class", classes)):
        if records:
            keys = list(dict.fromkeys(key for row in records for key in row))
            payload[f"recovery/{name}"] = wandb.Table(columns=keys, data=[[row.get(key) for key in keys] for row in records])
    for path in sorted((_EdaPath(folder) / "eda").glob("comparison_*.png")):
        payload[f"recovery/{path.stem}"] = wandb.Image(str(path))
    run.log(payload)


def recovery_suite_tables(report, folder):
    """Export summaries for all completed/failed targets and the measured trade-offs."""
    folder = _EdaPath(folder)
    rows, classes, summary = [], [], []
    for job in report.get("jobs", []):
        metrics, per_class = recovery_comparison_rows(job)
        rows.extend(metrics)
        classes.extend(per_class)
        recovered = metrics[-1]
        summary.append({**_recovery_eda_metadata(job), **{key: job.get(key) for key in (
            "status", "input_status", "training_status", "structural_status", "reload_status", "validation_status",
            "latency_status", "eda_status", "upload_status", "completed_epochs", "best_epoch", "raw_artifact_ref")},
            "actual_gflops_speedup": job.get("actual_gflops_speedup"), "baseline_map50_95": metrics[0]["map50_95"],
            "raw_map50_95": metrics[1]["map50_95"], "recovered_map50_95": recovered["map50_95"],
            "map50_95_recovery_gain_pp": recovered["map50_95_recovery_gain_pp"],
            "parameters": recovered["parameters"], "gflops": recovered["gflops"],
            "latency_median_ms": recovered["latency_median_ms"], "latency_p95_ms": recovered["latency_p95_ms"]})
    for name, table in (("recovery_summary", summary), ("stage_metrics", rows), ("per_class_metrics", classes)):
        write_json(folder / (name + ".json"), table)
        write_csv(folder / (name + ".csv"), table)
    if rows:
        fig, axes = _eda_plt.subplots(1, 3, figsize=(14, 4.3))
        for ax, (metric, xlabel) in zip(axes, (("gflops", "GFLOPs"), ("parameters", "Parameter"),
                                             ("latency_median_ms", "Raw forward p50 (ms)"))):
            for stage, (label, color) in RECOVERY_STAGE_STYLES.items():
                points = {(row[metric], row["map50_95"]) for row in rows if row["stage"] == stage
                          and _eda_finite(row[metric]) and _eda_finite(row["map50_95"])}
                if points:
                    ax.scatter(*zip(*sorted(points)), label=label, color=color, alpha=.85)
            ax.set(xlabel=xlabel, ylabel="mAP50–95")
            ax.grid(alpha=.18)
            if ax.collections:
                ax.legend(frameon=False, fontsize=8)
            else:
                ax.text(.5, .5, "Belum terukur", ha="center", transform=ax.transAxes)
        fig.suptitle("Trade-off semua target pada suite dan perangkat yang sama")
        fig.tight_layout()
        _eda_save_figure(fig, folder / "recovery_tradeoffs")
    return {"summary": summary, "stage_metrics": rows, "per_class_metrics": classes}

In [ ]:
"""Run independent standard fine-tuning jobs from verified raw artifacts."""
# ruff: noqa: F821 -- Standalone notebook declarations share one namespace.
# ruff: noqa: BLE001 -- Independent stages record failures and keep completed configurations.
import shutil


def recovery_read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def recovery_dataset(folder):
    folder = Path(folder)
    expected = None
    portable = None
    if DATASET_MANIFEST_PATH:
        portable = recovery_read_json(DATASET_MANIFEST_PATH)
        if portable["dataset_id"] != DATASET_ID or portable["provenance_status"] != "VERIFIED":
            raise ValueError("Gunakan manifest Poltrada V3 sumber yang VERIFIED.")
        expected = portable["val_fingerprint"]
    dataset = gflops_prepare_dataset(MODE, DATA_YAML, DATASET_ROOT, folder, seed=SEED,
                                    expected_fingerprint=expected, dataset_id=DATASET_ID)
    if dataset is None:
        raise FileNotFoundError("Isi DATASET_ROOT dan DATA_YAML dari Poltrada V3 asli, dengan train dan val.")
    record, train, _ = workflow_dataset_manifest(dataset, DATA_YAML, DATASET_ROOT, folder)
    if portable and record["train_fingerprint"] != portable["train_fingerprint"]:
        raise ValueError("Fingerprint train berbeda dari sumber dataset.")
    if MODE == "smoke":
        root = Path(DATASET_ROOT).resolve()
        train = sorted(sorted(train, key=lambda image: hashlib.sha256(
            f"{SEED}:{image.relative_to(root).as_posix()}".encode()).hexdigest())[:128])
    train_fp, files = _gflops_fingerprint(Path(DATASET_ROOT).resolve(), train, "train", record["class_names"])
    train_file = folder / "train_images.txt"
    train_file.write_text("".join(f"{path.as_posix()}\n" for path in train), encoding="utf-8")
    train_yaml = folder / "train_val_data.yaml"
    train_yaml.write_text(yaml.safe_dump({**dataset["info"], "train": str(train_file.resolve())},
                                         sort_keys=False, allow_unicode=True), encoding="utf-8")
    write_json(folder / "train_manifest.json", {"image_count": len(train), "fingerprint": train_fp, "files": files})
    return dataset, record, train_yaml, train_fp


def recovery_discover_inputs(folder):
    """Accept portable pruning suites, explicit manifests and pinned W&B versions."""
    sources = {}

    def add(path, reference=None):
        path = Path(path).expanduser().resolve()
        manifest = recovery_read_json(path)
        target = manifest.get("target_speedup")
        if manifest.get("model_variant") != NOTEBOOK_VARIANT or target not in GFLOPS_TARGETS:
            return
        if manifest["configuration_id"] != f"target-{target:.2f}x":
            raise ValueError("ID manifest raw tidak sesuai target GFLOPs.")
        existing = sources.get(target)
        if existing and existing["manifest"]["raw_checkpoint"]["sha256"] != manifest["raw_checkpoint"]["sha256"]:
            raise ValueError(f"Dua raw berbeda diajukan untuk target {target:.2f}×.")
        if not existing or reference:
            sources[target] = {"path": path, "manifest": manifest, "reference": reference}

    errors = []
    if PRUNING_SUITE_DIR:
        root = Path(PRUNING_SUITE_DIR).expanduser().resolve()
        try:
            rows = recovery_read_json(root / "student_candidates.json")["rows"]
        except Exception as error:
            rows = []
            errors.append({"source": str(root), "error": safe_error(error)})
        for row in rows:
            relative = row.get("checkpoint_relative_path")
            if row["model_variant"] == NOTEBOOK_VARIANT and row.get("target_speedup") in GFLOPS_TARGETS and relative:
                try:
                    # Resume a partial pruning suite from whichever manifest passed structural checks.
                    add((root / relative.replace("\\", "/")).parent / "raw_pruned_manifest.json")
                    target = row["target_speedup"]
                    # An aliased target reuses the exact saved structure and raw version.
                    if target not in sources and row.get("configuration_id") == f"target-{target:.2f}x":
                        canonical = next(value for value in sources.values()
                                         if value["manifest"]["raw_checkpoint"]["sha256"] == row["checkpoint_sha256"])
                        sources[target] = {**canonical, "alias": True}
                except Exception as error:
                    errors.append({"source": relative, "target_speedup": row.get("target_speedup"), "error": safe_error(error)})
    for filename in RAW_MANIFEST_PATHS:
        try:
            add(filename)
        except Exception as error:
            errors.append({"source": str(filename), "error": safe_error(error)})
    if RAW_ARTIFACT_REFS:
        import wandb
        workflow_login()
        api = wandb.Api()
        for ref in RAW_ARTIFACT_REFS:
            try:
                ref = workflow_version_ref(ref)
                artifact = api.artifact(ref, type="model")
                location = Path(artifact.download(root=str(folder / "inputs" / workflow_digest(ref)[:16])))
                add(location / "raw_pruned_manifest.json", ref)
            except Exception as error:
                errors.append({"source": str(ref), "error": safe_error(error)})
    selected = sorted(sources if not RECOVERY_TARGETS else set(sources) & set(RECOVERY_TARGETS))
    for target in set(RECOVERY_TARGETS) - set(selected):
        errors.append({"target_speedup": target, "error": "Manifest raw target belum tersedia."})
    return [(target, sources[target]) for target in selected], errors


def recovery_input(source, dataset, data_record):
    manifest, path = source["manifest"], source["path"]
    variant = NOTEBOOK_VARIANT
    if (manifest.get("format_version") != 2 or manifest.get("stage") != "raw_pruned"
            or manifest["model_variant"] != variant or manifest["source_baseline"]["sha256"] != SOURCE_CHECKPOINT_SHA256[variant]
            or any(manifest.get(key) != "PASS" for key in ("structural_status", "oracle_status", "reload_status"))
            or manifest["target_status"] not in {"REACHED", "TARGET_NOT_REACHED"}):
        raise ValueError("Input bukan raw pruned yang lulus struktur, oracle, dan reload dari baseline terkunci.")
    identity = manifest["dataset"]
    if (identity["dataset_id"] != data_record["dataset_id"] or identity["class_names"] != data_record["class_names"]
            or identity.get("train_fingerprint") != data_record["train_fingerprint"] or manifest["seed"] != SEED):
        raise ValueError("Dataset train, urutan kelas, atau seed berbeda dari manifest raw.")
    if manifest["evaluation_mode"] == "full_val":
        if identity["fingerprint"] != data_record["val_fingerprint"]:
            raise ValueError("Fingerprint full val berbeda dari manifest raw.")
    elif MODE != "smoke" or identity["fingerprint"] != dataset["fingerprint"]:
        raise ValueError("Recovery full_val memerlukan raw full_val; subset smoke juga harus cocok.")
    raw_meta, baseline_meta = manifest["raw_checkpoint"], manifest["baseline_checkpoint"]
    raw_path = (path.parent / raw_meta["relative_path"]).resolve(strict=True)
    local_baseline = path.parent / baseline_meta["relative_path"].replace("\\", "/")
    packaged_baseline = path.parent / baseline_meta["artifact_path"]
    baseline_path = (packaged_baseline if packaged_baseline.is_file() else local_baseline).resolve(strict=True)
    if sha256_file(raw_path) != raw_meta["sha256"] or sha256_file(baseline_path) != baseline_meta["sha256"]:
        raise ValueError("SHA-256 input raw/baseline tidak cocok; training ditolak.")
    payload = torch.load(raw_path, map_location="cpu", weights_only=False)
    raw = payload.get("model")
    if (type(raw) is not _RecoveryDetectionModel or payload.get("recovery_enabled") is not False
            or not raw.end2end or any(p.dtype != torch.float32 for p in raw.parameters()) or not _recovery_finite_state(raw)):
        raise ValueError("Raw wajib DetectionModel FP32 native dual-head dari notebook pruning.")
    signature = recovery_structure_signature(raw)
    serialized_signature = {"parameters": signature["parameters"], "conv_bn": {
        name: [layer["in"], layer["out"], layer["groups"]] if "in" in layer else [layer["features"]]
        for name, layer in signature["layers"].items()}}
    if (model_state_sha256(raw) != manifest["state_sha256"] or serialized_signature != manifest["structure_signature"]
            or recovery_ordered_names(raw.names) != data_record["class_names"]):
        raise ValueError("Tensor, signature struktur, atau kelas tidak identik dengan manifest raw.")
    files = {"pruned_fp32.pt": raw_path, "baseline_fp32.pt": baseline_path, "raw_pruned_manifest.json": path}
    for name, digest in manifest.get("audit_files", {}).items():
        file = path.parent / name
        if sha256_file(file) != digest:
            raise ValueError(f"SHA-256 bukti pruning tidak cocok: {name}")
        files[name] = file
    return raw, raw_path, baseline_path, files


def recovery_raw_reference(source, files, folder, target):
    if not WANDB_TRACKING:
        return None
    if source["reference"]:
        return workflow_version_ref(source["reference"])
    journal = source["path"].parent / "artifact_publications.json"
    publication = recovery_read_json(journal).get("raw-pruned") if journal.is_file() else None
    hashes = {name: sha256_file(value) for name, value in files.items()}
    if publication and publication.get("status") == "CONFIRMED" and publication.get("files_sha256") == hashes:
        return workflow_version_ref(publication["reference"])
    # The registration run is independent of the recovery run and is resumable itself.
    registry = folder / "register_raw"
    workflow_bind(registry, "input_contract.json", {"files_sha256": hashes, "platform": PLATFORM,
                  "variant": NOTEBOOK_VARIANT, "target": target, "mode": MODE, "seed": SEED})
    with workflow_tracking(registry, NOTEBOOK_VARIANT, target, registration=True) as run:
        record = workflow_publish(run, registry, "raw-pruned", files, aliases=["raw-pruned", "latest"],
                                  metadata_info={"stage": "raw_pruned", "raw_sha256": hashes["pruned_fp32.pt"]})
    return workflow_version_ref(record["reference"])


def recovery_epoch_publication(run, folder, index, row, raw_ref):
    if run is None:
        return
    epoch = index["epoch"]
    label = f"epoch-{epoch:03d}"
    # The upload package uses the same relative filenames as latest_state.json.
    state = folder / index["relative_path"]
    best = folder / index["best"]["relative_path"]
    if sha256_file(state) != index["sha256"] or sha256_file(best) != index["best"]["sha256"]:
        raise ValueError("File training yang akan diunggah berbeda dari commit epoch.")
    aliases = [label, "latest"] + (["best"] if index["best"]["epoch"] == epoch else [])
    stage_log = folder / "epoch_publications"
    stage_log.mkdir(parents=True, exist_ok=True)
    epoch_record = stage_log / (label + ".json")
    if not epoch_record.is_file():
        write_json(epoch_record, {**row, "raw_artifact_ref": raw_ref})
    files = {index["relative_path"]: state, index["best"]["relative_path"]: best,
             "epoch.json": epoch_record, "latest_state.json": folder / "latest_state.json",
             "resume_contract.json": folder / "resume_contract.json", "wandb_identity.json": folder / "wandb_identity.json",
             "training_config.json": folder / "training_config.json", "epoch_metrics.json": folder / "epoch_metrics.json"}
    marker = stage_log / (label + "-logged.json")
    if not marker.is_file():
        try:
            run.log({"epoch": epoch, "optimizer_steps": row["optimizer_steps"], "fitness": row["fitness"],
                     **{f"train/{name.removeprefix('train/')}": value for name, value in row["losses"].items()},
                     **row["metrics"], **{f"lr/pg{idx}": value for idx, value in enumerate(row["learning_rates"])}}, step=epoch)
            write_json(marker, {"status": "LOGGED", "epoch": epoch})
        except Exception as error:
            raise WorkflowUploadPending(safe_error(error)) from error
    workflow_publish(run, folder, "training-state", files, aliases=aliases, event=label,
                     metadata_info={"epoch": epoch, "raw_artifact_ref": raw_ref, "state_sha256": index["sha256"]},
                     kind="training-state")


def recovery_measure(path, dataset, folder, image):
    folder.mkdir(parents=True, exist_ok=True)
    measurement_file = folder / "measurement.json"
    digest = sha256_file(path)
    row = recovery_read_json(measurement_file) if measurement_file.is_file() else {}
    if row and row["checkpoint_sha256"] != digest:
        raise ValueError("Checkpoint berubah sesudah pengukuran.")
    if not row:
        core = YOLO(str(path), task="detect").model.to(TARGET_DEVICE).float().eval()
        core.end2end = True
        forward_snapshot(core, image)
        row = {**model_measurements(core, IMGSZ), "checkpoint_sha256": digest, "checkpoint_bytes": Path(path).stat().st_size,
               "conv_output_channels": sum(m.out_channels for m in core.modules() if isinstance(m, torch.nn.Conv2d)),
               "validation_status": "NOT_RUN", "errors": [], "latency": gflops_latency(core, image)}
    elif row["latency"]["status"] not in {"PASS", "MEASURED"} and RUN_LATENCY_BENCHMARK:
        core = YOLO(str(path), task="detect").model.to(TARGET_DEVICE).float().eval()
        core.end2end = True
        row["latency"] = gflops_latency(core, image)
    if row["validation_status"] != "PASS":
        gflops_validation(row, path, dataset, folder)
    write_json(measurement_file, row)
    return row


def recovery_run_job(folder, target, source, dataset, data_record, train_yaml, train_fp, settings, image, method):
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "recovery_manifest.json"
    previous = recovery_read_json(path) if path.is_file() else {}
    manifest = source["manifest"]
    trainer_class, trainer_kwargs, method_contract = method
    result = previous or {"format_version": 2, "model_variant": NOTEBOOK_VARIANT, "platform": PLATFORM, "mode": MODE,
        "method": RECOVERY_METHOD,
        "configuration_id": f"target-{target:.2f}x", "target_speedup": target, "target_status": manifest["target_status"],
        "actual_gflops_speedup": manifest["actual_gflops_speedup"], "errors": [], "status": "NOT_RUN",
        **{key + "_status": "NOT_RUN" for key in ("input", "training", "structural", "reload", "validation", "latency", "eda", "upload")}}
    stage = "input"
    committed = False
    try:
        raw, raw_path, baseline_path, files = recovery_input(source, dataset, data_record)
        raw_sha = manifest["raw_checkpoint"]["sha256"]
        if target in EXPECTED_RAW_SHA256 and EXPECTED_RAW_SHA256[target].lower() != raw_sha:
            raise ValueError("SHA raw berbeda dari kondisi awal KD/fine-tuning yang dikunci.")
        if previous and previous.get("raw_sha256") not in {None, raw_sha}:
            raise ValueError("Input raw berbeda dari konfigurasi recovery yang sudah tersimpan.")
        stage = "upload"
        raw_ref = recovery_raw_reference(source, files, folder, target)
        if WANDB_TRACKING and not raw_ref:
            raise WorkflowUploadPending("Artefak raw belum dikonfirmasi.")
        contract = workflow_bind(folder, "resume_contract.json", {"implementation": IMPLEMENTATION_SHA256,
            "platform": PLATFORM, "mode": MODE, "variant": NOTEBOOK_VARIANT, "target": target,
            "target_status": manifest["target_status"], "raw_sha256": raw_sha, "raw_artifact_ref": raw_ref,
            "baseline_sha256": manifest["baseline_checkpoint"]["sha256"], "settings": settings, "validation_batch": BATCH,
            "val_fingerprint": dataset["fingerprint"], "train_fingerprint": train_fp,
            "gpu": ENVIRONMENT.get("gpu"), "cuda_build": ENVIRONMENT.get("cuda_build"), "packages": ENVIRONMENT.get("packages"),
            "validation_workers": WORKERS, "latency_warmup": LATENCY_WARMUP, "latency_repeats": LATENCY_REPEATS,
            "run_latency": RUN_LATENCY_BENCHMARK, "method": method_contract})
        committed = True
        result.pop("interrupted", None)
        result.update(input_status="PASS", raw_sha256=raw_sha, raw_artifact_ref=raw_ref,
                      baseline_sha256=manifest["baseline_checkpoint"]["sha256"], dataset=data_record,
                      validation_fingerprint=dataset["fingerprint"], train_fingerprint=train_fp,
                      raw_initial_state_sha256=model_state_sha256(raw), contract_sha256=contract["sha256"], method_config=method_contract)
        write_json(path, result)
        # All three inference evaluations are remeasured with this exact device and protocol.
        stage = "validation"
        baseline = recovery_measure(baseline_path, dataset, folder.parent.parent / "baselines" / result["baseline_sha256"][:16], image)
        raw_metrics = recovery_measure(raw_path, dataset, folder / "raw_validation", image)
        if raw_metrics["parameters"] >= baseline["parameters"]:
            raise ValueError("Raw tidak lebih sempit dari baseline; training tidak dimulai.")
        result["stages"] = {**result.get("stages", {}), "baseline": baseline, "raw_pruned": raw_metrics}
        write_json(path, result)
        stage = "upload"
        with workflow_tracking(folder, NOTEBOOK_VARIANT, target, recovery=True, source_ref=raw_ref) as run:
            teacher_ref = method_contract.get("teacher_artifact_ref")
            if run is not None and teacher_ref:
                run.use_artifact(workflow_version_ref(teacher_ref), type="model")
            if run is not None:
                run.config.update({"recovery": method_contract, "training": settings, "raw_artifact_ref": raw_ref,
                                   "raw_sha256": raw_sha, "target_status": result["target_status"],
                                   "actual_gflops_speedup": result["actual_gflops_speedup"],
                                   "val_fingerprint": dataset["fingerprint"], "train_fingerprint": train_fp}, allow_val_change=False)
            stage = "training"
            state, index = recovery_fit(raw, raw_path, raw_sha, train_yaml, dataset["class_names"], folder, settings,
                contract["sha256"], val_batch=BATCH, trainer_class=trainer_class, trainer_kwargs=trainer_kwargs,
                committed_callback=lambda idx, row: recovery_epoch_publication(run, folder, idx, row, raw_ref))
            result.update(training_status="PASS", completed_epochs=state["epoch"], optimizer_steps=state["optimizer_steps"],
                          best_epoch=state["best"]["epoch"], best_fitness=state["best"]["fitness"], resume_checkpoint=index,
                          final_live_state_sha256=model_state_sha256(state["live_model"]),
                          final_ema_state_sha256=model_state_sha256(state["ema"]))
            write_csv(folder / "epoch_metrics.csv", recovery_epoch_rows(state["epoch_rows"]))
            write_json(path, result)
            stage = "structural"
            best = folder / state["best"]["relative_path"]
            recovered = folder / "recovered_fp32.pt"
            if not recovered.is_file():
                temp = recovered.with_name(f"recovered-{uuid.uuid4().hex}.tmp")
                shutil.copyfile(best, temp)
                os.replace(temp, recovered)
            recovered_sha = sha256_file(recovered)
            if recovered_sha != state["best"]["sha256"] or sha256_file(raw_path) != raw_sha:
                raise ValueError("Checkpoint recovered atau raw berubah di luar commit.")
            core = YOLO(str(recovered), task="detect").model.to(TARGET_DEVICE).float().eval()
            core.end2end = True
            signature = recovery_structure_signature(raw)
            if (recovery_structure_signature(core) != signature
                    or recovery_state_sha256(core, parameters_only=True) == recovery_state_sha256(raw, parameters_only=True)):
                raise ValueError("Checkpoint recovered berbeda struktur atau tidak memperbarui bobot raw.")
            result.update(structural_status="PASS", recovered_checkpoint={"relative_path": recovered.name, "sha256": recovered_sha},
                          structure_signature=signature, recovered_state_sha256=model_state_sha256(core), raw_unchanged=True)
            stage = "reload"
            if result["reload_status"] != "PASS":
                (folder / "reload").mkdir(exist_ok=True)
                result["reload"] = verify_fresh_reload(recovered, core, image, folder / "reload", timeout=RELOAD_TIMEOUT_SECONDS)
                result["reload_status"] = "PASS"
            write_json(path, result)
            stage = "validation"
            recovered_metrics = recovery_measure(recovered, dataset, folder / "recovered_validation", image)
            if (recovered_metrics["parameters"] != raw_metrics["parameters"]
                    or not math.isclose(recovered_metrics["gflops"], raw_metrics["gflops"], rel_tol=1e-10)):
                raise ValueError("Parameter/GFLOPs berubah sesudah fine-tuning struktur raw.")
            result["stages"]["recovered"] = recovered_metrics
            result["validation_status"] = workflow_status(row["validation_status"] for row in result["stages"].values())
            latency = [row["latency"]["status"] for row in result["stages"].values()]
            result["latency_status"] = workflow_status("PASS" if status == "MEASURED" else status for status in latency)
            stage = "eda"
            eda_metrics = folder / "eda/stage_metrics.json"
            if (result["eda_status"] != "PASS" or not eda_metrics.is_file()
                    or recovery_read_json(eda_metrics) != recovery_comparison_rows(result)[0]):
                try:
                    result["eda"] = recovery_build_eda(result, folder, state["epoch_rows"], files)
                    result["eda_status"] = result["eda"]["status"]
                except Exception as error:
                    result["eda_status"] = "FAIL"
                    result["errors"].append({"stage": "eda", "error": safe_error(error)})
            write_json(path, result)
            stage = "upload"
            # Keep the immutable model manifest independent of report/upload state.
            model_manifest = folder / "recovered_model.json"
            write_json(model_manifest, {"format_version": 2, "stage": "recovered", "model_variant": NOTEBOOK_VARIANT,
                    "configuration_id": result["configuration_id"], "target_speedup": target, "target_status": result["target_status"],
                    "checkpoint": result["recovered_checkpoint"], "state_sha256": result["recovered_state_sha256"],
                    "structure_signature": signature, "raw_artifact_ref": raw_ref, "raw_sha256": raw_sha,
                    "method": method_contract, "epochs": state["epoch"], "best_epoch": state["best"]["epoch"], "dataset": data_record})
            if run is not None:
                result["recovered_publication"] = workflow_publish(run, folder, "recovered", {
                    recovered.name: recovered, model_manifest.name: model_manifest}, aliases=["recovered", "latest"],
                    metadata_info={"raw_artifact_ref": raw_ref, "raw_sha256": raw_sha, "recovered_sha256": recovered_sha})
                recovery_log_comparison(run, result, folder)
                write_json(path, result)
                report_files = workflow_report_files(folder)
                report_files.pop(path.name, None)  # Mutable run status is not part of the immutable report version.
                report_files["protocol_environment.json"] = folder.parent.parent / "environment.json"
                result["results_publication"] = workflow_publish(run, folder, "results", report_files, kind="results")
                result["upload_status"] = "PASS"
    except (Exception, KeyboardInterrupt) as error:
        if isinstance(error, KeyboardInterrupt):
            result["interrupted"] = True
        if previous and not committed and stage != "upload":
            return {**previous, "resume_rejected": safe_error(error)}
        if isinstance(error, WorkflowUploadPending):
            stage = "upload"
        result[stage + "_status"] = "FAIL"
        result["errors"].append({"stage": stage, "error": safe_error(error) or "Interrupted"})
        if (folder / "latest_state.json").is_file():
            result["completed_epochs"] = recovery_read_json(folder / "latest_state.json")["epoch"]
            if result["training_status"] != "PASS":
                result["training_status"] = "PARTIAL"
    statuses = [result[key + "_status"] for key in ("input", "training", "structural", "reload", "validation", "eda")]
    if RUN_LATENCY_BENCHMARK:
        statuses.append(result["latency_status"])
    if WANDB_TRACKING:
        statuses.append(result["upload_status"])
    result["status"] = workflow_status(statuses, useful=bool(result.get("completed_epochs") or result.get("stages")))
    write_json(path, result)
    print(result["configuration_id"], result["status"], "| epoch", result.get("completed_epochs", 0), flush=True)
    return result


def recovery_execute(folder):
    report = {"format_version": 2, "model_variant": NOTEBOOK_VARIANT, "mode": MODE, "platform": PLATFORM,
              "method": RECOVERY_METHOD, "status": "NOT_RUN", "jobs": [], "errors": []}
    folder = Path(folder)
    path = folder / "suite_manifest.json"
    previous = recovery_read_json(path) if path.is_file() else None
    bound = False
    try:
        if MODE not in {"smoke", "full_val"} or not set(RECOVERY_TARGETS) <= set(GFLOPS_TARGETS) or NOTEBOOK_VARIANT not in {"n", "s"}:
            raise ValueError("Recovery memerlukan smoke/full_val dan target dari notebook pruning.")
        if not (PRUNING_SUITE_DIR or RAW_MANIFEST_PATHS or RAW_ARTIFACT_REFS):
            report["reason"] = "Isi PRUNING_SUITE_DIR, RAW_MANIFEST_PATHS atau RAW_ARTIFACT_REFS."
            return report
        if RECOVERY_METHOD != "finetune" and not (TEACHER_ARTIFACT_REF or TEACHER_LOCAL_PATH):
            report["reason"] = "Isi TEACHER_ARTIFACT_REF atau TEACHER_LOCAL_PATH beserta TEACHER_SHA256."
            return report
        if MODE == "full_val" and (RECOVERY_EPOCHS != 100 or BATCH != 16 or IMGSZ != 640 or SEED != 42
                                    or not WANDB_TRACKING or TARGET_DEVICE.type != "cuda"):
            raise ValueError("Recovery full_val memakai 100 epoch, batch 16, 640 px, seed 42, CUDA dan W&B aktif.")
        dataset, data_record, train_yaml, train_fp = recovery_dataset(folder / "dataset_checks" / uuid.uuid4().hex[:8])
        settings = recovery_settings()
        method = ((ResumableStructuralTrainer, {}, {"method": "finetune", "loss": "native detection"})
                  if RECOVERY_METHOD == "finetune" else recovery_prepare_method(folder))
        workflow_bind(folder, "suite_contract.json", {"implementation": IMPLEMENTATION_SHA256, "platform": PLATFORM,
            "variant": NOTEBOOK_VARIANT, "mode": MODE, "settings": settings, "val_fingerprint": dataset["fingerprint"],
            "train_fingerprint": train_fp, "dataset_provenance": data_record["provenance_status"],
            "validation_batch": BATCH, "validation_workers": WORKERS, "latency_warmup": LATENCY_WARMUP,
            "latency_repeats": LATENCY_REPEATS, "run_latency": RUN_LATENCY_BENCHMARK, "wandb": WANDB_TRACKING,
            "method": method[2],
            **{key: ENVIRONMENT.get(key) for key in ("packages", "gpu", "cuda_build")}})
        bound = True
        write_json(folder / "dataset_manifest.json", data_record)
        write_json(folder / "environment.json", ENVIRONMENT)
        sources, report["errors"] = recovery_discover_inputs(folder)
        completed = {job["target_speedup"]: job for job in (previous or {}).get("jobs", [])}
        by_raw = {}
        image = torch.rand(1, 3, IMGSZ, IMGSZ, generator=torch.Generator().manual_seed(SEED)).to(TARGET_DEVICE)
        for target, source in sources:
            raw_sha = source["manifest"]["raw_checkpoint"]["sha256"]
            configuration = f"target-{target:.2f}x"
            job_folder = folder / "jobs" / configuration
            if target in EXPECTED_RAW_SHA256 and EXPECTED_RAW_SHA256[target].lower() != raw_sha:
                report["errors"].append({"target_speedup": target, "error": "SHA raw berbeda dari kondisi awal yang dikunci."})
                continue
            if target in completed and completed[target].get("raw_sha256") not in {None, raw_sha}:
                report["errors"].append({"target_speedup": target, "error": "SHA raw berbeda dari konfigurasi recovery tersimpan."})
                continue
            if raw_sha in by_raw and not (job_folder / "latest_state.json").is_file():
                canonical = by_raw[raw_sha]
                status = ("REACHED" if source["manifest"]["actual_gflops_speedup"] >= target - 1e-12 else "TARGET_NOT_REACHED")
                job = {**canonical, "configuration_id": configuration, "target_speedup": target, "target_status": status,
                       "alias_of": canonical["configuration_id"]}
                if canonical.get("recovered_checkpoint"):
                    job["recovered_checkpoint"] = {**canonical["recovered_checkpoint"],
                        "relative_path": f"../{canonical['configuration_id']}/recovered_fp32.pt"}
                recovery_build_alias_eda(job, job_folder)
                write_json(job_folder / "recovery_manifest.json", job)
            else:
                job = recovery_run_job(job_folder, target, source, dataset, data_record, train_yaml, train_fp, settings, image, method)
                if job.get("training_status") == "PASS":
                    by_raw[raw_sha] = job
            completed[target] = job
            report["jobs"] = [completed[key] for key in sorted(completed)]
            write_json(path, report)
            if job.get("interrupted"):
                break
        report["jobs"] = [completed[key] for key in sorted(completed)]
        report["status"] = workflow_status([job["status"] for job in report["jobs"]] + ["FAIL"] * bool(report["errors"]))
        recovery_suite_tables(report, folder)
    except (Exception, KeyboardInterrupt) as error:
        if previous and not bound:
            return {**previous, "resume_rejected": safe_error(error) or "Interrupted"}
        report["status"] = "PARTIAL" if report["jobs"] else "FAIL"
        report["errors"].append({"error": safe_error(error) or "Interrupted"})
    finally:
        if bound or previous is None:
            write_json(path, report)
    return report

In [ ]:
CUSTOM_EDA_SHA256 = '7b0b8132055a2ea2059b3f4ab293fbe0ee24d61123c188f48b9141fb537f7396'
"""Explicit three-condition charts, Tables, verified EDA packages and W&B Reports.

This presentation layer never loads, trains, or changes a model. It consumes the
measurements made by the recovery evaluator, with their missing-value statuses.
"""
# ruff: noqa: F821 -- Embedded after the notebook's evaluator declarations.
import csv as _ceda_csv
import hashlib as _ceda_hashlib
import json as _ceda_json
import math as _ceda_math
import os as _ceda_os
from pathlib import Path as _CedaPath

CEDA_STAGES = ("baseline", "raw_pruned", "recovered")
CEDA_COLORS = ("#2863AD", "#BA492E", "#217854")
CEDA_METRICS = {
    "map50_95": ("mAP50–95", "Proporsi", 1, "01_accuracy"),
    "map50": ("mAP50", "Proporsi", 1, "01_accuracy"),
    "precision": ("Precision", "Proporsi", 1, "01_accuracy"),
    "recall": ("Recall", "Proporsi", 1, "01_accuracy"),
    "parameters": ("Parameter", "Juta parameter", 1e-6, "02_efficiency"),
    "gflops": ("GFLOPs", "GFLOPs", 1, "02_efficiency"),
    "conv_output_channels": ("Channel Conv2d", "Jumlah channel", 1, "02_efficiency"),
    "checkpoint_bytes": ("Ukuran checkpoint", "MiB", 1 / 2**20, "02_efficiency"),
    "latency_median_ms": ("Latency p50", "ms · raw forward batch 1", 1, "03_latency"),
    "latency_p95_ms": ("Latency p95", "ms · raw forward batch 1", 1, "03_latency"),
    "gflops_speedup": ("Speed-up GFLOPs", "× terhadap baseline asal", 1, "03_latency"),
    "latency_speedup": ("Speed-up latency", "× terhadap baseline asal", 1, "03_latency"),
}
CEDA_CLASS_METRICS = {"ap50_95": "AP50–95", "ap50": "AP50", "precision": "Precision", "recall": "Recall", "f1": "F1", "support": "Support"}
CEDA_METHODS = {"finetune": "Fine-tuned", "native": "Native KD", "crosskd": "CrossKD", "mgd": "MGD"}


def ceda_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and _ceda_math.isfinite(value)


def ceda_hash(path):
    value = _ceda_hashlib.sha256()
    with _CedaPath(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(2**20), b""):
            value.update(chunk)
    return value.hexdigest()


def ceda_read(path, default=None):
    path = _CedaPath(path)
    return _ceda_json.loads(path.read_text(encoding="utf-8")) if path.is_file() else default


def ceda_write(path, value):
    path = _CedaPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(_ceda_json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    _ceda_os.replace(temp, path)


def ceda_csv(path, rows):
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with _CedaPath(path).open("w", encoding="utf-8", newline="") as stream:
        writer = _ceda_csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


def ceda_identity(result, rows):
    first = rows[0] if rows else {}
    seed = result.get("seed", result.get("recovery_seed", first.get("seed", globals().get("SEED"))))
    method = result.get("method", first.get("method", globals().get("RECOVERY_METHOD", "finetune")))
    target = result.get("target_speedup", first.get("target_speedup"))
    actual = result.get("raw_gflops_speedup", result.get("actual_gflops_speedup"))
    if not ceda_number(actual) and len(rows) > 1:
        actual = rows[1].get("gflops_speedup")
    return {"seed": seed, "method": method, "model_variant": result.get("model_variant", first.get("model_variant", "s")),
            "target_speedup": target, "actual_gflops_speedup": actual,
            "platform": result.get("platform", first.get("platform", globals().get("PLATFORM"))),
            "mode": result.get("mode", first.get("mode", globals().get("MODE"))),
            "target_status": result.get("target_status", first.get("target_status", "NOT_RUN")),
            "raw_artifact_ref": result.get("raw_artifact_ref", first.get("raw_artifact_ref")),
            "raw_sha256": result.get("raw_sha256", first.get("raw_sha256")),
            "baseline_artifact_ref": result.get("baseline_artifact_ref", first.get("baseline_artifact_ref")),
            "baseline_sha256": result.get("baseline_sha256", result.get("baseline_source_sha256", first.get("baseline_sha256"))),
            "baseline_fp32_sha256": result.get("baseline_fp32_sha256", first.get("baseline_fp32_sha256")),
            "teacher_artifact_ref": result.get("teacher_artifact_ref", first.get("teacher_artifact_ref")),
            "teacher_sha256": result.get("teacher_sha256", first.get("teacher_sha256"))}


def ceda_title(identity):
    def speed(value):
        return f"{value:.3f}×" if ceda_number(value) else "NOT_RUN"
    target = f"{identity['target_speedup']:.2f}x" if ceda_number(identity['target_speedup']) else "NOT_RUN"
    return (f"YOLO26{identity.get('model_variant', 's')} · Seed {identity['seed']} · su{target} · aktual {speed(identity['actual_gflops_speedup'])}"
            f"\n{CEDA_METHODS.get(identity['method'], identity['method'])} · {identity['platform']} · {identity['mode']}")


def ceda_rows(result):
    compare = globals().get("fine_tuned_comparison") or recovery_comparison_rows
    rows, classes = compare({**result, "stages": result.get("stages", {})})
    identity = ceda_identity(result, rows)
    return [{**identity, **row} for row in rows], [{**identity, **row} for row in classes]


def ceda_status(row, metric):
    if ceda_number(row.get(metric)):
        return "MEASURED"
    if metric.startswith("latency"):
        return row.get("latency_status") or "NOT_RUN"
    if metric in {"map50_95", "map50", "precision", "recall"}:
        value = row.get("validation_status") or "NOT_RUN"
        return "NOT_MEASURED" if value == "PASS" else value
    return "NOT_RUN"


def ceda_build(result, folder, *, rows=None, classes=None):
    """One axis per metric; null measurements are labelled instead of drawn as zero."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    folder = _CedaPath(folder)
    target = folder / "custom_eda"
    target.mkdir(parents=True, exist_ok=True)
    if rows is None or classes is None:
        rows, classes = ceda_rows(result)
    identity = ceda_identity(result, rows)
    rows = [{**identity, **row} for row in rows]
    classes = [{**identity, **row} for row in classes]
    if [row["stage"] for row in rows] != list(CEDA_STAGES):
        raise ValueError("EDA custom memerlukan tepat tiga kondisi berurutan: baseline/raw_pruned/recovered.")
    digest = _ceda_hashlib.sha256(_ceda_json.dumps({"rows": rows, "classes": classes,
        "renderer": globals().get("CUSTOM_EDA_SHA256", "standalone")}, sort_keys=True, allow_nan=False).encode()).hexdigest()
    receipt_path = folder / "custom_eda_receipt.json"
    previous = ceda_read(receipt_path, {})
    if previous.get("input_sha256") == digest and previous.get("status") == "PASS":
        if any(not (target / name).is_file() or ceda_hash(target / name) != value for name, value in previous["files_sha256"].items()):
            raise ValueError("Berkas EDA custom berubah setelah verifikasi.")
        return previous
    # A separate package keeps pre-existing immutable results artifacts untouched.
    labels = ["Baseline asal", "Raw pruned", CEDA_METHODS.get(identity["method"], "Hasil recovery")]
    long, deltas, images = [], [], {}

    def save(fig, name, key):
        fig.suptitle(ceda_title(identity), fontsize=10, y=.99)
        fig.tight_layout(rect=(0, .08, 1, .85 if fig.legends else .96))
        fig.text(.5, .015, "Biru: baseline asal  |  Merah: raw pruned  |  Hijau: hasil recovery\n"
                 "NOT_RUN/NOT_COMPARABLE bukan nilai nol.", ha="center", fontsize=8, color="#52606D")
        fig.savefig(target / f"{name}.png", dpi=globals().get("EDA_DPI", 120), bbox_inches="tight")
        fig.savefig(target / f"{name}.svg", bbox_inches="tight", metadata={"Date": None})
        plt.close(fig)
        images[key] = f"{name}.png"

    for metric, (title, unit, scale, section) in CEDA_METRICS.items():
        fig, ax = plt.subplots(figsize=(7.8, 4.8))
        for i, row in enumerate(rows):
            value = row.get(metric)
            status = ceda_status(row, metric)
            long.append({**identity, "stage": row["stage"], "condition": labels[i], "metric": metric,
                "value": value, "display_value": value * scale if ceda_number(value) else None, "unit": unit, "status": status,
                "latency_device": row.get("latency_device"), "validation_fingerprint": row.get("validation_fingerprint")})
            if ceda_number(value):
                bar = ax.bar(i, value * scale, color=CEDA_COLORS[i], width=.6)
                ax.bar_label(bar, labels=[f"{value * scale:.5g}"], padding=4, fontsize=10)
            else:
                ax.text(i, .07, status, transform=ax.get_xaxis_transform(), ha="center", fontsize=9, rotation=15, color="#52606D")
        ax.set(title=title, ylabel=unit, xticks=range(3), xticklabels=labels)
        ax.set_xlim(-.6, 2.6)
        if metric in {"map50_95", "map50", "precision", "recall"}:
            ax.set_ylim(0, 1.1)
        else:
            ax.margins(y=.22)
        ax.grid(axis="y", alpha=.2)
        ax.set_axisbelow(True)
        save(fig, metric, f"custom_eda/{section}/{metric}")

    for metric in ("map50_95", "map50", "precision", "recall"):
        base, raw, recovered = [row.get(metric) for row in rows]
        for stage, value in zip(CEDA_STAGES, (base, raw, recovered), strict=True):
            deltas.append({**identity, "stage": stage, "metric": metric,
                "delta_vs_baseline_pp": 100 * (value - base) if ceda_number(value) and ceda_number(base) else None,
                "gain_vs_raw_pp": 100 * (value - raw) if ceda_number(value) and ceda_number(raw) else None,
                "status": "MEASURED" if ceda_number(value) and ceda_number(base) and ceda_number(raw) else "NOT_RUN"})
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.9))
    for ax, field, title in zip(axes, ("delta_vs_baseline_pp", "gain_vs_raw_pp"),
            ("Delta terhadap baseline asal", "Gain terhadap raw pruned"), strict=True):
        for i, stage in enumerate(CEDA_STAGES):
            items = [r for r in deltas if r["stage"] == stage]
            valid = [(j, r[field]) for j, r in enumerate(items) if ceda_number(r[field])]
            if valid:
                ax.bar([j + (i - 1) * .24 for j, _ in valid], [v for _, v in valid], width=.24, color=CEDA_COLORS[i])
        ax.axhline(0, color="#52606D", linewidth=.7)
        ax.set(title=title, ylabel="Poin persentase", xticks=range(4), xticklabels=["mAP50–95", "mAP50", "P", "R"])
        ax.grid(axis="y", alpha=.2)
    fig.legend([Patch(facecolor=c) for c in CEDA_COLORS], labels, loc="upper center", bbox_to_anchor=(.5, .89), ncol=3, frameon=False)
    save(fig, "delta_gain", "custom_eda/04_delta_gain/comparison")

    class_order = sorted({(r["class_id"], r["name"]) for r in classes})
    for metric, label in CEDA_CLASS_METRICS.items():
        fig, ax = plt.subplots(figsize=(max(8.5, len(class_order) * 1.1), 5))
        indexed = {(r["stage"], r["class_id"]): r for r in classes}
        for i, stage in enumerate(CEDA_STAGES):
            for j, (class_id, _name) in enumerate(class_order):
                value = indexed.get((stage, class_id), {}).get(metric)
                if ceda_number(value):
                    ax.bar(j + (i - 1) * .25, value, width=.25, color=CEDA_COLORS[i])
                else:
                    ax.text(j + (i - 1) * .25, .025, "N/A", transform=ax.get_xaxis_transform(), ha="center", fontsize=6, rotation=90)
        if not class_order:
            ax.text(.5, .5, "NOT_RUN · evaluasi per kelas belum tersedia", ha="center", transform=ax.transAxes)
        ax.set(title=f"{label} per kelas", ylabel="Objek" if metric == "support" else "Proporsi",
               xticks=range(len(class_order)), xticklabels=[name for _, name in class_order])
        if metric != "support":
            ax.set_ylim(0, 1.1)
        ax.grid(axis="y", alpha=.2)
        ax.legend([Patch(facecolor=c) for c in CEDA_COLORS], labels, ncol=3, loc="upper center", fontsize=8)
        save(fig, f"class_{metric}", f"custom_eda/05_per_class/{metric}")
    for name, records in (("stage_metrics", rows), ("per_class_metrics", classes), ("measurements", long), ("delta_gain", deltas)):
        ceda_write(target / f"{name}.json", records)
        ceda_csv(target / f"{name}.csv", records)
    ceda_write(target / "comparison_manifest.json", {"format_version": 1, "identity": identity, "input_sha256": digest,
        "custom_eda_sha256": globals().get("CUSTOM_EDA_SHA256"), "stage_order": CEDA_STAGES, "images": images,
        "meaning": "Baseline asal → raw pruned → hasil recovery; nilai kosong mempertahankan status pengukuran."})
    receipt = {"status": "PASS", "input_sha256": digest, "identity": identity, "images": images,
               "files_sha256": {p.name: ceda_hash(p) for p in sorted(target.iterdir()) if p.is_file()}}
    ceda_write(receipt_path, receipt)
    return receipt


def ceda_workspace_sdk():
    import importlib.metadata
    import subprocess
    import sys
    try:
        installed = importlib.metadata.version("wandb-workspaces")
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != "0.4.5":
        if not globals().get("INSTALL_DEPENDENCIES", True):
            raise RuntimeError("Halaman EDA W&B memerlukan wandb-workspaces==0.4.5.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "wandb-workspaces==0.4.5"], check=True)
    import wandb_workspaces.reports.v2 as reports
    return reports


def ceda_report_spec(entity, project, run_id, name, identity, media_keys, *, summary_prefix="custom_eda/values"):
    """Reports use an explicit run-ID filter, keeping concurrent sessions separate."""
    from wandb_workspaces import expr
    wr = ceda_workspace_sdk()
    runset = wr.Runset(entity=entity, project=project, name=name, filters=[expr.Metric("ID") == run_id])
    labels = ["Baseline asal", "Raw pruned", CEDA_METHODS.get(identity["method"], "Hasil recovery")]
    blocks = [wr.H1("Baseline asal → raw pruned → hasil recovery"), wr.P(ceda_title(identity).replace("\n", " · ")),
        wr.P(f"Run {run_id}. Warna: baseline biru, raw merah, recovery hijau. Baris kosong berarti belum terukur, "
             "bukan nilai nol. Full validation dan smoke dibedakan oleh mode. Latency dibandingkan hanya pada protokol/perangkat yang sama.")]
    for section, heading in (("01_accuracy", "Akurasi deteksi"), ("02_efficiency", "Parameter, GFLOPs, channel, dan checkpoint"),
                             ("03_latency", "Latency dan speed-up")):
        panels = []
        for metric, (title, unit, _scale, group) in CEDA_METRICS.items():
            if group != section:
                continue
            fields = [f"{summary_prefix}/{stage}/{metric}" if summary_prefix else f"{stage}/{metric}" for stage in CEDA_STAGES]
            # Native bars retain the exact original unit in summary (bytes and parameter counts).
            unit = {"parameters": "Jumlah parameter", "checkpoint_bytes": "Byte"}.get(metric, unit)
            panels.append(wr.BarPlot(title=f"{title} · tiga kondisi", metrics=fields, orientation="v", title_y=unit,
                line_titles=dict(zip(fields, labels, strict=True)), line_colors=dict(zip(fields, CEDA_COLORS, strict=True)),
                aggregate=False, layout=wr.Layout(x=(len(panels) % 2) * 12, y=(len(panels) // 2) * 7, w=12, h=7)))
        section_images = sorted(key for key in media_keys if key.startswith(f"custom_eda/{section}/"))
        if section_images:
            panels.append(wr.MediaBrowser(title="Diagram tiga kondisi dan status pengukuran", media_keys=section_images,
                num_columns=2, mode="gallery", layout=wr.Layout(y=14, w=24, h=12)))
        blocks.extend([wr.H2(heading), wr.PanelGrid(runsets=[runset], panels=panels)])
    categories = [
        ("Delta terhadap baseline dan gain recovery", [key for key in media_keys if "delta_gain" in key or "comparison_changes" in key]),
        ("AP, precision, recall, F1, dan support per kelas", [key for key in media_keys if "05_per_class/" in key or "comparison_per_class" in key]),
        ("Channel per layer, ledger, heatmap, dan importance LAMP", [key for key in media_keys if any(s in key.lower() for s in ("channel", "heatmap", "importance", "lamp", "dependency", "physical"))]),
        ("Kurva PR/F1 dan confusion matrix · masing-masing kondisi", [key for key in media_keys if any(s in key.lower() for s in ("confusion", "pr_curve", "f1_curve", "precision_curve", "recall_curve", "/matrix", "normalized_by_true", "evaluation_curve", "/curves/"))]),
        ("Training, validation, loss KD, dan learning rate", [key for key in media_keys if any(s in key.lower() for s in ("epoch_", "training/", "training_curves", "kd_curves", "results"))]),
        ("Trade-off akurasi dan efisiensi", [key for key in media_keys if "tradeoff" in key.lower() or "trade_off" in key.lower()]),
    ]
    for heading, keys in categories:
        blocks.append(wr.H2(heading))
        panels = []
        if heading.startswith("Training,"):
            blocks.append(wr.P("Kurva berikut memakai history per epoch dan sudah dapat dibaca selama training. "
                "Batang hijau pada evaluasi tiga kondisi tetap menunggu evaluasi checkpoint best sesudah recovery selesai."))
            series = [
                ("Loss deteksi per epoch", ["train/box_loss", "train/cls_loss", "train/l1_loss"]),
                ("Validation per epoch · bukan evaluasi akhir tiga kondisi", ["metrics/mAP50-95(B)", "metrics/mAP50(B)", "metrics/precision(B)", "metrics/recall(B)"]),
                ("Learning rate", ["lr/pg0", "lr/pg1", "lr/pg2"]),
                ("Fitness per epoch", ["fitness"]),
            ]
            if identity["method"] != "finetune":
                series.append(("Komponen KD dan total loss", ["train/dis_loss", "train/mgd_weighted_loss", "train/det_total_loss", "train/total_loss"]))
            for i, (title, metrics) in enumerate(series):
                panels.append(wr.LinePlot(title=title, x="epoch", y=metrics, smoothing_type="none",
                    layout=wr.Layout(x=(i % 2) * 12, y=(i // 2) * 7, w=12, h=7)))
        if keys:
            panels.append(wr.MediaBrowser(title=heading, media_keys=sorted(set(keys)), num_columns=2, mode="gallery",
                layout=wr.Layout(y=21 if panels else 0, w=24, h=12)))
        if panels:
            blocks.append(wr.PanelGrid(runsets=[runset], panels=panels))
        else:
            blocks.append(wr.P("NOT_RUN · panel belum tersedia pada run ini. Panel final diunggah sesudah evaluasi tiga kondisi selesai."))
    return wr.Report(entity=entity, project=project, title=f"EDA · {name} · {run_id[:8]}",
                     description="Perbandingan baseline asal, raw pruned, dan hasil recovery.", blocks=blocks, width="fluid")


def ceda_save_report(entity, project, run_id, name, identity, media_keys, folder, *, summary_prefix="custom_eda/values"):
    path = _CedaPath(folder) / "custom_eda_report.json"
    old = ceda_read(path, {})
    report = ceda_report_spec(entity, project, run_id, name, identity, media_keys, summary_prefix=summary_prefix)
    wr = ceda_workspace_sdk()
    if old.get("run_id") == run_id and old.get("url"):
        existing = wr.Report.from_url(old["url"].replace("\\", "/"))
        owners = {block.runsets[0].filters for block in existing.blocks if isinstance(block, wr.PanelGrid) and block.runsets}
        if owners and all(run_id in value for value in owners):
            existing.blocks, existing.title, existing.description = report.blocks, report.title, report.description
            report = existing
        else:
            ceda_write(_CedaPath(folder) / "custom_eda_report_conflict.json", {
                "old_url": old["url"], "reason": "Existing report filters belong to a different run; create an independent report.",
                "run_id": run_id, "observed_filters": sorted(owners)})
    # SDK 0.4.5's save() otherwise generates its server key from Python's seeded
    # random.random(). Identically seeded kernels can overwrite the same report.
    # Bind only this report instance's model name, preserving every RNG state.
    to_model = report._to_model

    def unique_model():
        model = to_model()
        model.name = "eda" + _ceda_hashlib.sha256(f"{entity}/{project}/{run_id}".encode()).hexdigest()[:24]
        return model

    object.__setattr__(report, "_to_model", unique_model)
    try:
        report.save()
    finally:
        # Pydantic validate_assignment may clear non-field instance overrides
        # when save() assigns the returned report ID.
        if "_to_model" in vars(report):
            object.__delattr__(report, "_to_model")
    # SDK 0.4.5 joins URL components with os.path.join on Windows.
    url = report.url.replace("\\", "/")
    ceda_write(path, {"status": "PENDING_VERIFICATION", "run_id": run_id, "url": url, "report_id": report.id})
    loaded = wr.Report.from_url(url)
    owners = [block.runsets[0].filters for block in loaded.blocks if isinstance(block, wr.PanelGrid) and block.runsets]
    if (loaded.id != report.id or len(loaded.blocks) != len(report.blocks) or loaded.title != report.title
            or not owners or any(run_id not in value for value in owners)):
        raise ValueError("Halaman EDA W&B belum terverifikasi setelah disimpan.")
    receipt = {"status": "CONFIRMED", "run_id": run_id, "url": url, "report_id": report.id,
               "blocks": len(loaded.blocks), "identity": identity}
    ceda_write(path, receipt)
    return receipt


def ceda_publish(run, folder, receipt):
    """The separate EDA artifact is retryable and checked by downloading every file."""
    import tempfile

    import wandb
    folder = _CedaPath(folder)
    path = folder / "custom_eda_publication.json"
    hashes = receipt["files_sha256"]
    previous = ceda_read(path, {})
    if previous.get("run_id") == run.id and previous.get("files_sha256") == hashes and previous.get("status") == "CONFIRMED":
        return previous
    api = wandb.Api(timeout=60)
    record = {"run_id": run.id, "name": f"{run.name}-eda", "status": "PENDING", "files_sha256": hashes,
              "input_sha256": receipt["input_sha256"]}
    ceda_write(path, record)
    remote = None
    for candidate in api.run(f"{run.entity}/{run.project}/{run.id}").logged_artifacts():
        if candidate.name.rsplit(":", 1)[0] == record["name"] and candidate.metadata.get("files_sha256") == hashes:
            remote = candidate
            break
    if remote is None:
        artifact = wandb.Artifact(record["name"], type="results", metadata={**receipt["identity"],
            "stage": "custom-three-condition-eda", "files_sha256": hashes, "input_sha256": receipt["input_sha256"],
            "custom_eda_sha256": globals().get("CUSTOM_EDA_SHA256")})
        for name in hashes:
            artifact.add_file(str(folder / "custom_eda" / name), name=name)
        uploaded = run.log_artifact(artifact, aliases=["latest", "three-conditions"])
        uploaded.wait(timeout=600)
        remote = api.artifact(uploaded.qualified_name)
    reference = remote.qualified_name
    if not reference.rsplit(":", 1)[-1].startswith("v"):
        raise ValueError("Publikasi EDA memerlukan versi W&B tetap :vN.")
    with tempfile.TemporaryDirectory(prefix="verify_custom_eda_", dir=folder) as cache:
        downloaded = _CedaPath(remote.download(root=cache))
        if any(ceda_hash(downloaded / name) != digest for name, digest in hashes.items()):
            raise ValueError("SHA-256 paket EDA W&B tidak cocok.")
    record.update(status="CONFIRMED", reference=reference, url=remote.url)
    ceda_write(path, record)
    return record


def ceda_log(run, result, folder):
    import wandb
    receipt = ceda_build(result, folder)
    folder = _CedaPath(folder)
    root = folder / "custom_eda"
    for name in ("stage_metrics", "per_class_metrics", "measurements", "delta_gain"):
        values = ceda_read(root / f"{name}.json", [])
        columns = list(dict.fromkeys(key for row in values for key in row))
        if columns:
            run.log({f"custom_eda/tables/{name}": wandb.Table(columns=columns, data=[[r.get(k) for k in columns] for r in values])})
    for key, name in receipt["images"].items():
        run.log({key: wandb.Image(str(root / name), caption=ceda_title(receipt["identity"]))})
    scalars = {}
    for row in ceda_read(root / "measurements.json"):
        prefix = f"custom_eda/values/{row['stage']}/{row['metric']}"
        scalars[prefix + "_status"] = row["status"]
        scalars[prefix] = row["value"] if ceda_number(row["value"]) else None
    run.summary.update(scalars)
    media = set(receipt["images"]) | set(ceda_read(folder / "wandb_media.json", {}).get("keys", []))
    # N01's legacy comparison logger has no media-index file.
    media |= {f"recovery/{p.stem}" for p in (folder / "eda").glob("comparison_*.png")}
    publication = ceda_publish(run, folder, receipt)
    report = ceda_save_report(run.entity, run.project, run.id, run.name, receipt["identity"], sorted(media), folder)
    status = {"custom_eda_status": "PASS", "custom_eda_report_status": "CONFIRMED", "custom_eda_report_url": report["url"], "custom_eda_artifact_ref": publication["reference"],
              "custom_eda_sha256": globals().get("CUSTOM_EDA_SHA256"), "custom_eda_image_count": len(receipt["images"])}
    run.summary.update(status)
    ceda_write(folder / "custom_eda_logging.json", {**status, "run_id": run.id, "image_keys": sorted(receipt["images"]),
        "table_keys": [f"custom_eda/tables/{n}" for n in ("stage_metrics", "per_class_metrics", "measurements", "delta_gain")]})
    result["custom_eda"] = {**status, "publication": publication}
    return result["custom_eda"]


def ceda_display(result, folder):
    import pandas as pd
    from IPython.display import Image, Markdown, display
    receipt = ceda_build(result, folder)
    folder = _CedaPath(folder)
    display(Markdown("### Baseline asal → raw pruned → hasil recovery\n\n" + ceda_title(receipt["identity"]).replace("\n", " · ")))
    stages = pd.DataFrame(ceda_read(folder / "custom_eda/stage_metrics.json"))
    display(stages[["stage", "validation_status", "map50_95", "map50", "precision", "recall"]])
    display(stages[["stage", "parameters", "gflops", "conv_output_channels", "checkpoint_bytes",
                    "latency_median_ms", "latency_p95_ms", "latency_status", "gflops_speedup"]])
    classes = pd.DataFrame(ceda_read(folder / "custom_eda/per_class_metrics.json"))
    if not classes.empty:
        display(classes[["stage", "name", "ap50_95", "ap50", "precision", "recall", "f1", "support"]])
    changes = pd.DataFrame(ceda_read(folder / "custom_eda/delta_gain.json"))
    display(changes[["stage", "metric", "delta_vs_baseline_pp", "gain_vs_raw_pp", "status"]])
    for name in receipt["images"].values():
        display(Image(filename=str(folder / "custom_eda" / name)))
    record = ceda_read(folder / "custom_eda_report.json", {})
    if record.get("url"):
        display(Markdown(f"**[Buka EDA custom tiga kondisi di W&B]({record['url']})**"))
    else:
        print("Halaman W&B: NOT_RUN. Aktifkan logging dan selesaikan evaluasi/publikasi untuk memperoleh tautannya.")

"""Connect the same charts and report to each YOLO26n recovery job."""
# ruff: noqa: F821 -- Notebook declarations provide the existing runtime.
_ceda_suite_build_eda = recovery_build_eda
_ceda_suite_log_comparison = recovery_log_comparison


def recovery_build_eda(result, folder, epoch_rows, files):
    report = _ceda_suite_build_eda(result, folder, epoch_rows, files)
    ceda_build(result, folder)
    return report


def recovery_log_comparison(run, result, folder):
    _ceda_suite_log_comparison(run, result, folder)
    # Include the full diagnostic gallery, which the legacy logger only stored in its artifact.
    import wandb
    keys = []
    for path in sorted((_CedaPath(folder) / "eda").rglob("*.png")):
        key = "visuals/eda/" + path.relative_to(_CedaPath(folder) / "eda").with_suffix("").as_posix()
        run.log({key: wandb.Image(str(path))})
        keys.append(key)
    ceda_write(_CedaPath(folder) / "wandb_media.json", {"keys": keys, "image_count": len(keys)})
    ceda_log(run, result, folder)


In [ ]:
STORAGE_SHA256 = 'af1dfeb57f881c5831bddbf106117889e90bf5c563129982a957ecbba224b319'
"""Best/latest checkpoint retention and verified-smoke artifact cleanup (I/O only)."""
# ruff: noqa: F821 -- Embedded after notebook persistence and evaluator declarations.
import re as _storage_re
from datetime import datetime as _StorageDateTime
from datetime import timedelta as _StorageDelta
from datetime import timezone as _StorageTimezone
from pathlib import Path as _StoragePath


def storage_time():
    return _StorageDateTime.now(_StorageTimezone(_StorageDelta(hours=7))).isoformat()


def storage_smoke(value):
    return bool(_storage_re.search(r"(?:^|[-_])smoke(?:[-_]|$)", value or "", _storage_re.I))


def storage_protected(name):
    name = name.rsplit("/", 1)[-1].split(":", 1)[0]
    if name.startswith("run-"):
        return False  # Baseline confusion-matrix Tables are diagnostics, not input models.
    return bool(_storage_re.match(r"^yolo26s-lamp-su\d+\.\d+x-seed\d+-raw$", name) or
                (("baseline" in name.lower() or "teacher" in name.lower()) and not storage_smoke(name)))


def storage_collection(reference):
    return reference.rsplit(":", 1)[0]


def storage_plain(value):
    """Detach nested W&B SummarySubDict wrappers before writing an audit JSON."""
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    if isinstance(value, (list, tuple)):
        return [storage_plain(item) for item in value]
    return {str(key): storage_plain(value[key]) for key in value.keys()}


def storage_prune_local(folder):
    """Remove only unreferenced generated snapshots after an atomic index commit."""
    folder = _StoragePath(folder)
    index = ceda_read(folder / "latest_state.json")
    if not index:
        return []
    keep = {index["relative_path"], index["best"]["relative_path"]}
    for name, digest in ((index["relative_path"], index["sha256"]),
                         (index["best"]["relative_path"], index["best"]["sha256"])):
        target = (folder / name).resolve()
        if not target.is_relative_to(folder.resolve()) or ceda_hash(target) != digest:
            raise ValueError("Best/latest tidak valid; snapshot lama tidak dihapus.")
    removed = []
    for path in (folder / "checkpoints").glob("*.pt"):
        if (path.relative_to(folder).as_posix() in keep
                or not _storage_re.fullmatch(r"(?:best|resume)-\d+-[a-f0-9]+\.pt", path.name)):
            continue
        if not path.resolve().is_relative_to((folder / "checkpoints").resolve()):
            raise ValueError("Snapshot berada di luar folder recovery.")
        removed.append({"path": path.relative_to(folder).as_posix(), "sha256": ceda_hash(path), "bytes": path.stat().st_size})
        path.unlink()
    if removed:
        history = ceda_read(folder / "local_retention_history.json", [])
        ceda_write(folder / "local_retention_history.json", history + [{"at": storage_time(), "removed": removed, "kept": sorted(keep)}])
    return removed


def storage_retire(run, folder, role, current, journal_name="artifact_publications.json"):
    """Keep one confirmed package containing both best and latest; protect other runs."""
    import wandb
    folder = _StoragePath(folder)
    reference = current.get("reference")
    if not reference or current.get("status") not in {"CONFIRMED", "PASS"}:
        return current
    if storage_protected(reference):
        return current
    api = wandb.Api(timeout=60)
    index = ceda_read(folder / "latest_state.json", {})
    best_epoch = index.get("best", {}).get("epoch")
    state = ceda_read(folder / "checkpoint_retention.json", {})
    previous = state.get(role, {})
    # Every resume package already includes the best FP32 model and latest full
    # training state. A second historical package would duplicate those weights.
    keep = {reference}
    remote = api.artifact(reference)
    aliases = set(remote.aliases) | {"latest"}
    aliases.update({"best", "last"})
    remote.aliases = sorted(aliases)
    remote.metadata.update(checkpoint_retention="best-and-latest-v1", best_model_epoch=best_epoch,
                           snapshot_epoch=index.get("epoch"), retention_scope="run")
    remote.save()
    journal = ceda_read(folder / journal_name, {})
    removed, protected, errors = [], [], []
    for event, item in journal.items():
        old_ref = item.get("reference")
        if (not old_ref or old_ref in keep or storage_collection(old_ref) != storage_collection(reference)
                or item.get("availability_status") in {"DELETED_FOR_RETENTION", "DELETED_AFTER_SMOKE"}
                or item.get("status") != "CONFIRMED"):
            continue
        try:
            old = api.artifact(old_ref)
            producer = old.logged_by()
            if producer is None or producer.id != run.id:
                protected.append({"reference": old_ref, "reason": "different producer"})
                continue
            other_users = [user.id for user in old.used_by() if user.id != run.id]
            if other_users:
                protected.append({"reference": old_ref, "reason": "used by another run", "users": other_users})
                continue
            old.delete(delete_aliases=True)
            item.update(availability_status="DELETED_FOR_RETENTION", deleted_at=storage_time())
            removed.append({"reference": old_ref, "event": event, "files_sha256": item.get("files_sha256", {})})
        except Exception as error:
            errors.append({"reference": old_ref, "error": str(error)})
    ceda_write(folder / journal_name, journal)
    state[role] = {"latest_ref": reference, "best_ref": reference, "best_epoch": best_epoch,
                   "policy": "best-and-latest-v1", "removed": previous.get("removed", []) + removed,
                   "protected": protected, "errors": errors, "updated_at": storage_time()}
    ceda_write(folder / "checkpoint_retention.json", state)
    run.summary.update({"checkpoint_retention": "best-and-latest-v1", "checkpoint_retention_status": "PARTIAL" if errors else "PASS"})
    return current


def storage_verified_retention(run, folder, role, publication):
    """Storage errors are visible and retryable without interrupting an epoch commit."""
    try:
        return storage_retire(run, folder, role, publication)
    except Exception as error:
        path = _StoragePath(folder) / "checkpoint_retention.json"
        state = ceda_read(path, {})
        state[role] = {**state.get(role, {}), "errors": [{"error": str(error)}], "updated_at": storage_time()}
        ceda_write(path, state)
        run.summary.update({"checkpoint_retention_status": "PARTIAL", "checkpoint_retention_error": str(error)})
        return publication


def storage_verify_package(reference, hashes, folder):
    import wandb
    from tempfile import TemporaryDirectory
    remote = wandb.Api(timeout=60).artifact(reference)
    if remote.qualified_name != reference:
        raise ValueError("Versi checkpoint tidak tetap.")
    with TemporaryDirectory(prefix="verify_retention_", dir=folder) as cache:
        downloaded = _StoragePath(remote.download(root=cache))
        for name, digest in hashes.items():
            path = (downloaded / name).resolve()
            if not path.is_relative_to(downloaded.resolve()) or ceda_hash(path) != digest:
                raise ValueError(f"SHA-256 paket best/latest berbeda: {name}")


def storage_cleanup_local_smoke(folder, result):
    """Remove only generated binary copies inside this completed diagnostic run."""
    folder = _StoragePath(folder).resolve()
    if result.get("mode") != "smoke" or result.get("status") != "PASS":
        raise ValueError("Cleanup lokal memerlukan smoke yang telah lulus verifikasi.")
    if not (folder / "wandb_identity.json").is_file() and not (folder / "latest_state.json").is_file():
        raise ValueError("Folder smoke tidak memiliki identitas atau commit training.")
    receipt = ceda_read(folder / "smoke_local_cleanup.json", {"removed": [], "errors": []})
    for path in sorted(folder.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".pt", ".pth", ".ckpt", ".safetensors", ".onnx", ".engine"}:
            continue
        if not path.resolve().is_relative_to(folder):
            raise ValueError("File berada di luar folder smoke; cleanup ditolak.")
        row = {"relative_path": path.relative_to(folder).as_posix(), "sha256": ceda_hash(path), "bytes": path.stat().st_size}
        try:
            path.unlink()
            receipt["removed"].append(row)
        except OSError as error:
            receipt["errors"].append({**row, "error": str(error)})
        ceda_write(folder / "smoke_local_cleanup.json", receipt)
    receipt.update(status="PASS" if not receipt["errors"] else "PARTIAL", completed_at=storage_time(),
                   bytes_removed=sum(row["bytes"] for row in receipt["removed"]), audit_files_preserved=True)
    ceda_write(folder / "smoke_local_cleanup.json", receipt)
    return receipt


def storage_prepare_retry(folder):
    """An intentionally deleted smoke publication must be re-uploaded on explicit local resume."""
    folder = _StoragePath(folder)
    cleanup = ceda_read(folder / "smoke_artifact_cleanup.json", {})
    deleted = {item["reference"] for item in cleanup.get("deleted", [])}
    if not deleted:
        return
    journal = ceda_read(folder / "artifact_publications.json", {})
    archived = {key: row for key, row in journal.items() if row.get("reference") in deleted}
    if archived:
        history = ceda_read(folder / "deleted_smoke_publication_history.json", [])
        ceda_write(folder / "deleted_smoke_publication_history.json", history + [{"at": storage_time(), "publications": archived}])
        ceda_write(folder / "artifact_publications.json", {k: v for k, v in journal.items() if k not in archived})
    custom = ceda_read(folder / "custom_eda_publication.json", {})
    if custom.get("reference") in deleted:
        custom["status"] = "DELETED_AFTER_SMOKE"
        ceda_write(folder / "custom_eda_publication.json", custom)


def storage_cleanup_smoke(folder, result, run_id):
    """Archive the API proof, then remove test artifacts while preserving run metrics and research inputs."""
    import wandb
    folder = _StoragePath(folder)
    api = wandb.Api(timeout=60)
    remote = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{run_id}")
    if not (storage_smoke(remote.name) or storage_smoke(remote.group) or remote.config.get("mode") == "smoke"):
        raise ValueError("Cleanup ditolak: run bukan smoke.")
    if remote.state in {"running", "pending", "preempting"}:
        return {"status": "DEFERRED_ACTIVE", "run_id": run_id}
    if result.get("status") != "PASS" or result.get("logging_verification", {}).get("status") != "PASS":
        raise ValueError("Smoke belum lulus evaluasi dan verifikasi logging; artefak tidak dihapus.")
    artifacts = list(remote.logged_artifacts())
    proof = {"at": storage_time(), "run_id": run_id, "run_name": remote.name,
             "summary_before_cleanup": storage_plain(remote.summary), "result_status": result.get("status"),
             "logging_verification": result.get("logging_verification"),
             "artifacts": [{"reference": a.qualified_name, "id": a.id, "size": a.size,
                            "files_sha256": a.metadata.get("files_sha256", {})} for a in artifacts]}
    proof_path = folder / "smoke_verification_before_cleanup.json"
    if not proof_path.is_file():
        ceda_write(proof_path, proof)
    else:
        history = ceda_read(folder / "smoke_cleanup_retry_history.json", [])
        ceda_write(folder / "smoke_cleanup_retry_history.json", history + [proof])
    prior = ceda_read(folder / "smoke_artifact_cleanup.json", {})
    receipt = {"status": "RUNNING", "run_id": run_id, "deleted": prior.get("deleted", []),
               "protected": [], "errors": [], "at": storage_time(),
               "previous_errors": prior.get("previous_errors", []) + prior.get("errors", []) +
                   ([{"error": prior["error"]}] if prior.get("error") else [])}
    ceda_write(folder / "smoke_artifact_cleanup.json", receipt)
    collections = {}
    for artifact in artifacts:
        reference = artifact.qualified_name
        if reference in {row["reference"] for row in receipt["deleted"]}:
            continue
        if artifact.type.startswith("wandb-"):
            receipt["protected"].append({"reference": reference, "reason": "system managed; retained with run history"})
            continue
        if storage_protected(reference):
            receipt["protected"].append(reference)
            continue
        try:
            non_smoke = [user.id for user in artifact.used_by() if user.id != run_id and
                         not (storage_smoke(user.name) or storage_smoke(user.group) or user.config.get("mode") == "smoke")]
            if non_smoke:
                receipt["protected"].append({"reference": reference, "users": non_smoke})
                continue
            artifact.delete(delete_aliases=True)
            receipt["deleted"].append({"reference": reference, "id": artifact.id, "size": artifact.size, "type": artifact.type,
                                       "files_sha256": artifact.metadata.get("files_sha256", {})})
            collections[(artifact.type, storage_collection(reference))] = True
        except Exception as error:
            receipt["errors"].append({"reference": reference, "error": str(error)})
        ceda_write(folder / "smoke_artifact_cleanup.json", receipt)
    receipt["empty_collections_removed"] = []
    for kind, collection_name in collections:
        try:
            collection = api.artifact_collection(type_name=kind, name=collection_name)
            if not list(collection.artifacts(per_page=100)):
                collection.delete()
                receipt["empty_collections_removed"].append(collection_name)
        except Exception as error:
            receipt["errors"].append({"collection": collection_name, "error": str(error)})
    receipt.update(status="PASS" if not receipt["errors"] else "PARTIAL", completed_at=storage_time(),
                   artifacts_available=False, run_metrics_preserved=True,
                   logical_bytes_deleted=sum(row["size"] or 0 for row in receipt["deleted"]))
    ceda_write(folder / "smoke_artifact_cleanup.json", receipt)
    remote.summary.update({"smoke_artifact_cleanup": receipt["status"], "artifact_availability": "DELETED_AFTER_SMOKE",
                           "checkpoint_retention": "best-and-latest-v1"})
    for journal_name in ("artifact_publications.json",):
        journal = ceda_read(folder / journal_name, {})
        deleted = {row["reference"] for row in receipt["deleted"]}
        for item in journal.values():
            if item.get("reference") in deleted:
                item["availability_status"] = "DELETED_AFTER_SMOKE"
        ceda_write(folder / journal_name, journal)
    return receipt

"""Keep the YOLO26n recovery workflow on the same best/latest storage policy."""
# ruff: noqa: F821 -- Embedded after the suite runtime.
_storage_workflow_publish = workflow_publish
_storage_suite_save_model = ResumableStructuralTrainer.save_model
_storage_recovery_run_job = recovery_run_job


def workflow_publish(run, folder, label, files, *, metadata_info=None, aliases=None, event=None, kind="model"):
    files = dict(files)
    if run is not None and label == "recovered":
        # Preserve the old recovered_fp32.pt entry and add explicit best/last
        # names without reconstructing a standard, unpruned model.
        index = ceda_read(_StoragePath(folder) / "latest_state.json")
        state = torch.load(_StoragePath(folder) / index["relative_path"], map_location="cpu", weights_only=False)
        last = _StoragePath(folder) / "last.pt"
        if not last.is_file():
            recovery_atomic_torch_save({"model": _recovery_native_snapshot(state["ema"]), "ema": None,
                "optimizer": None, "train_args": {}, "epoch": state["epoch"] - 1, "recovery_enabled": True}, last)
        files["best.pt"] = _StoragePath(folder) / index["best"]["relative_path"]
        files["last.pt"] = last
    record = _storage_workflow_publish(run, folder, label, files, metadata_info={**(metadata_info or {}),
        "storage_sha256": STORAGE_SHA256, "checkpoint_retention": CHECKPOINT_RETENTION},
        aliases=aliases, event=event, kind=kind)
    if run is not None and record.get("status") == "CONFIRMED":
        storage_verify_package(record["reference"], record["files_sha256"], folder)
        if label in {"recovered", "training-state"}:
            storage_verified_retention(run, folder, label, record)
    return record


def _storage_suite_save(self):
    result = _storage_suite_save_model(self)
    storage_prune_local(self.recovery_folder)
    return result


ResumableStructuralTrainer.save_model = _storage_suite_save


def _storage_verify_suite_logging(folder, result, run_id):
    import wandb
    remote = wandb.Api(timeout=60).run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{run_id}")
    epochs = remote.history(keys=["epoch", "fitness"], samples=max(500, remote.lastHistoryStep + 1), pandas=False)
    if sorted(int(row["epoch"]) for row in epochs) != list(range(1, result["completed_epochs"] + 1)):
        raise ValueError("History smoke belum lengkap atau memiliki epoch ganda.")
    custom = ceda_read(folder / "custom_eda_logging.json", {})
    images = ceda_read(folder / "wandb_media.json", {}).get("keys", []) + custom.get("image_keys", [])
    tables = ["recovery/three_stage_metrics", "recovery/three_stage_per_class"] + custom.get("table_keys", [])
    for key in images + tables:
        if not remote.summary.get(key, {}).get("_type"):
            raise ValueError(f"EDA smoke belum tercatat di W&B: {key}")
    proof = {"status": "PASS", "run_id": run_id, "epoch_count": len(epochs),
             "image_count": len(images), "table_count": len(tables)}
    ceda_write(folder / "wandb_logging_verification.json", proof)
    return proof


def recovery_run_job(folder, *args, **kwargs):
    if ceda_read(_StoragePath(folder) / "smoke_local_cleanup.json", {}).get("removed"):
        raise ValueError("Checkpoint smoke sudah dibersihkan; gunakan suite diagnostik baru.")
    storage_prepare_retry(folder)
    result = _storage_recovery_run_job(folder, *args, **kwargs)
    folder = _StoragePath(folder)
    if MODE == "smoke" and result.get("status") == "PASS":
        if WANDB_TRACKING and AUTO_DELETE_SMOKE_ARTIFACTS:
            identity = ceda_read(folder / "wandb_identity.json")
            result["logging_verification"] = _storage_verify_suite_logging(folder, result, identity["id"])
            cleanup = storage_cleanup_smoke(folder, result, identity["id"])
            result["smoke_artifact_cleanup"] = cleanup
            if cleanup["status"] == "PASS":
                result["artifact_availability"] = "DELETED_AFTER_SMOKE"
        if AUTO_DELETE_LOCAL_SMOKE_WEIGHTS:
            result["smoke_local_cleanup"] = storage_cleanup_local_smoke(folder, {**result, "mode": MODE})
            result["local_checkpoint_availability"] = "DELETED_AFTER_SMOKE"
        ceda_write(folder / "recovery_manifest.json", result)
    return result


In [ ]:
BUNDLE = recovery_execute(SUITE_DIR)
print("Output", SUITE_DIR, "| Status", BUNDLE["status"], "|", BUNDLE.get("reason", ""))
display(pd.DataFrame(BUNDLE.get("errors", [])))

## 06 — EDA dan evaluasi per target
Panel terpisah membandingkan akurasi, AP per kelas, ukuran model, channel, GFLOPs, latensi, speed-up, dan recovery gain. Nilai yang belum diukur tetap kosong.
Folder EDA juga memuat kurva training/validation, PR/F1, confusion matrix, serta audit channel, heatmap dan LAMP dari paket raw jika tersedia.

EDA custom memakai satu diagram batang per metrik: **baseline asal (biru) → raw pruned (merah) → hasil metode recovery ini (hijau)**. Tiap panel mencantumkan seed, target, speed-up aktual, metode, dan mode. Nilai yang belum terukur ditandai NOT_RUN, termasuk latency yang belum dijalankan.

Buka tautan **custom_eda_report_url** pada summary run W&B untuk halaman perbandingan yang sudah disusun: metrik utama, efisiensi, delta/gain, AP/P/R/F1/support per kelas, channel/heatmap/LAMP, PR/F1, confusion matrix, training, dan trade-off. Tabel serta gambar dicatat langsung pada run; paket `<nama-run>-eda:vN` menyimpan salinan terverifikasi. Dependency halaman W&B: `wandb-workspaces==0.4.5`, tanpa mengganti versi W&B training.


In [ ]:

for job in BUNDLE.get("jobs", []):
    ceda_display(job, SUITE_DIR / "jobs" / job["configuration_id"])

from IPython.display import Image as NotebookImage, Markdown

for job in BUNDLE["jobs"]:
    configuration = job["configuration_id"]
    display(Markdown(f"### {configuration} · {job['target_status']} · {job['status']}"))
    display(pd.DataFrame([{"stage": key, **{name: stage.get(name) for name in ("validation_status", "parameters", "gflops")}}
                          for key, stage in job.get("stages", {}).items()]))
    eda = SUITE_DIR / "jobs" / configuration / "eda"
    if (eda / "stage_metrics.csv").is_file():
        display(pd.read_csv(eda / "stage_metrics.csv"))
    if job.get("alias_of"):
        display(Markdown(f"Checkpoint dan audit struktur dibagi dengan {job['alias_of']}."))
    for figure in sorted(eda.rglob("*.png")):
        display(NotebookImage(filename=str(figure)))

## 07 — Ekspor hasil dan lineage
Gunakan manifest raw serta SHA-256 yang sama untuk membandingkan seluruh metode recovery. `recovered_fp32.pt` adalah model inference. State resume disimpan terpisah dalam `checkpoints/`.

In [ ]:
for name in ("suite_manifest.json", "recovery_summary.csv", "stage_metrics.csv", "per_class_metrics.csv"):
    path = SUITE_DIR / name
    if path.is_file():
        print(path)
        if path.suffix == ".csv" and path.with_suffix(".json").is_file():
            display(pd.DataFrame(recovery_read_json(path.with_suffix(".json"))))
for job in BUNDLE["jobs"]:
    print(job["configuration_id"], job.get("raw_artifact_ref"), job.get("raw_sha256"), job.get("recovered_publication"))